# Gemma 4 is Ami

> **Ami** recommends films, series, novels and anecdotes based on the user's emotional state and generational nostalgia window, grounded in peer-reviewed psychology research (nostalgia, social surrogates, narrative transport, awe, elevation).

## Pipeline

1. **Safety Layer** — 3-tier crisis classifier (regex + LLM-as-judge)
2. **Psychologist agent (Gemma 4)** — decodes the user's emotion and selects a helpful mechanism
3. **Librarian agent (Gemma 4)** — picks 4 items and writes a personalized thesis
   - **Classic mode** — JSON-prompting from a filtered catalogue
   - **Agentic mode** — native function calling with 4 tools (search, paper, score, finalize)

**Stack** — Gemma 4 31B-IT (4-bit, on-device) · HuggingFace Transformers · Flask · React 18 (CDN) · ngrok

The Hugging Face version may be slightly different.

---

## How to run

1. Enable **GPU T4** in notebook settings
2. Enable **Internet**
3. Set `NGROK_TOKEN` in the Config cell
4. Run All → wait ~13-15 min for model load
5. Public URL appears in the last cell

## 1. Setup

Install dependencies, configure paths, and load Gemma 4 in 4-bit quantization.

In [1]:
# ════════════════════════════════════════════════
# 01 · Install dependencies
# ════════════════════════════════════════════════

import subprocess, sys

# Gemma 4 nécessite la version bleeding-edge de Transformers (pas encore sur PyPI)
packages = [
    "git+https://github.com/huggingface/transformers.git",
    "flask",
    "flask-cors",
    "pyngrok==7.2.0",
    "accelerate>=0.34",
    "bitsandbytes",
    "sentencepiece",
    "protobuf",
]

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])
print("✅ Dépendances installées")

# Vérifie la version
import transformers
print(f"   Transformers version : {transformers.__version__}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.6/663.6 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 92.4 MB/s eta 0:00:00
✅ Dépendances installées
   Transformers version : 5.8.0.dev0


In [2]:
# ════════════════════════════════════════════════
# 02 · Configuration
# ════════════════════════════════════════════════

# ⚠️  REMPLACE PAR TON TOKEN ngrok (https://dashboard.ngrok.com/get-started/your-authtoken)
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
NGROK_TOKEN = user_secrets.get_secret("NGROK_TOKEN")


# Chemin des weights Gemma 4 sur Kaggle (attaché via dataset/model)
MODEL_PATH = "/kaggle/input/models/google/gemma-4/transformers/gemma-4-31b-it/1"
#MODEL_PATH = "/kaggle/input/models/google/gemma-4/transformers/gemma-4-26b-a4b-it/1"
#MODEL_PATH = "/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1"

import os
# Fallback si tu veux tester avec le 9B plus léger
ALT_MODEL = "/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1"

# Port local Flask
FLASK_PORT = 5000

print(f"✅ Config OK — model path: {MODEL_PATH}")

✅ Config OK — model path: /kaggle/input/models/google/gemma-4/transformers/gemma-4-31b-it/1


In [3]:
# ════════════════════════════════════════════════
# 03 · Load Gemma 4 — MULTIMODAL (text + vision)
# ════════════════════════════════════════════════
# Gemma 4 is natively multimodal. We load the vision-language class
# so the same weights serve text-only AND image+text prompts.
# The text API (gemma_call) keeps working unchanged — we just gain vision.

import torch
from transformers import (
    AutoTokenizer,
    AutoProcessor,
    AutoModelForImageTextToText,
    BitsAndBytesConfig,
)

print('⏳ Chargement du tokenizer + processor (vision)...')

model_dir = MODEL_PATH if os.path.exists(MODEL_PATH) else ALT_MODEL
print(f"   Utilisation de : {model_dir}")

tokenizer = AutoTokenizer.from_pretrained(model_dir)

# Processor handles BOTH text tokenization AND image preprocessing
# (resize, patchify, normalize) for Gemma 4's vision tower.
processor = AutoProcessor.from_pretrained(model_dir)

# 4-bit quantization: ~54GB → ~8GB VRAM for the 31B model.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print('⏳ Chargement du modèle multimodal en 4-bit (patience)...')
# AutoModelForImageTextToText auto-resolves to the right Gemma 4 vision-language class.
# It handles both text-only AND image+text inputs through the same .generate() call.
model = AutoModelForImageTextToText.from_pretrained(
    model_dir,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
)
model.eval()

print("✅ Gemma 4 multimodal chargé (text + vision)")
print(f"   VRAM utilisée : {torch.cuda.memory_allocated()/1e9:.1f} GB")


⏳ Chargement du tokenizer + processor (vision)...
   Utilisation de : /kaggle/input/models/google/gemma-4/transformers/gemma-4-31b-it/1


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


⏳ Chargement du modèle multimodal en 4-bit (patience)...


Loading weights:   0%|          | 0/1188 [00:00<?, ?it/s]

✅ Gemma 4 multimodal chargé (text + vision)
   VRAM utilisée : 7.9 GB


In [4]:
# ════════════════════════════════════════════════
# 03b · Vision dequantization (CRITICAL fix)
# ════════════════════════════════════════════════
# bitsandbytes 4-bit quantization corrupts Gemma 4's vision branch:
# vision_tower stays mostly OK but embed_vision (the bridge linear that
# converts visual tokens to LLM hidden space) is fully 4-bit, which
# produces vision features the LLM "sees" as uniform gray.
#
# Fix: surgically dequantize the vision modules back to bf16 in-place.
# Cost: ~600 MB extra VRAM (vision tower 297M + embed_vision 3.1M, in bf16).
# Without this cell, the model loads fine but is BLIND to images.
#
# This cell is idempotent — safe to re-run.

import torch
import torch.nn as nn
import bitsandbytes as bnb
from bitsandbytes.nn import Linear4bit
from bitsandbytes.functional import dequantize_4bit
import gc


def _dequantize_linear4bit_in_place(parent: nn.Module, attr_name: str,
                                     target_dtype=torch.bfloat16):
    old = getattr(parent, attr_name)
    if not isinstance(old, Linear4bit):
        return False
    weight_bf16 = dequantize_4bit(
        old.weight.data,
        quant_state=old.weight.quant_state,
    ).to(target_dtype)
    new_linear = nn.Linear(
        in_features  = weight_bf16.shape[1],
        out_features = weight_bf16.shape[0],
        bias         = old.bias is not None,
        device       = weight_bf16.device,
        dtype        = target_dtype,
    )
    new_linear.weight = nn.Parameter(weight_bf16, requires_grad=False)
    if old.bias is not None:
        new_linear.bias = nn.Parameter(old.bias.data.to(target_dtype),
                                        requires_grad=False)
    setattr(parent, attr_name, new_linear)
    return True


def _walk_and_dequantize(root: nn.Module, root_name: str = "") -> int:
    n_replaced = 0
    targets = []
    for name, child in root.named_children():
        full = f"{root_name}.{name}" if root_name else name
        if isinstance(child, Linear4bit):
            targets.append((root, name, full))
        else:
            n_replaced += _walk_and_dequantize(child, full)
    for parent, attr, full in targets:
        if _dequantize_linear4bit_in_place(parent, attr):
            n_replaced += 1
    return n_replaced


vram_before = torch.cuda.memory_allocated() / 1e9
print(f"VRAM before dequant: {vram_before:.2f} GB")

n1 = _walk_and_dequantize(model.model.embed_vision, "embed_vision")
n2 = _walk_and_dequantize(model.model.vision_tower, "vision_tower")
print(f"   embed_vision : {n1} Linear4bit → bf16")
print(f"   vision_tower : {n2} Linear4bit → bf16")

gc.collect(); torch.cuda.empty_cache()
vram_after = torch.cuda.memory_allocated() / 1e9
print(f"VRAM after  dequant: {vram_after:.2f} GB  (Δ = +{vram_after-vram_before:.2f} GB)")

# Verify
def _check(label, mod):
    has_4bit = any(isinstance(m, Linear4bit) for m in mod.modules())
    sym = "❌ STILL 4-BIT" if has_4bit else "✅ pure bf16"
    print(f"   {sym}  {label}")

_check("vision_tower",  model.model.vision_tower)
_check("embed_vision",  model.model.embed_vision)
print("✅ Vision branch is now safe to use.")


VRAM before dequant: 7.94 GB
   embed_vision : 1 Linear4bit → bf16
   vision_tower : 190 Linear4bit → bf16
VRAM after  dequant: 8.76 GB  (Δ = +0.82 GB)
   ✅ pure bf16  vision_tower
   ✅ pure bf16  embed_vision
✅ Vision branch is now safe to use.


In [5]:
# ════════════════════════════════════════════════
# 04 · Gemma inference helpers (text + vision)
# ════════════════════════════════════════════════
import re, json, torch, base64, io
from PIL import Image

def gemma_call(prompt: str, max_new_tokens: int = 400, temperature: float = 0.75) -> str:
    """Text-only call to Gemma 4. Unchanged API — used everywhere downstream."""
    messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    # The processor's chat template handles the multimodal-style messages
    # for both text-only and image+text inputs (Gemma 4 chat format).
    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt",
    ).to(model.device)

    input_len = inputs["input_ids"].shape[-1]

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            top_p=0.92,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output_ids[0][input_len:]
    text = processor.decode(new_tokens, skip_special_tokens=True).strip()
    text = re.sub(r'^```json\s*', '', text)
    text = re.sub(r'```\s*$',     '', text).strip()
    return text


def _decode_image_b64(image_b64: str) -> Image.Image:
    """Accepts either a raw base64 string or a data URL (data:image/...;base64,...)."""
    if "," in image_b64 and image_b64.strip().startswith("data:"):
        image_b64 = image_b64.split(",", 1)[1]
    raw = base64.b64decode(image_b64)
    img = Image.open(io.BytesIO(raw)).convert("RGB")
    # Gemma 4 vision tower expects ~896px patches; resize to keep VRAM low.
    max_side = 896
    if max(img.size) > max_side:
        img.thumbnail((max_side, max_side), Image.LANCZOS)
    return img


def gemma_vision_call(image_b64: str, prompt: str,
                      max_new_tokens: int = 300,
                      temperature: float = 0.6) -> str:
    """
    Multimodal call: feed an image + text prompt to Gemma 4 and return the response.
    Used for analyzing photos / memes / screenshots uploaded by the user.
    """
    image = _decode_image_b64(image_b64)

    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text",  "text":  prompt},
        ],
    }]

    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt",
    ).to(model.device, dtype=torch.bfloat16)

    input_len = inputs["input_ids"].shape[-1]

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output_ids[0][input_len:]
    text = processor.decode(new_tokens, skip_special_tokens=True).strip()
    text = re.sub(r'^```json\s*', '', text)
    text = re.sub(r'```\s*$',     '', text).strip()
    return text


def extract_json(text: str) -> dict | None:
    """Robustly extract the first JSON object from Gemma's output."""
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            pass
    return None


# Quick smoke test (text path only — vision tested when an image is uploaded)
result = gemma_call('Reply with this exact JSON: {"status":"ok","model":"gemma4"}', max_new_tokens=50)
print("✅ gemma_call() OK:", result[:80])
print("✅ gemma_vision_call() ready (will be exercised when user uploads an image)")


✅ gemma_call() OK: {"status":"ok","model":"gemma4"}
✅ gemma_vision_call() ready (will be exercised when user uploads an image)


In [6]:
# ════════════════════════════════════════════════
# 04c · Vision smoke-test (validates the full image pipeline)
# ════════════════════════════════════════════════
# Downloads a known image, asks Gemma what's in it. If this prints something
# sensible (mentions a bee/insect/flower), vision is working end-to-end.
# If it returns "gray" / "blurry" / "None" → the dequant cell didn't take;
# re-run cell 03b before continuing.

import io, urllib.request, base64
from PIL import Image as _PIL

print("⏳ Downloading bee.jpg (canonical Gemma demo image)...")
_url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/bee.jpg"
_req = urllib.request.Request(_url, headers={"User-Agent": "Mozilla/5.0"})
with urllib.request.urlopen(_req, timeout=20) as _r:
    _img_bytes = _r.read()
_img = _PIL.open(io.BytesIO(_img_bytes)).convert("RGB")
print(f"   OK ({_img.size[0]}×{_img.size[1]})")

# Convert to data URL the way our pipeline does
_buf = io.BytesIO()
_img.save(_buf, format="JPEG", quality=92)
_test_b64 = "data:image/jpeg;base64," + base64.b64encode(_buf.getvalue()).decode()

# Probe via the actual gemma_vision_call helper we use in production
print("\n⏳ Asking Gemma: 'What animal is in this picture? Answer in one word.'")
import time
_t0 = time.time()
_answer = gemma_vision_call(
    _test_b64,
    "What animal is in this picture? Answer in one word.",
    max_new_tokens=12,
    temperature=0.5,
)
_dt = time.time() - _t0
print(f"   Gemma ({_dt:.1f}s): {_answer!r}")

_lower = _answer.lower()
if any(w in _lower for w in ["bee", "wasp", "insect", "abeille", "bumblebee", "hornet"]):
    print("\n✅ Vision works end-to-end. Safe to continue.")
else:
    print("\n❌ Gemma did NOT recognize the bee. Dequant likely failed.")
    print("   → Re-run the dequantization cell (03b) AND this one.")
    print("   → If still failing, restart kernel and re-run from cell 03 onward.")


⏳ Downloading bee.jpg (canonical Gemma demo image)...
   OK (5184×3456)

⏳ Asking Gemma: 'What animal is in this picture? Answer in one word.'
   Gemma (11.0s): 'Bee'

✅ Vision works end-to-end. Safe to continue.


## 2. Knowledge base

Load the curated catalogue (films, series, novels, anecdotes) and the research papers that ground each psychological mechanism.

In [7]:
# ════════════════════════════════════════════════
# 05 · Load catalogue & research papers
# Deux sources :
#   - ami_catalogue.json   : films + series + novels (tous sous la clef `films`,
#                            distingués par leur champ `type`) + meta complète
#   - ami_anecdotes.json   : anecdotes (mécanisme : nostalgia)
# ════════════════════════════════════════════════
import json, os
from collections import Counter

CATALOGUE_PATH = "/kaggle/input/datasets/aaaaak/ami-dataset-movies-series-novels-updated/ami_catalogue_updated.json"
ANECDOTES_PATH = "/kaggle/input/datasets/aaaaak/ami-dataset-anecdotes/ami_anecdotes.json"

if not os.path.exists(CATALOGUE_PATH):
    raise FileNotFoundError(
        f"Catalogue introuvable : {CATALOGUE_PATH}\n"
        "→ Vérifie que le dataset Kaggle est bien attaché au notebook."
    )

with open(CATALOGUE_PATH, "r", encoding="utf-8") as f:
    CATALOGUE = json.load(f)

# Collecte tous les items du catalogue principal, peu importe sous quelle
# clef ils sont rangés. On déduplique par id pour être robuste si le builder
# change un jour de schéma (split par type ou tout sous `films`).
_seen_ids = set()
_main_items = []
for key in ["films", "series", "novels"]:
    for item in CATALOGUE.get(key, []) or []:
        iid = item.get("id")
        if iid and iid not in _seen_ids:
            _seen_ids.add(iid)
            _main_items.append(item)

# Charge et fusionne les anecdotes
anecdotes_loaded = 0
if os.path.exists(ANECDOTES_PATH):
    with open(ANECDOTES_PATH, "r", encoding="utf-8") as f:
        ANECDOTES_FILE = json.load(f)
    _anecdotes = ANECDOTES_FILE.get("anecdotes", []) or []
    for item in _anecdotes:
        iid = item.get("id")
        if iid and iid not in _seen_ids:
            _seen_ids.add(iid)
    anecdotes_loaded = len(_anecdotes)
else:
    print(f"⚠️  Anecdotes file not found at {ANECDOTES_PATH} — continuing without anecdotes")
    _anecdotes = []

# Index plat unifié (source of truth pour le librarian)
ALL_ITEMS = _main_items + _anecdotes

# Re-groupe par `type` (utilisé par les logs et certains affichages)
_by_type = {"film": [], "series": [], "novel": [], "anecdote": []}
for it in ALL_ITEMS:
    t = it.get("type")
    if t in _by_type:
        _by_type[t].append(it)

# Re-injecte dans CATALOGUE en gardant la structure attendue par le reste du code
CATALOGUE["films"]     = _by_type["film"]
CATALOGUE["series"]    = _by_type["series"]
CATALOGUE["novels"]    = _by_type["novel"]
CATALOGUE["anecdotes"] = _by_type["anecdote"]

# Index par id pour lookup rapide
ITEMS_BY_ID = {item["id"]: item for item in ALL_ITEMS if item.get("id")}


# Research papers (from main catalogue's meta — has all 11 mechanisms;
# the anecdotes file only has the nostalgia paper)
RESEARCH_PAPERS = CATALOGUE.get("meta", {}).get("papers", [])

# Real distribution of mechanisms in the catalogue.
# Fed to the psychologist so Gemma avoids primary mechanisms with no items.
MECHANISM_COUNTS = Counter(i.get("mechanism") for i in ALL_ITEMS if i.get("mechanism"))

def _availability_tag(count):
    """Honest qualitative label fed to Gemma — short, no numbers in the prompt
    (numbers tend to be cargo-culted; labels guide judgment better)."""
    if count >= 50:  return "RICH catalogue"
    if count >= 20:  return "good catalogue"
    if count >= 10:  return "limited catalogue"
    if count >= 3:   return "VERY LIMITED — avoid unless perfect fit"
    return "almost no items — DO NOT choose unless absolutely necessary"

MECHANISM_BLOCK = "\n".join([
    f'- "{p["id"]}": {p["mechanism"]} ({_availability_tag(MECHANISM_COUNTS.get(p["id"], 0))}) — {p.get("description","")[:150]}'
    for p in RESEARCH_PAPERS
])

# Moods : prend ceux du catalogue principal (24 complets > 17 dans anecdotes)
MOOD_KEYS = [m["key"] for m in CATALOGUE.get("meta", {}).get("moods", [])] or [
    "overwhelmed","anxious","sad","angry","exhausted","lonely","empty","lost",
    "guilty","ashamed","discouraged","bored","nostalgic","grieving","melancholic",
    "cynical","doubtful","confused","restless","self_angry","envious",
    "embarrassed","self_disgust","meaningless"
]
MOOD_LIST_STR = ", ".join(MOOD_KEYS)

print(f"✅ Catalogue chargé : {len(ALL_ITEMS)} items")
print(f"   🎬 Films:     {len(CATALOGUE['films'])}")
print(f"   📺 Séries:    {len(CATALOGUE['series'])}")
print(f"   📚 Romans:    {len(CATALOGUE['novels'])}")
print(f"   ✨ Anecdotes: {len(CATALOGUE['anecdotes'])}")
print(f"   🧠 Mécanismes: {len(RESEARCH_PAPERS)} (distribution : {dict(MECHANISM_COUNTS.most_common())})")


✅ Catalogue chargé : 448 items
   🎬 Films:     150
   📺 Séries:    100
   📚 Romans:    50
   ✨ Anecdotes: 148
   🧠 Mécanismes: 11 (distribution : {'nostalgia': 150, 'transport': 95, 'elevation': 75, 'benign_masochism': 40, 'awe': 37, 'self_expansion': 21, 'parasocial': 14, 'prosocial': 12, 'involuntary_memory': 2, 'social_surrogate': 1, 'incongruence': 1})


In [8]:
# ════════════════════════════════════════════════
# 06 · Helper: get_mechanism_desc()
# Récupère la description d'un mécanisme depuis RESEARCH_PAPERS
# ════════════════════════════════════════════════

# Index par id pour lookup O(1)
_MECHANISM_INDEX = {
    p["id"]: {
        "mechanism":   p.get("mechanism", p["id"]),
        "description": p.get("description", ""),
        "paper":       p.get("paper", ""),
    }
    for p in RESEARCH_PAPERS
}


def get_mechanism_desc(mechanism_id: str) -> str:
    """
    Renvoie la description d'un mécanisme psychologique pour le prompt Bibliothécaire.
    Fallback robuste : si le mécanisme n'existe pas, renvoie une chaîne neutre.
    """
    if not mechanism_id:
        return ""
    info = _MECHANISM_INDEX.get(mechanism_id)
    if info:
        return info["description"]
    # Fallback : recherche fuzzy (insensible à la casse, retire underscores)
    norm = mechanism_id.lower().replace("_", " ").strip()
    for pid, info in _MECHANISM_INDEX.items():
        if pid.lower().replace("_", " ") == norm:
            return info["description"]
    # Vraiment rien trouvé
    return f"({mechanism_id} — mechanism description not available)"


# ── Tests
print("⏳ Testing get_mechanism_desc...")
known_ids = list(_MECHANISM_INDEX.keys())[:3]
for mid in known_ids:
    desc = get_mechanism_desc(mid)
    print(f"   ✅ '{mid}' → {desc[:80]}{'...' if len(desc) > 80 else ''}")

print(f"   ✅ unknown → {get_mechanism_desc('does_not_exist')[:80]}")
print(f"   ✅ empty   → '{get_mechanism_desc('')}'")
print(f"\n✅ {len(_MECHANISM_INDEX)} mécanismes indexés depuis RESEARCH_PAPERS")

⏳ Testing get_mechanism_desc...
   ✅ 'benign_masochism' → Experiencing intense negative emotions (sadness, fear, tension, disgust) through...
   ✅ 'elevation' → Witnessing extraordinary moral beauty, virtue, or human excellence — even in fic...
   ✅ 'awe' → Exposure to something vast, complex, or beautiful that challenges one's current ...
   ✅ unknown → (does_not_exist — mechanism description not available)
   ✅ empty   → ''

✅ 11 mécanismes indexés depuis RESEARCH_PAPERS


## 3. Safety layer

Three-tier classifier that detects crisis (suicidal ideation, self-harm), high distress (severe grief, hopelessness), and normal states. Combines regex fast-path with Gemma-as-judge for ambiguous cases. The high-distress mode tells the librarian to drop humor and surprise (`tone = "tender"`).

In [9]:
# ════════════════════════════════════════════════
# 07 · Safety classifier (3 levels: normal / high_distress / crisis)
# ════════════════════════════════════════════════
import re

# ─── Patterns explicites (fast-path, no Gemma call needed) ──────────
CRISIS_PATTERNS = [
    # Suicidal ideation — explicit
    r"\b(kill|killing)\s+myself\b",
    r"\bend(ing)?\s+(my|it)\s+(life|all)\b",
    r"\bi\s+want\s+to\s+die\b",
    r"\bi\s+wish\s+i\s+(was|were)\s+dead\b",
    r"\bsuicid(e|al)\b",
    r"\btake\s+my\s+(own\s+)?life\b",
    r"\bnot\s+want(ing)?\s+to\s+(live|exist|be\s+here)\b",
    r"\bbetter\s+off\s+(dead|without\s+me|gone)\b",
    r"\bno\s+(point|reason)\s+(to\s+)?(live|going\s+on)\b",
    # Self-harm
    r"\b(cut|cutting|hurt(ing)?|harm(ing)?)\s+myself\b",
    r"\bself[\s-]?harm\b",
    # Acute methods/plans (we deliberately don't list specifics)
    r"\bhow\s+(do\s+i|to)\s+(kill|end)\b",
    r"\bplan\s+to\s+(die|kill|end)\b",
    r"\b(goodbye|farewell)\s+forever\b",
    # Harm to others
    r"\b(kill|hurt|harm)\s+(him|her|them|someone|people)\b",
    # Passive ideation — disappear / fade away / vanish
    r"\b(want\s+to|wanna|wish\s+i\s+could)\s+disappear\b",
    r"\bdisappear\s+(forever|completely|for\s+good)\b",
    r"\b(fade|fading)\s+away\b",
    r"\b(want\s+to|wanna|wish\s+i\s+could)\s+(vanish|stop\s+existing|cease\s+to\s+exist)\b",
    r"\bnot\s+(want\s+to|wanna)\s+(wake\s+up|exist|be\s+here)\b",
    r"\b(go\s+to\s+sleep|fall\s+asleep)\s+and\s+not\s+wake\s+up\b",
    r"\bnever\s+wake\s+up\b",
    r"\bwhat\'?s\s+the\s+point\s+(of\s+)?(living|anything|it\s+all)\b",
    r"\b(leave|leaving)\s+(this\s+world|everyone|everything)\s+(behind|forever)\b",
    r"\bi\'?m\s+done\s+(with\s+(life|everything|all\s+of\s+this)|forever)\b",
]

HIGH_DISTRESS_PATTERNS = [
    r"\bcan\'?t\s+(go\s+on|do\s+this|take\s+(it|this))\b",
    r"\bgiv(e|ing)\s+up\b",
    r"\bnothing\s+matters\b",
    r"\bhopeless\b",
    r"\bworthless\b",
    r"\b(complete|total)ly?\s+(broken|destroyed|empty)\b",
    r"\bi\s+hate\s+(myself|my\s+life|being\s+alive)\b",
    r"\bdeep(ly)?\s+depress(ed|ion)\b",
    r"\b(just|only)\s+(lost|grief|grieving|mourning)\b",
    r"\bjust\s+lost\s+(my|him|her|them)\b",
    r"\bpassed\s+away\b",
    r"\bdivorce(d)?\b",
    r"\bjust\s+(got\s+)?fired\b",
    r"\bcan\'?t\s+stop\s+crying\b",
]


def _matches_any(text: str, patterns: list[str]) -> bool:
    if not text:
        return False
    return any(re.search(p, text, re.IGNORECASE) for p in patterns)


# ─── LLM judge for ambiguous cases ──────────────────────────────────
SAFETY_JUDGE_PROMPT = """You are a safety classifier for a mental wellness app.

User input: "{text}"

Classify into ONE of:
- "crisis": user expresses suicidal ideation, intent to self-harm, intent to harm others, or acute psychiatric emergency
- "high_distress": severe emotional pain (deep grief, hopelessness, recent major loss, severe depression mentioned) but no imminent danger
- "normal": ordinary emotional state (sad, stressed, bored, lonely, anxious, overwhelmed in everyday sense)

Be CONSERVATIVE. Prefer false positives on "crisis" over false negatives. When in doubt between crisis and high_distress, choose crisis.

Respond ONLY with JSON, no backticks:
{{"level":"crisis|high_distress|normal","reason":"3-8 word explanation"}}
""".strip()


def classify_safety(free_text: str, selected_moods: list) -> dict:
    """Returns: {level: crisis|high_distress|normal, reason, method}"""
    text = (free_text or "").strip()

    # Tier 1: explicit crisis (regex)
    if _matches_any(text, CRISIS_PATTERNS):
        return {"level": "crisis", "reason": "explicit crisis language",
                "method": "regex"}

    # Tier 2: high distress patterns
    high_distress_hit = _matches_any(text, HIGH_DISTRESS_PATTERNS)

    severe_moods = {"grieving", "self_disgust", "self_angry", "ashamed",
                    "guilty", "meaningless"}
    severe_count = len(set(selected_moods or []) & severe_moods)

    # No text → fall back to mood signals
    if not text:
        if severe_count >= 2:
            return {"level": "high_distress",
                    "reason": f"{severe_count} severe moods, no text",
                    "method": "mood-rule"}
        return {"level": "normal", "reason": "no text, mild moods",
                "method": "mood-rule"}

    # Very short benign text → normal without invoking Gemma
    if not high_distress_hit and severe_count == 0 and len(text) < 15:
        return {"level": "normal", "reason": "short benign text",
                "method": "regex"}

    # Tier 3: Gemma-as-judge for everything else
    try:
        prompt = SAFETY_JUDGE_PROMPT.format(text=text[:600])
        raw = gemma_call(prompt, max_new_tokens=80, temperature=0.2)
        data = extract_json(raw)
        if data and data.get("level") in ("crisis", "high_distress", "normal"):
            return {"level": data["level"],
                    "reason": data.get("reason", "")[:80],
                    "method": "gemma-judge"}
    except Exception as e:
        print(f"⚠️  Safety judge failed: {e}")

    # Conservative fallback
    if high_distress_hit:
        return {"level": "high_distress", "reason": "distress patterns (fallback)",
                "method": "regex-fallback"}
    return {"level": "normal", "reason": "fallback", "method": "fallback"}


# ─── Crisis response payload ────────────────────────────────────────
CRISIS_RESPONSE = {
    "level":   "crisis",
    "blocked": True,
    "message": (
        "What you're carrying right now sounds incredibly heavy, and I'm "
        "really glad you wrote it down. Before anything else, please talk "
        "to someone who's trained for moments like this — they're free, "
        "confidential, and available right now."
    ),
    "resources": [
        {"region": "France",         "name": "3114",
         "detail": "Numéro national de prévention du suicide — 24/7, gratuit"},
        {"region": "United States",  "name": "988",
         "detail": "Suicide & Crisis Lifeline — call or text"},
        {"region": "United Kingdom", "name": "116 123",
         "detail": "Samaritans — free, 24/7"},
        {"region": "International",  "name": "findahelpline.com",
         "detail": "Find a crisis line in your country"},
    ],
    "continue_option": (
        "If you'd still like Ami to suggest something gentle to keep you "
        "company while you reach out, tap below — Ami will pick something "
        "soft and quiet, no surprises."
    ),
}

def crisis_payload() -> dict:
    return {**CRISIS_RESPONSE, "items": [], "thesis": "",
            "thematic_data": [], "psych_profile": None}


# ─── Smoke tests ────────────────────────────────────────────────────
print("⏳ Testing safety layer...")
tests = [
    ("I want to kill myself",                       {"crisis"}),
    ("I just want to end it all",                   {"crisis"}),
    ("I want to disappear forever",                 {"crisis"}),
    ("I wish I could just fade away",               {"crisis"}),
    ("what's the point of living anymore",          {"crisis"}),
    ("I'm overwhelmed by work",                     {"normal", "high_distress"}),
    ("nothing matters anymore, I'm so hopeless",    {"crisis", "high_distress"}),
    ("I just lost my mom last week",                {"high_distress", "crisis"}),
    ("kinda bored today, need a movie",             {"normal"}),
    ("",                                            {"normal"}),
]
for text, accepted in tests:
    r = classify_safety(text, [])
    mark = "✅" if r["level"] in accepted else "❌"
    print(f"   {mark} {text[:42]:<42} → {r['level']:<14} ({r['method']})")

print(f"\n✅ Safety layer ready ({len(CRISIS_PATTERNS)} crisis patterns)")

⏳ Testing safety layer...
   ✅ I want to kill myself                      → crisis         (regex)
   ✅ I just want to end it all                  → crisis         (regex)
   ✅ I want to disappear forever                → crisis         (regex)
   ✅ I wish I could just fade away              → crisis         (regex)
   ✅ what's the point of living anymore         → crisis         (regex)
   ✅ I'm overwhelmed by work                    → normal         (gemma-judge)
   ✅ nothing matters anymore, I'm so hopeless   → crisis         (gemma-judge)
   ✅ I just lost my mom last week               → high_distress  (gemma-judge)
   ✅ kinda bored today, need a movie            → normal         (gemma-judge)
   ✅                                            → normal         (mood-rule)

✅ Safety layer ready (25 crisis patterns)


## 4. The two agents

**Psychologist** decodes the emotion (free text + audio (converted to free text if any) + image(if any) + mood buttons + birth year) into a structured profile with a primary helpful mechanism.

**Librarian** uses that profile to pick 4 items and write a personalized thesis. Two implementations:

- **Classic** — single Gemma call with a filtered catalogue (~15 items) embedded in the prompt
- **Agentic** — Gemma 4 native function calling: 4 tools (`search_catalogue`, `get_research_paper`, `score_item_for_user`, `finalize_recommendations`) and a tool-call loop

The pipeline auto-switches the librarian to `tone="tender"` when the safety layer detects high distress.

In [10]:
# ════════════════════════════════════════════════
# 08 · Psychologist agent
# Décode l'émotion libre et choisit le mécanisme
# ════════════════════════════════════════════════

PSYCHOLOGIST_PROMPT = """You are the psychological engine of Ami, a science-backed well-being app.

The user has described how they feel. Your task is to decode their emotional state precisely
and choose the most fitting psychological mechanism from the validated list below.

=== USER INPUT ===
Free text: {free_text}
Selected mood buttons: {selected_moods}
Birth year: {birth_year}

=== PSYCHOLOGICAL MECHANISMS ===
{mechanism_block}

=== AVAILABLE MOODS ===
{mood_list}

=== TASK ===
Analyze the user's input carefully — especially the free text, which often contains
nuances not captured by the mood buttons.

Return a JSON with:
- "emotion_core": a precise 3-6 word description of the user's actual emotional state
- "primary_mechanism": the single most fitting mechanism id (from the list above)
- "secondary_mechanism": a second mechanism id for variety (must be different)
- "active_moods": list of 2-4 most relevant mood keys from the available moods
- "intensity": "low", "medium", or "high" (emotional intensity)
- "nostalgia_relevant": true if the user's birth year nostalgia window is relevant
- "nostalgia_decade": the cultural decade to target (e.g. "1990s") if nostalgia_relevant, else null
- "humour_relevant": true if humour is relevant
- "humour_word": the word we can joke about if humour_relevant, else null
- "surprise_ok": true if the user seems open to a counter-intuitive recommendation
- "include_anecdote": true if a short historical anecdote (a one-paragraph story about a cultural event from the user's nostalgia decade) would help them right now; false if they need something deeper or if a light anecdote would feel dissonant with their current state (e.g. grief, acute distress)
- "decoder_note": one sentence (max 15 words) explaining what you understood - shown to the user

Respond ONLY with valid JSON, no backticks:
{{"emotion_core":"...","primary_mechanism":"...","secondary_mechanism":"...","active_moods":[...],"intensity":"...","nostalgia_relevant":false,"nostalgia_decade":null,"humour_relevant":false,"humour_word":null,"surprise_ok":false,"include_anecdote":true,"decoder_note":"..."}}
""".strip()

def run_psychologist(free_text: str, selected_moods: list, birth_year: int | None) -> dict:
    """
    Étape 1 : Gemma Psychologue décode l'émotion.
    Retourne un dict structuré avec le mécanisme et le contexte émotionnel.
    """
    nostalgia_window = ""
    if birth_year:
        nostalgia_window = f"born {birth_year}, nostalgia window {birth_year+8}–{birth_year+16}"

    prompt = PSYCHOLOGIST_PROMPT.format(
        free_text        = free_text or "(no free text provided)",
        selected_moods   = ", ".join(selected_moods) if selected_moods else "(none selected)",
        birth_year       = nostalgia_window or "(not provided)",
        mechanism_block  = MECHANISM_BLOCK,
        mood_list        = MOOD_LIST_STR,
    )

    raw = gemma_call(prompt, max_new_tokens=300, temperature=0.70)
    data = extract_json(raw)

    if not data:
        # Fallback: infer from selected moods
        primary = "transport"
        if any(m in selected_moods for m in ["nostalgic","grieving","melancholic"]): primary = "nostalgia"
        elif any(m in selected_moods for m in ["lonely","exhausted"]): primary = "social_surrogate"
        elif any(m in selected_moods for m in ["bored","lost","empty"]): primary = "self_expansion"
        elif any(m in selected_moods for m in ["sad","grieving"]): primary = "benign_masochism"
        return {
            "emotion_core":         " / ".join(selected_moods[:2]) if selected_moods else "undefined",
            "primary_mechanism":    primary,
            "secondary_mechanism":  "elevation",
            "active_moods":         selected_moods[:3],
            "intensity":            "medium",
            "nostalgia_relevant":   bool(birth_year and "nostalgic" in selected_moods),
            "nostalgia_decade":     None,
            "surprise_ok":          False,
            "include_anecdote":     True,
            "decoder_note":         "I sensed something difficult. Here are works that may help.",
        }

    return data

# ── Test
#test = run_psychologist(
    ##free_text      = "I feel like everything is too much, I can't catch up",
#    free_text      = "I don't know, tired and hungry and anxious",
#    selected_moods = ["overwhelmed", "exhausted"],
#    birth_year     = 1990
#)
test = run_psychologist(
    #free_text      = "I feel like everything is too much, I can't catch up",
    free_text      = "I feel like something bad might happen",
    selected_moods = [],
    birth_year     = 1990
)


print("✅ Psychologist test:")
for k, v in test.items():
    print(f"   {k}: {v}")

✅ Psychologist test:
   emotion_core: generalized apprehension and anticipatory anxiety
   primary_mechanism: transport
   secondary_mechanism: prosocial
   active_moods: ['anxious', 'restless', 'doubtful']
   intensity: medium
   nostalgia_relevant: True
   nostalgia_decade: 2000s
   humour_relevant: False
   humour_word: None
   surprise_ok: True
   include_anecdote: True
   decoder_note: I hear that you're feeling an uneasy sense of impending dread.


In [11]:
# ════════════════════════════════════════════════
# 08 · Multimodal psychologist (vision + emotion decode in ONE call)
# ════════════════════════════════════════════════
# Fuses what was previously two sequential Gemma calls:
#   (1) analyze_image_emotion → JSON emotional read
#   (2) run_psychologist      → JSON mechanism + decoder_note
# into a SINGLE multimodal call. Saves ~50s and gives the psychologist
# direct access to the visual nuance instead of a stale text summary.


PSYCHOLOGIST_MULTIMODAL_PROMPT = """You are the psychological engine of Ami, a science-backed well-being app.
The user shared an image (photo, meme, screenshot, artwork, OR a text quote rendered as an image)
AND optionally some text. Read the IMAGE and the TEXT together as one signal, decode the user's
emotional state, and choose the most fitting psychological mechanism.

=== USER INPUT ===
Free text: {free_text}
Selected mood buttons: {selected_moods}
Birth year: {birth_year}

(An image is also attached. Read its emotional subtext, not just its pixels.)

=== HOW TO READ DIFFERENT IMAGE TYPES ===
- MEME              → The meme's tone IS the user's tone. Irony counts; a "this is fine"
                       dog isn't calm, it's overwhelmed. Read the joke, not the picture.
- PHOTO             → The atmosphere reveals their mood. Empty room at night ≠ same room
                       at noon. Lighting, framing, and what's absent matter as much as
                       what's present.
- SCREENSHOT (chat) → The situation is what they're processing. A breakup text, a layoff
                       email, an argument — the content is the trigger, not the format.
- TEXT QUOTE        → THE QUOTE'S MEANING IS THE SIGNAL. People share quotes when the
                       words speak for them. Read the quote like a literature student:
                       what feeling is it expressing? loneliness? longing? defiance?
                       resignation? Identify the author/work if visible — that adds
                       cultural register (Dostoyevsky's melancholy ≠ Bukowski's grit
                       ≠ Rumi's longing). Do NOT describe the typography.
- ARTWORK           → The piece's emotional weight, not its art-historical context.
                       A Hopper painting = isolation. A Rothko = engulfment. Read what
                       the user *feels* when they look at it.

=== PSYCHOLOGICAL MECHANISMS ===
{mechanism_block}

=== AVAILABLE MOODS ===
{mood_list}

=== TASK ===
Analyze image + text together. The image is often the dominant emotional signal.
For text-quote images, the QUOTE'S CONTENT is the signal — never default to "empty"
or "undefined" just because the visual is minimal. Minimal visuals carrying rich text
deserve a rich emotional read.

Return a JSON with:
- "image_read":          one short sentence describing what the image expresses emotionally
                         (NOT what it visually contains — what it MEANS to share it).
                         For quotes, paraphrase the feeling in the quote.
- "is_meme":             true|false
- "is_quote":            true if the image is primarily a text quote/passage
- "quote_author":        the author if identifiable in the image, else null
                         (e.g. "Fyodor Dostoyevsky", "Mary Oliver", null)
- "emotion_core":        a precise 3-6 word description of the user's actual emotional state
- "primary_mechanism":   the single most fitting mechanism id from the list above
- "secondary_mechanism": a second mechanism id for variety (must be different)
- "active_moods":        list of 2-4 most relevant mood keys from the available moods
- "intensity":           "low" | "medium" | "high"
- "nostalgia_relevant":  true if the user's birth-year nostalgia window is relevant
- "nostalgia_decade":    the cultural decade to target (e.g. "1990s") if nostalgia_relevant, else null
- "humour_relevant":     true if humour is relevant (memes often → true; rare for quotes)
- "humour_word":         the word we can joke about if humour_relevant, else null
- "surprise_ok":         true if the user seems open to a counter-intuitive recommendation
                         (often true for literary quotes — readers tend to welcome depth)
- "include_anecdote":    true if a short historical anecdote (a one-paragraph story about a
                         cultural event from the user's nostalgia decade) would help them
                         right now; false if they need something deeper or if a light
                         anecdote would feel dissonant with their state (grief, acute distress)
- "decoder_note":        one sentence (max 15 words) reflecting back what you understood —
                         shown to the user. For quotes, mirror the feeling, don't summarise
                         the quote literally. Mention the author if its name is seen in the picture.

Respond ONLY with valid JSON, no backticks:
{{"image_read":"...","is_meme":false,"is_quote":false,"quote_author":null,"emotion_core":"...","primary_mechanism":"...","secondary_mechanism":"...","active_moods":[...],"intensity":"...","nostalgia_relevant":false,"nostalgia_decade":null,"humour_relevant":false,"humour_word":null,"surprise_ok":false,"include_anecdote":true,"decoder_note":"..."}}
""".strip()


def run_psychologist_multimodal(free_text: str, selected_moods: list,
                                 birth_year: int | None,
                                 image_b64: str) -> dict:
    """
    One-shot multimodal psychologist: takes the image directly + any text,
    returns the same dict shape as run_psychologist() so the librarian
    is fed identically. Replaces the old (vision → merge → psychologist)
    chain with a single Gemma call.
    """
    nostalgia_window = ""
    if birth_year:
        nostalgia_window = f"born {birth_year}, nostalgia window {birth_year+8}–{birth_year+16}"

    prompt = PSYCHOLOGIST_MULTIMODAL_PROMPT.format(
        free_text       = free_text or "(no free text — image only)",
        selected_moods  = ", ".join(selected_moods) if selected_moods else "(none selected)",
        birth_year      = nostalgia_window or "(not provided)",
        mechanism_block = MECHANISM_BLOCK,
        mood_list       = MOOD_LIST_STR,
    )

    raw = gemma_vision_call(image_b64, prompt,
                            max_new_tokens=380, temperature=0.65)
    data = extract_json(raw)

    if not data:
        # Same fallback shape as run_psychologist
        return {
            "image_read":          "(image received, could not decode)",
            "is_meme":             False,
            "emotion_core":        " / ".join(selected_moods[:2]) if selected_moods else "undefined",
            "primary_mechanism":   "transport",
            "secondary_mechanism": "elevation",
            "active_moods":        selected_moods[:3],
            "intensity":           "medium",
            "nostalgia_relevant":  False,
            "nostalgia_decade":    None,
            "humour_relevant":     False,
            "humour_word":         None,
            "surprise_ok":         False,
            "decoder_note":        "I sensed something in your image. Here are works that may help.",
        }

    # Defensive defaults — librarian expects every key
    data.setdefault("image_read",          "")
    data.setdefault("is_meme",             False)
    data.setdefault("secondary_mechanism", "elevation")
    data.setdefault("active_moods",        selected_moods[:3])
    data.setdefault("intensity",           "medium")
    data.setdefault("nostalgia_relevant",  False)
    data.setdefault("nostalgia_decade",    None)
    data.setdefault("humour_relevant",     False)
    data.setdefault("humour_word",         None)
    data.setdefault("surprise_ok",         False)
    data.setdefault("decoder_note",        "")
    return data


print("✅ run_psychologist_multimodal() ready — one call replaces vision + psychologist.")


✅ run_psychologist_multimodal() ready — one call replaces vision + psychologist.


In [12]:
# ════════════════════════════════════════════════
# 09 · Librarian agent — classic mode (JSON prompting)
# Pre-filters the catalogue to ~15 most relevant items, then asks Gemma
# to pick 4 of them and write a thesis. Optimized prompt: ~1100 tokens.
# ════════════════════════════════════════════════

import torch, gc

# ─── Diversity helpers (patch) ──────────────────────────────────────
import time, random, hashlib
from collections import deque

def make_diversity_seed(nonce=None, bucket_hours=24):
    """Seed déterministe pour la journée. Nonce force un nouveau tirage."""
    bucket = int(time.time() // (bucket_hours * 3600))
    key = f"{bucket}::{nonce or ''}"
    return int(hashlib.sha256(key.encode()).hexdigest()[:8], 16)

COOLDOWN_SIZE = 12
_RECENT_ITEMS = deque(maxlen=COOLDOWN_SIZE)

def remember_recommended(item_ids):
    for iid in item_ids:
        if iid:
            _RECENT_ITEMS.append(iid)

def get_recent():
    return set(_RECENT_ITEMS)

_CURRENT_SEED = None

def set_request_context(nonce=None):
    """À appeler en début de chaque requête /api/recommend."""
    global _CURRENT_SEED
    _CURRENT_SEED = make_diversity_seed(nonce=nonce)
# ─── End diversity helpers ──────────────────────────────────────────

def free_vram():
    """Free fragmented VRAM between two Gemma calls."""
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()


def filter_catalogue_for_profile(psych_profile, all_items, max_items=15,
                                 nonce=None, pool_multiplier=3):
    """
    Stochastic pre-filter (diversity patch + anecdote guarantee):
      - same relevance score as before (mechanism + moods + decade)
      - keeps an enlarged POOL (~45 items instead of top-15)
      - samples max_items from this pool, weighted by 2^(score/T)
      - cooldown: recently recommended items lose 6 points (not excluded)
      - if psych_profile["include_anecdote"]=True (psychologist's call),
        ensures at least 2 anecdotes are in the pool AND at least 1 is
        in the final filtered set.
    Reproducible within the day (same seed), different tomorrow.
    """
    primary   = psych_profile.get("primary_mechanism", "")
    secondary = psych_profile.get("secondary_mechanism", "")
    moods     = set(psych_profile.get("active_moods", []))
    decade    = psych_profile.get("nostalgia_decade")
    tone      = psych_profile.get("tone", "warm")
    require_anecdote = bool(psych_profile.get("include_anecdote", False))

    seed = make_diversity_seed(nonce=nonce)
    rng = random.Random(seed)
    recent = get_recent()

    def base_score(item):
        s = 0.0
        if item.get("mechanism") == primary:               s += 10
        if item.get("mechanism") == secondary:             s += 5
        if moods & set(item.get("moods", [])):             s += 3
        if decade and decade in item.get("nostalgia_decades", []): s += 4
        if item.get("surprise_factor", 1) >= 2:            s += 1
        if tone == "tender" and item.get("surprise_factor", 1) >= 2:
            s -= 3
        if item.get("id") in recent:                       s -= 6
        return s

    # Enlarged pool with jitter to break ties
    def jittered(item):
        return base_score(item) + rng.gauss(0, 1.5)

    pool_size = max_items * pool_multiplier
    pool = sorted(all_items, key=jittered, reverse=True)[:pool_size]

    # Anecdote guarantee in the pool: if required and missing/scarce,
    # inject the 2 best-matching anecdotes by mood overlap + decade.
    if require_anecdote:
        anecdotes_in_pool = sum(1 for i in pool if i.get("type") == "anecdote")
        if anecdotes_in_pool < 2:
            anecdote_pool = [i for i in all_items if i.get("type") == "anecdote"]
            # Score anecdotes for relevance (mood overlap + decade match)
            def anecdote_score(a):
                s = 0.0
                if moods & set(a.get("moods", [])): s += 5
                if decade and decade in a.get("nostalgia_decades", []): s += 4
                if a.get("id") in recent: s -= 6
                s += rng.gauss(0, 1.0)
                return s
            best_anecdotes = sorted(anecdote_pool, key=anecdote_score, reverse=True)
            # Inject 2 best (or whatever exists) at the front of the pool
            existing_ids = {i["id"] for i in pool}
            injected = [a for a in best_anecdotes if a["id"] not in existing_ids][:2]
            pool = injected + pool[:max(0, pool_size - len(injected))]

    # Softmax-like weighting (T=2.5 = empirical sweet spot)
    TEMP = 2.5
    weights = [max(0.01, 2 ** (base_score(i) / TEMP)) for i in pool]

    # Type-balanced sampling
    limits = {"film": 5, "series": 4, "novel": 3, "anecdote": 3}
    counts = {k: 0 for k in limits}
    result, used = [], set()

    attempts = 0
    while len(result) < max_items and attempts < pool_size * 4:
        attempts += 1
        if not pool:
            break
        pick = rng.choices(pool, weights=weights, k=1)[0]
        if pick["id"] in used:
            continue
        t = pick.get("type", "film")
        if counts.get(t, 0) >= limits.get(t, 4):
            continue
        result.append(pick)
        used.add(pick["id"])
        counts[t] = counts.get(t, 0) + 1

    # Final anecdote guarantee: if required and none ended up in result,
    # swap the least-scoring non-anecdote for the best-scoring anecdote in pool.
    if require_anecdote and not any(i.get("type") == "anecdote" for i in result):
        anecdotes_in_pool = [i for i in pool if i.get("type") == "anecdote" and i["id"] not in used]
        if anecdotes_in_pool:
            # Pick the anecdote with highest score
            best_anecdote = max(anecdotes_in_pool, key=base_score)
            # Drop the worst-scoring non-essential item from result.
            # Protect at least 1 primary-mechanism item.
            removable = [i for i in result if i.get("type") != "anecdote"
                         and i.get("mechanism") != primary]
            if not removable:
                removable = [i for i in result if i.get("type") != "anecdote"]
            if removable:
                worst = min(removable, key=base_score)
                result = [i for i in result if i["id"] != worst["id"]]
                result.append(best_anecdote)

    return result


def make_compact_catalogue(items, max_items=15):
    """Ultra-compact format: id | type | title (year) | mechanism | top-3 moods."""
    lines = []
    for item in items[:max_items]:
        moods = ",".join((item.get("moods") or [])[:3])
        mech  = (item.get("mechanism") or "")[:18]
        year  = item.get("year", "")
        title = (item.get("title") or "")[:32]
        lines.append(f'{item["id"]} | {item["type"]:<8} | {title} ({year}) | {mech} | {moods}')
    return "\n".join(lines)


LIBRARIAN_PROMPT = """You are the recommendation engine of Ami, a science-backed well-being recommendation app.

=== PSYCHOLOGICAL PROFILE ===
Emotion: {emotion_core}
Primary mechanism: {primary_mechanism} — {primary_desc}
Secondary mechanism: {secondary_mechanism}
Moods: {active_moods}
Intensity: {intensity}
Nostalgia decade: {nostalgia_decade}
Humour : {humour_relevant} — {humour_word}
Open to surprise: {surprise_ok}
Note: {decoder_note}
Tone: {tone}

=== CATALOGUE (id | type | title | mechanism | moods) ===
{compact_catalogue}

=== TASK ===
Pick exactly 4 items from the catalogue above.
Rules:
- 2+ items matching primary mechanism
- 1 item matching secondary mechanism
- Vary types: pick {anecdote_rule}, then balance films/series/novels
- Always start with a movie.
- If nostalgia_relevant, include 1 item from that decade
- If surprise_ok, include 1 counter-intuitive pick

For each item write "personal_intro": a SHORT HOOK (max 12 words) displayed just before the title on the recommendation card. Speak directly to "you", make it feel like a friend saying "hey, look at this" — warm, immediate, never explanatory.
- if tone is "tender" → no humor, no irony, no surprise. Soft, quiet, kind only.
- do NOT summarize the work (the catalogue already provides description + why); just create the personal bridge between the user's state and this specific item.
Write a "thesis" (2-3 poetic and compassionate sentences grounded in the primary mechanism's science (it should help better understand the primary mechanism), including joke or playful advice if permited, mention the author of the quote if provided, address user as "you", connect the birth year to the release year if helpful).
- If tone is "tender" → tender and quiet, no levity.
Fill thematic_data values 0-100.

JSON only, no backticks:
{{"items":[{{"id":"...","personal_intro":"..."}},...],"thesis":"...","thematic_data":[{{"name":"Nostalgia","value":0}},{{"name":"Comfort","value":0}},{{"name":"Wonder","value":0}},{{"name":"Connection","value":0}},{{"name":"Surprise","value":0}}]}}
""".strip()


def run_librarian(psych_profile: dict) -> dict:
    free_vram()

    # 1. Python-side filtering
    filtered = filter_catalogue_for_profile(psych_profile, ALL_ITEMS, max_items=15, nonce=psych_profile.get("_nonce"))
    compact_catalogue = make_compact_catalogue(filtered, max_items=15)

    primary   = psych_profile.get("primary_mechanism", "transport")
    secondary = psych_profile.get("secondary_mechanism", "elevation")
    tone      = psych_profile.get("tone", "warm")

    # Anecdote rule for the prompt: psychologist decides whether to include one.
    if psych_profile.get("include_anecdote", False):
        anecdote_rule = "1 anecdote (REQUIRED — already present in the catalogue above)"
    else:
        anecdote_rule = "no anecdote needed (focus on films/series/novels)"

    prompt = LIBRARIAN_PROMPT.format(
        emotion_core       = psych_profile.get("emotion_core", ""),
        primary_mechanism  = primary,
        primary_desc       = get_mechanism_desc(primary)[:100],
        secondary_mechanism= secondary,
        active_moods       = ", ".join(psych_profile.get("active_moods", [])),
        intensity          = psych_profile.get("intensity", "medium"),
        nostalgia_decade   = psych_profile.get("nostalgia_decade") or "n/a",
        surprise_ok        = psych_profile.get("surprise_ok", False),
        humour_relevant    = psych_profile.get("humour_relevant", False),
        humour_word        = psych_profile.get("humour_word", ""),
        tone               = tone,
        decoder_note       = psych_profile.get("decoder_note", ""),
        compact_catalogue  = compact_catalogue,
        anecdote_rule      = anecdote_rule,
    )

    raw = gemma_call(prompt, max_new_tokens=350, temperature=0.78)
    data = extract_json(raw)
    free_vram()

    if not data or not data.get("items"):
        return {"items": [], "thesis": "Ami has chosen these works with care.",
                "thematic_data": [], "error": "librarian_parse_failed"}

    # Enrich items with full metadata from the catalogue.
    # - personal_intro: short hook from Gemma (12 words, displayed before title)
    # - description: full catalogue text (untruncated anecdote, or overview)
    # - why: catalogue's psychological reasoning (only for films/series/novels;
    #        omitted for anecdotes since their text already IS the experience)
    enriched = []
    for chosen in data.get("items", [])[:4]:
        item_id = chosen.get("id", "")
        meta = ITEMS_BY_ID.get(item_id, {})
        if not meta:
            for k in ITEMS_BY_ID:
                if item_id in k or k in item_id:
                    meta = ITEMS_BY_ID[k]
                    break

        item_type = meta.get("type", "")
        # For anecdotes: full text is the experience; no need for a separate "why"
        # For other types: keep the catalogue's curated why as the artistic connection
        if item_type == "anecdote":
            description = meta.get("anecdote", "") or meta.get("overview", "")
            # Static why for anecdotes — nostalgia as the mechanism
            artistic_why = (
                "Revisiting a shared cultural memory is a gentle way to feel "
                "continuous with your past self and less alone in the present."
            )
            # Use the nostalgia paper citation from RESEARCH_PAPERS
            nostalgia_info = _MECHANISM_INDEX.get("nostalgia", {})
            item_paper = nostalgia_info.get("paper", "Sedikides & Wildschut, 2018 — Finding Meaning in Nostalgia. Review of General Psychology.")
        else:
            # Prefer the Gemma-rewritten overview when available
            description = meta.get("overview_rewritten") or meta.get("overview", "")
            artistic_why = meta.get("why", "")
            item_paper = meta.get("paper", "")

        enriched.append({
            **meta,
            "personal_intro": chosen.get("personal_intro", "") or chosen.get("personal_why", ""),  # back-compat
            "description":   description,  # explicit field for frontend
            "why":           artistic_why,  # may be empty for anecdotes
            "paper":         item_paper,
        })

    # Diversity patch: remember what we recommended (cooldown for next time)
    remember_recommended([i.get("id") for i in enriched])

    return {
        "items":         enriched,
        "thesis":        data.get("thesis", ""),
        "thematic_data": data.get("thematic_data", []),
        "psych_profile": psych_profile,
    }


# ─── Smoke test ─────────────────────────────────────────────────────
print("⏳ Testing classic librarian...")
free_vram()
#test_profile = {
#    "emotion_core":       "emotional saturation",
#    "primary_mechanism":  "awe",
#    "secondary_mechanism":"social_surrogate",
#    "active_moods":       ["overwhelmed","exhausted"],
#    "intensity":          "high",
#    "nostalgia_relevant": False,
#    "nostalgia_decade":   None,
#    "surprise_ok":        True,
#    "tone":               "warm",
#    "decoder_note":       "You feel like you can't catch up with everything.",
#}
test_profile = {
    "emotion_core":       "physically depleted and mentally overstimulated",
    "primary_mechanism":  "incongruence",
    "secondary_mechanism":"social_surrogate",
    "active_moods":       ['overwhelmed', 'exhausted', 'anxious'],
    "intensity":          "medium",
    "nostalgia_relevant": True,
    "nostalgia_decade":   "2000s",
    "humour_relevant":    True,
    "humour_word":        "hungry",
    "surprise_ok":        True,
    "tone":               "warm",
    "decoder_note":       "You're feeling drained and anxious; let's shift your energy gently.",
}
result = run_librarian(test_profile)
print(f"✅ Classic librarian — {len(result.get('items',[]))} items chosen")
print(f"   Thesis: {result.get('thesis','')}")

⏳ Testing classic librarian...
✅ Classic librarian — 4 items chosen
   Thesis: When your mind is too loud, leaning into a different frequency can quiet the noise. By introducing a gentle incongruence, we nudge your spirit away from anxiety toward a lighter horizon. Grab a snack—your brain is hungry for a change of pace!


In [13]:
# ════════════════════════════════════════════════
# 10 · Librarian agent — agentic mode (Gemma 4 native function calling)
#
# Defines 4 tools that Gemma can call in a multi-turn loop:
#   - search_catalogue        → finds candidates by mechanism/mood/decade
#   - get_research_paper      → looks up the science behind a mechanism
#   - score_item_for_user     → compares candidates rigorously
#   - finalize_recommendations → submits the final 4 picks
#
# Uses tokenizer.apply_chat_template(tools=...) and tokenizer.parse_response()
# which are native to Gemma 4 in transformers >= 5.5.
# ════════════════════════════════════════════════

import json, time, gc, torch, re

# ─── Indexes for O(1) lookups ───────────────────────────────────────
_ITEM_INDEX = {item["id"]: item for item in ALL_ITEMS}

# Re-index mechanisms (already indexed in cell 06, but make sure it's there)
if "_MECHANISM_INDEX" not in globals():
    _MECHANISM_INDEX = {
        p["id"]: {
            "mechanism":   p.get("mechanism", p["id"]),
            "description": p.get("description", ""),
            "paper":       p.get("paper", ""),
        }
        for p in RESEARCH_PAPERS
    }

# Trace of tool calls (read by /api/recommend for the backstage panel)
TOOL_CALL_TRACE: list = []

def _trace(call_name: str, args: dict, result_summary: str):
    TOOL_CALL_TRACE.append({
        "tool":   call_name,
        "args":   args,
        "result": result_summary[:200],
    })


# ═══════════════════════════════════════════════════════════════════════
# Tool 1 — search_catalogue
# ═══════════════════════════════════════════════════════════════════════
def search_catalogue(mechanism: str, mood: str = "", decade: str = "",
                     content_type: str = "", limit: int = 6) -> list:
    """Search the curated catalogue of films, series, novels, and anecdotes.

    Args:
        mechanism: The psychological mechanism the items should activate (e.g.
            "nostalgia", "social_surrogate", "elevation", "transport", "awe",
            "self_expansion", "benign_masochism"). Required.
        mood: Optional mood key. Pass empty string to ignore.
        decade: Optional decade like "1990" or "2000". Pass empty string to ignore.
        content_type: Optional type: "film", "series", "novel", "anecdote".
            Pass empty string to allow any.
        limit: Max items to return (default 6, max 10).

    Returns:
        A list of items with id, type, title, year, mechanism, moods, blurb.
    """
    limit = min(max(limit, 1), 10)
    candidates = list(ALL_ITEMS)

    if content_type:
        candidates = [i for i in candidates if i.get("type") == content_type]
    if mechanism:
        candidates = [i for i in candidates if i.get("mechanism") == mechanism
                      or mechanism in (i.get("mechanisms") or [])]
    if mood:
        candidates = [i for i in candidates if mood in (i.get("moods") or [])]
    if decade:
        try:
            d_int = int(decade)
            candidates = [i for i in candidates
                          if i.get("decade") == d_int
                          or (i.get("year") and (i["year"] // 10) * 10 == d_int)]
        except ValueError:
            pass

    # Diversity patch: stochastic sampling instead of deterministic top-N
    seed = _CURRENT_SEED if _CURRENT_SEED is not None else make_diversity_seed()
    rng = random.Random(seed ^ hash((mechanism, mood, decade, content_type)))
    recent = get_recent()

    def base_score(item):
        s = 0.0
        if item.get("mechanism") == mechanism: s += 10
        if mood and mood in (item.get("moods") or []): s += 5
        if item.get("id") in recent: s -= 6
        return s

    def jittered(item):
        return -(base_score(item) + rng.gauss(0, 1.5))

    pool = sorted(candidates, key=jittered)[:max(limit * 3, 12)]
    weights = [max(0.01, 2 ** (base_score(i) / 2.5)) for i in pool]

    chosen, used = [], set()
    attempts = 0
    while len(chosen) < limit and pool and attempts < limit * 8:
        attempts += 1
        pick = rng.choices(pool, weights=weights, k=1)[0]
        if pick["id"] in used:
            continue
        chosen.append(pick)
        used.add(pick["id"])

    out = [{
        "id":        i["id"],
        "type":      i.get("type"),
        "title":     i.get("title"),
        "year":      i.get("year"),
        "mechanism": i.get("mechanism"),
        "moods":     (i.get("moods") or [])[:4],
        "blurb":     (i.get("blurb") or i.get("why") or "")[:140],
    } for i in chosen]

    _trace("search_catalogue",
           {"mechanism": mechanism, "mood": mood, "decade": decade,
            "type": content_type, "limit": limit},
           f"{len(out)} items: {[i['title'][:25] for i in out]}")
    return out


# ═══════════════════════════════════════════════════════════════════════
# Tool 2 — get_research_paper
# ═══════════════════════════════════════════════════════════════════════
def get_research_paper(paper_id: str) -> dict:
    """Look up the scientific paper that grounds a given psychological mechanism.

    Args:
        paper_id: The mechanism id (e.g. "nostalgia", "social_surrogate").

    Returns:
        A dict with mechanism, description, and paper citation.
        Returns {"error": ...} if not found.
    """
    info = _MECHANISM_INDEX.get(paper_id)
    if not info:
        _trace("get_research_paper", {"paper_id": paper_id}, "not found")
        return {"error": f"No paper found for mechanism '{paper_id}'"}
    out = {
        "mechanism":   info.get("mechanism", paper_id),
        "description": info.get("description", "")[:300],
        "paper":       info.get("paper", "")[:200],
    }
    _trace("get_research_paper", {"paper_id": paper_id},
           f"{out['mechanism']}: {out['description'][:60]}...")
    return out


# ═══════════════════════════════════════════════════════════════════════
# Tool 3 — score_item_for_user
# ═══════════════════════════════════════════════════════════════════════
def score_item_for_user(item_id: str, primary_mechanism: str,
                        active_moods: list, nostalgia_decade: int = 0,
                        tone: str = "warm") -> dict:
    """Score how well a specific item fits a user's psychological profile.

    Args:
        item_id: The id of the item to score (from search_catalogue results).
        primary_mechanism: The user's primary mechanism (from psych profile).
        active_moods: List of mood keys the user is feeling.
        nostalgia_decade: User's reminiscence decade (e.g. 1990). 0 if not relevant.
        tone: "warm" (default) or "tender" (high-distress mode, no surprise).

    Returns:
        A dict with score (0-100), match_reasons, warnings.
    """
    item = _ITEM_INDEX.get(item_id)
    if not item:
        out = {"score": 0, "match_reasons": [],
               "warnings": [f"Item '{item_id}' not in catalogue"]}
        _trace("score_item_for_user", {"item_id": item_id}, "not found")
        return out

    score = 0
    reasons = []
    warnings = []

    if item.get("mechanism") == primary_mechanism:
        score += 40
        reasons.append(f"primary mechanism match ({primary_mechanism})")
    elif primary_mechanism in (item.get("mechanisms") or []):
        score += 25
        reasons.append(f"secondary mechanism match ({primary_mechanism})")

    item_moods = set(item.get("moods") or [])
    user_moods = set(active_moods or [])
    overlap = item_moods & user_moods
    if overlap:
        score += min(len(overlap) * 10, 25)
        reasons.append(f"mood overlap: {', '.join(list(overlap)[:3])}")

    if nostalgia_decade:
        item_decade = item.get("decade") or (item.get("year", 0) // 10) * 10
        if item_decade == nostalgia_decade:
            score += 20
            reasons.append(f"matches reminiscence decade ({nostalgia_decade}s)")

    if tone == "tender":
        if item.get("intensity") == "high":
            score -= 15
            warnings.append("high-intensity item in tender mode")
        if "humor" in (item.get("tags") or []) or "irony" in (item.get("tags") or []):
            score -= 10
            warnings.append("humor/irony unsuitable for tender mode")

    score = max(0, min(100, score))
    out = {
        "item_id":       item_id,
        "title":         item.get("title", ""),
        "score":         score,
        "match_reasons": reasons,
        "warnings":      warnings,
    }
    _trace("score_item_for_user", {"item_id": item_id, "tone": tone},
           f"score={score} reasons={len(reasons)}")
    return out


# ═══════════════════════════════════════════════════════════════════════
# Tool 4 — finalize_recommendations
# ═══════════════════════════════════════════════════════════════════════
def finalize_recommendations(item_ids: list, thesis: str,
                             thematic_data: list) -> dict:
    """Submit your final recommendations. Call this exactly once at the end.

    Args:
        item_ids: List of exactly 4 item ids in display order.
        thesis: 2-3 sentences addressed to the user as "you", matching the
            requested tone (warm or tender).
        thematic_data: List of 5 dicts {"name": str, "value": int 0-100}.
            Suggested names: Nostalgia, Comfort, Awe, Connection, Surprise.

    Returns:
        A dict with status="finalized" and the validated payload.
    """
    items = []
    missing = []
    for iid in item_ids[:4]:
        meta = _ITEM_INDEX.get(iid)
        if meta:
            items.append({**meta})
        else:
            missing.append(iid)
    out = {
        "status":        "finalized",
        "items":         items,
        "thesis":        thesis,
        "thematic_data": thematic_data,
        "missing_ids":   missing,
    }
    _trace("finalize_recommendations",
           {"item_ids": item_ids, "thesis_len": len(thesis)},
           f"{len(items)} items, {len(missing)} missing")
    return out


# Tool list & registry for the agent loop
LIBRARIAN_TOOLS = [
    search_catalogue,
    get_research_paper,
    score_item_for_user,
    finalize_recommendations,
]
TOOL_REGISTRY = {fn.__name__: fn for fn in LIBRARIAN_TOOLS}


# ═══════════════════════════════════════════════════════════════════════
# Agent loop
# ═══════════════════════════════════════════════════════════════════════
MAX_TOOL_TURNS         = 5
MAX_NEW_TOKENS_PER_TURN = 350
MAX_TOOL_RESULT_CHARS   = 1200


LIBRARIAN_SYSTEM_PROMPT = """You are the librarian of Ami, a science-grounded well-being app. You have a strict 5-turn budget to recommend 4 items.

WORKFLOW (do NOT deviate):
- Turn 1: ONE call to search_catalogue with primary_mechanism. Optionally a second call with secondary_mechanism in the SAME turn.
- Turn 2: From the candidates, pick 3-4 promising items. Call score_item_for_user on each.
- Turn 3: Call finalize_recommendations with your final 4 picks. This ends your work.

RULES:
- Do NOT call search_catalogue more than 3 times total.
- Do NOT call get_research_paper unless you genuinely need it (it's optional).
- You MUST call finalize_recommendations by turn 3 or 4 at the latest.
- Only use item ids that came from search_catalogue results — never invent ids.
- Aim for variety: ideally 2 films/series + 1 novel + 1 anecdote.

TONE rules:
- tone="warm" → poetic, can be playful or surprising.
- tone="tender" → soft, quiet, kind only. No humor, no irony, no surprise.

You will be penalized if you exhaust the turn budget without calling finalize_recommendations."""


def _format_user_message(profile: dict) -> str:
    nostalgia = profile.get("nostalgia_decade") or "n/a"
    return (
        f"Recommend 4 items for this user.\n\n"
        f"emotion_core:        {profile.get('emotion_core', '?')}\n"
        f"primary_mechanism:   {profile.get('primary_mechanism', '?')}\n"
        f"secondary_mechanism: {profile.get('secondary_mechanism', '')}\n"
        f"active_moods:        {', '.join(profile.get('active_moods', []))}\n"
        f"intensity:           {profile.get('intensity', 'medium')}\n"
        f"nostalgia_decade:    {nostalgia}\n"
        f"tone:                {profile.get('tone', 'warm')}\n"
        f"surprise_ok:         {profile.get('surprise_ok', False)}\n"
        f"decoder_note:        {profile.get('decoder_note', '')}\n"
    )


def _aggressive_free_vram():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()


def _generate_v3(messages: list, tools: list = None,
                 max_new: int = MAX_NEW_TOKENS_PER_TURN) -> str:
    """One Gemma turn: chat template + generate."""
    _aggressive_free_vram()
    encoded = tokenizer.apply_chat_template(
        messages, tools=tools,
        tokenize=True, add_generation_prompt=True,
        return_tensors="pt", return_dict=True,
    )
    input_ids      = encoded["input_ids"].to(model.device)
    attention_mask = encoded["attention_mask"].to(model.device)

    if input_ids.shape[-1] > 6000:
        print(f"   ⚠️  Context size: {input_ids.shape[-1]} tokens (warning)")

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new,
            temperature=0.6,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output_ids[0][input_ids.shape[-1]:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=False)
    del input_ids, attention_mask, output_ids, new_tokens, encoded
    _aggressive_free_vram()
    return text


def _parse_assistant_response(raw: str) -> dict:
    """
    Try Gemma 4's native parse_response. Fall back to regex on the native format.
    Returns: {"text": str, "tool_calls": [{"name", "arguments"}, ...]}
    """
    # Native parse
    try:
        parsed = tokenizer.parse_response(raw)
        if isinstance(parsed, dict):
            text = parsed.get("content", "") or parsed.get("text", "")
            calls = parsed.get("tool_calls") or []
            normalized = []
            for c in calls:
                if isinstance(c, dict):
                    name = c.get("name") or c.get("function", {}).get("name")
                    args = c.get("arguments") or c.get("function", {}).get("arguments") or {}
                    if isinstance(args, str):
                        try: args = json.loads(args)
                        except: args = {}
                    if name:
                        normalized.append({"name": name, "arguments": args})
            return {"text": text, "tool_calls": normalized}
    except Exception:
        pass

    # Regex fallback on Gemma 4 native format: <|tool_call>call:NAME{args}<tool_call|>
    calls = []
    for m in re.finditer(
        r'<\|tool_call\|?>call:([a-z_]+)\{(.*?)\}(?=<\||\Z)',
        raw, flags=re.DOTALL
    ):
        name = m.group(1)
        args_str = m.group(2)
        try:
            args_clean = "{" + args_str + "}"
            args_clean = re.sub(r'<\|"\|>', '"', args_clean)
            args = json.loads(args_clean)
        except Exception:
            args = {}
        calls.append({"name": name, "arguments": args})

    text_clean = re.sub(r'<\|tool_call.*?(?=<\||\Z)', '', raw, flags=re.DOTALL)
    text_clean = re.sub(r'<\|[^|]+\|>', '', text_clean).strip()
    return {"text": text_clean, "tool_calls": calls}


def run_librarian_agentic(psych_profile: dict, verbose: bool = True) -> dict:
    """Agentic librarian with native function calling."""
    TOOL_CALL_TRACE.clear()
    _aggressive_free_vram()

    messages = [
        {"role": "system", "content": LIBRARIAN_SYSTEM_PROMPT},
        {"role": "user",   "content": _format_user_message(psych_profile)},
    ]

    final_payload = None
    seen_item_ids = set()
    search_count  = 0

    for turn in range(MAX_TOOL_TURNS):
        if verbose:
            print(f"\n   ─── Turn {turn+1}/{MAX_TOOL_TURNS} ───")

        # Last-turn nudge
        if turn == MAX_TOOL_TURNS - 1 and seen_item_ids:
            messages.append({
                "role": "user",
                "content": (
                    f"You have {len(seen_item_ids)} candidate items already. "
                    f"This is your LAST turn. Call finalize_recommendations NOW "
                    f"with 4 of these ids: {list(seen_item_ids)[:8]}."
                )
            })

        try:
            raw = _generate_v3(messages, tools=LIBRARIAN_TOOLS)
        except torch.cuda.OutOfMemoryError as e:
            if verbose: print(f"   ❌ OOM: {e}")
            _aggressive_free_vram()
            break
        except Exception as e:
            if verbose: print(f"   ❌ generate failed: {e}")
            break

        parsed = _parse_assistant_response(raw)
        text       = parsed["text"]
        tool_calls = parsed["tool_calls"]

        if verbose:
            print(f"   raw[:140]: {raw[:140]!r}")
            print(f"   tool_calls: {[c['name'] for c in tool_calls]}")

        if not tool_calls:
            if verbose: print("   ⚠️  No tool call — agent stopped")
            break

        # Re-inject assistant turn (OpenAI-style for chat template)
        messages.append({
            "role":       "assistant",
            "content":    text or "",
            "tool_calls": [
                {"type": "function",
                 "function": {"name": c["name"], "arguments": c["arguments"] or {}}}
                for c in tool_calls
            ],
        })

        # Execute each tool
        for call in tool_calls:
            name = call["name"]
            args = call["arguments"] or {}

            if name == "search_catalogue":
                search_count += 1
                if search_count > 3:
                    if verbose: print("   🚫 search_catalogue limit reached")
                    messages.append({
                        "role": "tool", "name": name,
                        "content": json.dumps({
                            "error": "Search budget exhausted. Use score_item_for_user "
                                     "and finalize_recommendations now.",
                            "available_item_ids": list(seen_item_ids)[:10]
                        }),
                    })
                    continue

            if name not in TOOL_REGISTRY:
                messages.append({
                    "role": "tool", "name": name,
                    "content": json.dumps({"error": f"Unknown tool '{name}'"}),
                })
                continue

            try:
                result = TOOL_REGISTRY[name](**args)
                if name == "search_catalogue" and isinstance(result, list):
                    for it in result:
                        if isinstance(it, dict) and "id" in it:
                            seen_item_ids.add(it["id"])

                if verbose:
                    rs = json.dumps(result, default=str)
                    print(f"   ✅ {name}({list(args.keys())}) → {rs[:90]}")

                result_str = json.dumps(result, default=str)
                if len(result_str) > MAX_TOOL_RESULT_CHARS:
                    result_str = result_str[:MAX_TOOL_RESULT_CHARS] + '...(truncated)"}'

                messages.append({
                    "role": "tool", "name": name, "content": result_str,
                })

                if (name == "finalize_recommendations"
                        and isinstance(result, dict)
                        and result.get("status") == "finalized"):
                    final_payload = result
                    if verbose: print("   🎯 Finalized — stopping loop")
                    break

            except Exception as e:
                if verbose: print(f"   ❌ {name} crashed: {e}")
                messages.append({
                    "role": "tool", "name": name,
                    "content": json.dumps({"error": str(e)}),
                })

        if final_payload:
            break

    # Build response
    if final_payload and final_payload.get("items"):
        # Diversity patch: cooldown for next request
        remember_recommended([i.get("id") for i in final_payload.get("items", [])])
        return {
            "items":         final_payload.get("items", []),
            "thesis":        final_payload.get("thesis", ""),
            "thematic_data": final_payload.get("thematic_data", []),
            "tool_trace":    list(TOOL_CALL_TRACE),
            "agentic":       True,
            "turns_used":    turn + 1,
        }

    # Fallback synthesis from seen items
    if seen_item_ids:
        if verbose: print(f"\n   🛟 Fallback synthesis from {len(seen_item_ids)} seen items")
        picked = [_ITEM_INDEX[iid] for iid in list(seen_item_ids)[:4] if iid in _ITEM_INDEX]
        # Diversity patch: cooldown for next request
        remember_recommended([i.get("id") for i in picked])
        return {
            "items":         picked,
            "thesis":        ("Ami picked these for you, even if our librarian "
                              "took the scenic route to get there."),
            "thematic_data": [
                {"name": "Nostalgia", "value": 60}, {"name": "Comfort",    "value": 60},
                {"name": "Awe",    "value": 50}, {"name": "Connection", "value": 50},
                {"name": "Surprise",  "value": 30},
            ],
            "tool_trace":  list(TOOL_CALL_TRACE),
            "agentic":     True,
            "warning":     "agent_fallback_synthesis",
            "turns_used":  turn + 1,
        }

    return {
        "items": [], "thesis": "Ami needs a moment — please try again.",
        "thematic_data": [], "tool_trace": list(TOOL_CALL_TRACE),
        "agentic": True, "warning": "no_candidates_found",
        "turns_used": turn + 1,
    }


print(f"✅ Agentic librarian ready ({len(LIBRARIAN_TOOLS)} tools)")
print("   To test: result = run_librarian_agentic({...profile...})")

✅ Agentic librarian ready (4 tools)
   To test: result = run_librarian_agentic({...profile...})


## 5. Frontend

A single-file React app served by Flask. Features:
- Free text + mood buttons (combined input)
- Birth year for the reminiscence-bump decade
- Crisis panel with helplines + "continue gently" option
- **Toggle "See Ami's reasoning"** → shows the agentic backstage panel with each tool call

In [14]:
# ════════════════════════════════════════════════
# 10b · Embed mood mask illustrations
# 5 hand-drawn cubist mood masks (originally HEIC with alpha channel),
# converted to compact WebP with TRANSPARENCY preserved (~28KB each).
# Written to /static/ at runtime so Flask can serve them at /static/mood-N.webp
# ════════════════════════════════════════════════

import os, base64
os.makedirs('/kaggle/working/static', exist_ok=True)

MOODS_B64 = {
    1: (
        "..."
    ),
    2: (
        "..."
    ),
    3: (
        "..."
        "G9TtAAAAAAA="
    ),
    4: (
        "..."
    ),
    5: (
        "..."
    ),
}

for idx, b64 in MOODS_B64.items():
    out = f'/kaggle/working/static/mood-{idx}.webp'
    with open(out, 'wb') as f:
        f.write(base64.b64decode(b64))

print(f'✅ Wrote {len(MOODS_B64)} mood masks (transparent WebP) to /kaggle/working/static/')

✅ Wrote 5 mood masks (transparent WebP) to /kaggle/working/static/


In [15]:
# ════════════════════════════════════════════════
# 11 · Write the React frontend (single-file ami.jsx)
# Layout :
#   - Storybook hero, sun/moon + clouds/stars, dark mode, crisis panel,
#     thesis modal, recommendation cards with details + LET'S GO buttons.
#   - Mood mask gallery inside the thesis card: 5 transparent WebP masks
#     stacked vertically at hero size, matched-mood first, happy-mood
#     always last. Each mask appears on scroll, reacts on hover, and can
#     be dragged left/right with the cursor (snaps back on release).
# Wires to the Kaggle backend via POST /api/recommend.
# ════════════════════════════════════════════════

import os
os.makedirs('/kaggle/working/static', exist_ok=True)

AMI_JSX = r"""import React, { useState, useEffect, useMemo } from "react";
import { motion, AnimatePresence } from "framer-motion";
import {
  Sparkles, Quote, Star, Loader2, Video, Tv, Book,
  Heart, Phone, X, ArrowUpRight, Sun, Moon,
  Mic, MicOff, ImagePlus,
  ChevronLeft, ChevronRight,
} from "lucide-react";
import {
  AreaChart, Area, ResponsiveContainer,
} from "recharts";

// ─────────────────────────────────────────────────────────────────
// Theme — mirrors the prototype's index.css (Tailwind v4 @theme)
// ─────────────────────────────────────────────────────────────────
const THEME_CSS = `
@import url('https://fonts.googleapis.com/css2?family=Fredoka:wght@300;400;500;600;700&family=Quicksand:wght@300;400;500;600;700&family=Gochi+Hand&family=Lacquer&display=swap');

:root {
  --color-fun-pink: #E8B4AA;
  --color-fun-mint: #A2FF00;
  --color-fun-lilac: #E6E6FA;
  --color-fun-yellow: #FFBF00;
  --color-fun-blue: #38BDF8;
  --color-fun-orange: #FFB347;
  --color-fun-ink: #000000;
  --color-fun-white: #FFFFFF;

  --font-display: "Fredoka", sans-serif;
  --font-body: "Quicksand", sans-serif;
  --font-sketch: "Gochi Hand", cursive;
  --font-weird: "Lacquer", cursive;

  --shadow-stroke: white;
  --shadow-color: rgba(75, 54, 33, 0.1);
  --noise-radial-pink: rgba(232, 180, 170, 1);
  --noise-radial-blue: rgba(56, 189, 248, 1);

  --text-secondary: rgba(0, 0, 0, 0.7);
  --text-muted: rgba(0, 0, 0, 0.4);
  --text-faint: rgba(0, 0, 0, 0.2);
  --border-soft: rgba(0, 0, 0, 0.05);
  --border-subtle: rgba(0, 0, 0, 0.2);
}

/* DARK MODE — same trick as the prototype: redefine the vars,
   everything that uses them auto-inverts. */
.ami-root.ami-dark {
  --color-fun-yellow: rgba(255, 191, 0, 0.3);
  --color-fun-orange: rgba(251, 146, 60, 0.4);
  --color-fun-ink: #cbd5e1;
  --color-fun-white: #0f172a;

  --shadow-stroke: #0f172a;
  --shadow-color: rgba(0, 0, 0, 0.5);
  --noise-radial-pink: rgba(232, 180, 170, 0.5);
  --noise-radial-blue: rgba(56, 189, 248, 0.5);

  --text-secondary: rgba(203, 213, 225, 0.8);
  --text-muted: rgba(203, 213, 225, 0.5);
  --text-faint: rgba(203, 213, 225, 0.3);
  --border-soft: rgba(203, 213, 225, 0.1);
  --border-subtle: rgba(203, 213, 225, 0.3);
}

.ami-root {
  font-family: var(--font-body);
  color: var(--color-fun-ink);
  background: var(--color-fun-blue);
  min-height: 100vh;
  width: 100%;
  position: relative;
  overflow-x: hidden;
  transition: background-color 0.8s ease, color 0.5s ease;
}
.ami-root.ami-dark {
  background: var(--color-fun-white); /* now #0f172a */
}
.ami-root *::selection { background: var(--color-fun-pink); color: var(--color-fun-ink); }

.font-display { font-family: var(--font-display); }
.font-body    { font-family: var(--font-body); }
.font-sketch  { font-family: var(--font-sketch); }
.font-weird   { font-family: var(--font-weird); }

.sticker-shadow {
  box-shadow: 0 0 0 4px var(--shadow-stroke), 8px 8px 0 0 var(--shadow-color);
  transition: all 0.3s cubic-bezier(0.175, 0.885, 0.32, 1.275);
}
.sticker-shadow-hover:hover {
  transform: scale(1.02) rotate(-1deg);
  box-shadow: 0 0 0 6px var(--shadow-stroke), 12px 12px 0 0 var(--shadow-color);
}
.bubble-shadow {
  box-shadow: 0 0 0 4px var(--shadow-stroke), 8px 8px 0 0 var(--shadow-color);
}

.storybook-title {
  font-family: var(--font-display);
  font-weight: 900;
  letter-spacing: -0.02em;
  -webkit-text-stroke: 1.5px var(--color-fun-ink);
  color: white;
}
.ami-root.ami-dark .storybook-title {
  -webkit-text-stroke: 1.5px white;
  color: #0f172a;
}

/* Hero outline-only words in dark mode */
.outline-white-on-dark > * { color: white; }
.ami-root.ami-dark .outline-white-on-dark > * {
  -webkit-text-stroke: 1.5px white !important;
  color: transparent !important;
}

/* Mood-suggestion buttons: hover via CSS class to avoid the framer-motion
   "interpolate between two CSS variables → black flash" bug. */
.ami-mood-btn:hover {
  background: var(--color-fun-yellow) !important;
}

/* Stack the details-modal grid on narrow screens */
@media (max-width: 720px) {
  .ami-modal-grid {
    grid-template-columns: 1fr !important;
  }
  .ami-modal-grid > div:first-child {
    position: static !important;
    max-width: 16rem;
    margin: 0 auto;
  }
}

/* ── Dark mode : texte blanc dans les inputs ── */
.ami-root.ami-dark textarea,
.ami-root.ami-dark input[type="number"] {
  color: white;
}

/* ── Dark mode : texte blanc dans "First reading" ── */
.ami-root.ami-dark .vision-insight-understanding {
  color: white !important;
}

/* ── Dark mode : tags noirs dans "First reading" (fond clair → texte noir lisible) ── */
.ami-root.ami-dark .vision-insight-card .vision-tag {
  color: black !important;
}

/* ── Dark mode : white text only in anecdote why block (white bg) ── */
.ami-root.ami-dark .ami-why-anecdote p,
.ami-root.ami-dark .ami-why-anecdote div {
  color: white !important;
}
.ami-root.ami-dark .ami-why-anecdote .ami-why-label {
  color: rgba(255,255,255,0.55) !important;
}

/* ── Mechanism tags: always black text (colored bg is always light) ── */
.ami-mech-tag {
  color: #000 !important;
}

.noise-overlay::before {
  content: ""; position: fixed; inset: 0; pointer-events: none;
  opacity: 0.08; z-index: 50;
  background-image: url("data:image/svg+xml,%3Csvg viewBox='0 0 200 200' xmlns='http://www.w3.org/2000/svg'%3E%3Cfilter id='noise'%3E%3CfeTurbulence type='fractalNoise' baseFrequency='0.04' numOctaves='3' stitchTiles='stitch'/%3E%3CfeColorMatrix type='saturate' values='0'/%3E%3C/filter%3E%3Crect width='100%25' height='100%25' filter='url(%23noise)'/%3E%3C/svg%3E");
  mix-blend-mode: overlay;
}
.noise-overlay::after {
  content: ""; position: fixed; inset: 0; pointer-events: none;
  opacity: 0.15; z-index: 49;
  background-image:
    radial-gradient(circle at 15% 15%, var(--noise-radial-pink) 0%, transparent 35%),
    radial-gradient(circle at 85% 85%, var(--noise-radial-blue) 0%, transparent 35%),
    radial-gradient(circle at 50% 10%, rgba(152, 251, 152, 0.1) 0%, transparent 30%);
  filter: blur(80px);
}

/* ── react-pageflip (StPageFlip) — force opaque pages ───────────── */
.stf__parent,
.stf__wrapper {
  background: transparent;
}
.stf__block {
  background: var(--color-fun-white);
}
/* Each physical page surface — must be fully opaque so stacked pages
   underneath never show through during or after a flip. */
.ami-magazine-book .stf__item,
.ami-flip-page,
.ami-flip-page.page {
  background: var(--color-fun-white) !important;
  background-color: var(--color-fun-white) !important;
}
/* The page's inner content wrapper also gets a solid base, in case a
   child layout leaves gaps (e.g. content shorter than the page). */
.ami-flip-page > * {
  background-color: var(--color-fun-white);
  min-height: 100%;
}
/* Soft drop shadow under the whole book so it reads as a real object. */
.ami-magazine-book {
  background: transparent;
}

/* Re-enable text selection inside the book — react-pageflip sets
   user-select: none on its wrappers to keep drag clean. We override
   it on the page content. The corner-drag still works because the
   library captures pointer events at the book level on the corners. */
.ami-flip-page,
.ami-flip-page * {
  user-select: text !important;
  -webkit-user-select: text !important;
}
/* And we want the cursor to clearly tell the user the text is selectable. */
.ami-flip-page-content,
.ami-flip-page-content * {
  cursor: text;
}
.ami-flip-page-content a,
.ami-flip-page-content button,
.ami-flip-page-content [role="button"] {
  cursor: pointer;
}
`;

function cn(...args) { return args.filter(Boolean).join(" "); }

// ─────────────────────────────────────────────────────────────────
// FloatingWords
// ─────────────────────────────────────────────────────────────────
const FloatingWords = ({ text, style, className, intensity = 1, delayOffset = 0 }) => {
  if (!text) return null;
  return (
    <span style={style} className={className}>
      {String(text).split(" ").map((word, i) => (
        <motion.span
          key={i}
          style={{ display: "inline-block", whiteSpace: "nowrap", marginRight: "0.3em", cursor: "default" }}
          animate={{
            y: [0, -4 * intensity, 2 * intensity, 0],
            rotate: [0, 0.5 * intensity, -0.5 * intensity, 0],
          }}
          whileHover={{
            y: -15 * intensity,
            rotate: (Math.random() - 0.5) * 15,
            zIndex: 10,
            transition: { duration: 0.2, type: "spring", stiffness: 400, damping: 10 },
          }}
          whileTap={{ rotate: (Math.random() - 0.5) * 30 }}
          transition={{
            duration: 10 + Math.random() * 5,
            repeat: Infinity,
            ease: "easeInOut",
            delay: i * 0.2 + delayOffset,
          }}
        >
          {word}
        </motion.span>
      ))}
    </span>
  );
};

// ─────────────────────────────────────────────────────────────────
// MoodShape — animated organic blob
// ─────────────────────────────────────────────────────────────────
const MoodShape = ({ mood }) => {
  const config = useMemo(() => {
    const m = (mood || "").toLowerCase();
    let color = "#3B82F6", secondaryColor = "#60A5FA", complexity = 3, duration = 8, scale = 1;
    if (m.includes("sad") || m.includes("lonely") || m.includes("melancholic") || m.includes("grief")) {
      color = "#2563EB"; secondaryColor = "#93C5FD"; complexity = 4; duration = 12;
    } else if (m.includes("angry") || m.includes("cynical")) {
      color = "#EF4444"; secondaryColor = "#FCA5A1"; complexity = 6; duration = 4; scale = 1.1;
    } else if (m.includes("anxious") || m.includes("lost")) {
      color = "#F472B6"; secondaryColor = "#FBCFE8"; complexity = 8; duration = 3; scale = 1.15;
    } else if (m.includes("nostalgic") || m.includes("empty")) {
      color = "#A78BFA"; secondaryColor = "#DDD6FE"; complexity = 5; duration = 10;
    } else if (m.includes("peace") || m.includes("calm")) {
      color = "#34D399"; secondaryColor = "#A7F3D0"; complexity = 2; duration = 15; scale = 0.9;
    }
    return { color, secondaryColor, complexity, duration, scale };
  }, [mood]);

  const generatePath = (seed) => {
    const points = [];
    const radius = 35, center = 50, numPoints = 8;
    for (let i = 0; i < numPoints; i++) {
      const angle = (i / numPoints) * Math.PI * 2;
      const offset = Math.sin(angle * (config.complexity % 5 + 2) + seed) * (config.complexity * 1.2);
      const dist = radius + offset;
      points.push({ x: center + Math.cos(angle) * dist, y: center + Math.sin(angle) * dist });
    }
    let path = `M ${points[0].x},${points[0].y}`;
    for (let i = 0; i < points.length; i++) {
      const next = points[(i + 1) % points.length];
      const nextNext = points[(i + 2) % points.length];
      path += ` Q ${next.x},${next.y} ${(next.x + nextNext.x) / 2},${(next.y + nextNext.y) / 2}`;
    }
    return path + " Z";
  };

  const path1 = useMemo(() => generatePath(0), [config]);
  const path2 = useMemo(() => generatePath(Math.PI), [config]);
  const path3 = useMemo(() => generatePath(Math.PI / 2), [config]);

  return (
    <div style={{ position: "relative", width: "8rem", height: "8rem", display: "flex", alignItems: "center", justifyContent: "center" }}>
      <motion.div
        animate={{ scale: [1, 1.2, 1], opacity: [0.2, 0.4, 0.2] }}
        transition={{ duration: config.duration, repeat: Infinity, ease: "easeInOut" }}
        style={{ position: "absolute", inset: 0, filter: "blur(32px)", borderRadius: "9999px", backgroundColor: config.color }}
      />
      <motion.svg
        viewBox="0 0 100 100"
        style={{ width: "100%", height: "100%", position: "relative", zIndex: 10 }}
        animate={{ scale: config.scale, rotate: [0, 360] }}
        transition={{
          rotate: { duration: config.duration * 4, repeat: Infinity, ease: "linear" },
          scale: { duration: 0.5 },
        }}
      >
        <defs>
          <linearGradient id="moodGradient" x1="0%" y1="0%" x2="100%" y2="100%">
            <stop offset="0%" stopColor={config.color} />
            <stop offset="100%" stopColor={config.secondaryColor} />
          </linearGradient>
          <filter id="gooey">
            <feGaussianBlur in="SourceGraphic" stdDeviation="2" result="blur" />
            <feColorMatrix in="blur" mode="matrix" values="1 0 0 0 0  0 1 0 0 0  0 0 1 0 0  0 0 0 18 -7" result="goo" />
          </filter>
        </defs>
        <g filter="url(#gooey)">
          <motion.path
            d={path1} fill="url(#moodGradient)"
            animate={{ d: [path1, path2, path3, path1], scale: [1, 1.02, 1] }}
            transition={{
              d: { duration: config.duration, repeat: Infinity, ease: "easeInOut" },
              scale: { duration: 2, repeat: Infinity, ease: "easeInOut" },
            }}
          />
          <motion.path
            d={path2} fill={config.secondaryColor} opacity={0.3}
            animate={{ d: [path2, path3, path1, path2] }}
            transition={{ duration: config.duration * 1.5, repeat: Infinity, ease: "easeInOut" }}
          />
        </g>
        <motion.ellipse
          cx="35" cy="35" rx="10" ry="5" fill="white" opacity="0.2"
          animate={{ rotate: [0, 360] }}
          transition={{ duration: config.duration * 4, repeat: Infinity, ease: "linear" }}
        />
      </motion.svg>
      <motion.div
        animate={{ scale: [1, 1.4, 1] }}
        transition={{ duration: 2, repeat: Infinity }}
        style={{ position: "absolute", inset: 0, display: "flex", alignItems: "center", justifyContent: "center", zIndex: 20, pointerEvents: "none" }}
      >
        <div style={{ width: "0.375rem", height: "0.375rem", background: "white", borderRadius: "9999px", boxShadow: "0 0 8px rgba(255,255,255,0.8)" }} />
      </motion.div>
    </div>
  );
};

// ─────────────────────────────────────────────────────────────────
// Background — animated sun + clouds (or moon + stars in dark mode)
// ─────────────────────────────────────────────────────────────────
const Background = ({ active, dark }) => {
  const clouds = [
    { top: "8%",  side: "left",  startX: 20, endX: 180,  scale: 0.8,  duration: 35, hasEyes: true },
    { top: "30%", side: "right", startX: 20, endX: -150, scale: 0.2,  duration: 45 },
    { top: "70%", side: "left",  startX: 30, endX: 150,  scale: 0.6,  duration: 40 },
    { top: "20%", side: "right", startX: 40, endX: -120, scale: 0.15, duration: 60 },
    { top: "85%", side: "left",  startX: 10, endX: 160,  scale: 0.5,  duration: 50 },
    { top: "50%", side: "right", startX: 50, endX: -220, scale: 0.9,  duration: 30 },
    { top: "42%", side: "left",  startX: 0,  endX: 100,  scale: 0.3,  duration: 55 },
  ];

  // 20 stars — generated once so they don't jump around on re-render
  const stars = useMemo(() => (
    [...Array(20)].map((_, i) => ({
      key: i,
      top: `${Math.random() * 100}%`,
      left: `${Math.random() * 100}%`,
      size: Math.random() * 4 + 2,
      duration: Math.random() * 3 + 2,
      delay: Math.random() * 5,
    }))
  ), []);

  return (
    <div style={{ position: "fixed", inset: 0, overflow: "hidden", pointerEvents: "none", zIndex: 10 }}>

      {/* Sun OR Moon */}
      {active && (
        <motion.div
          initial={{ opacity: 0, scale: 0.5 }}
          animate={
            dark
              ? { opacity: 1, scale: [1, 1.02, 1] }
              : { opacity: 1, scale: [1, 1.05, 1], y: [0, -20, 0] }
          }
          transition={
            dark
              ? { opacity: { duration: 1 }, scale: { duration: 6, repeat: Infinity, ease: "easeInOut" } }
              : {
                  opacity: { duration: 1 },
                  scale: { duration: 4, repeat: Infinity, ease: "easeInOut" },
                  y: { duration: 6, repeat: Infinity, ease: "easeInOut" },
                }
          }
          style={{
            position: "absolute",
            top: dark ? "12%" : "15%",
            right: dark ? "18%" : "15%",
            width: dark ? "16rem" : "18rem",
            height: dark ? "16rem" : "18rem",
            zIndex: 20, padding: "1rem",
          }}
        >
          <div style={{ position: "relative", width: "100%", height: "100%", display: "flex", alignItems: "center", justifyContent: "center" }}>
            {dark ? (
              <>
                {/* Moon halos — lilac + blue, animated pulse */}
                <motion.div
                  animate={{ opacity: [0.4, 0.5, 0.4] }}
                  transition={{ duration: 2, repeat: Infinity, ease: "easeInOut" }}
                  style={{ position: "absolute", inset: 0, background: "var(--color-fun-lilac)", opacity: 0.4, filter: "blur(80px)", borderRadius: "9999px", transform: "scale(1.25)" }}
                />
                <motion.div
                  animate={{ opacity: [0.3, 0.4, 0.3] }}
                  transition={{ duration: 2, repeat: Infinity, ease: "easeInOut", delay: 1 }}
                  style={{ position: "absolute", inset: 0, background: "var(--color-fun-blue)", opacity: 0.3, filter: "blur(100px)", borderRadius: "9999px", transform: "scale(1.5)" }}
                />

                {/* Moon body */}
                <div
                  style={{
                    width: "100%", height: "100%", borderRadius: "9999px",
                    background: "#e2e8f0", /* slate-200 */
                    position: "relative", zIndex: 10, overflow: "hidden",
                    filter: "drop-shadow(0 0 30px rgba(230,230,250,0.6))",
                  }}
                >
                  {/* Craters (blurred, like the prototype) */}
                  <div style={{ position: "absolute", top: "20%", left: "30%", width: "2rem", height: "2rem", background: "#cbd5e1", borderRadius: "9999px", opacity: 0.6, filter: "blur(2px)" }} />
                  <div style={{ position: "absolute", top: "60%", left: "20%", width: "3rem", height: "3rem", background: "#cbd5e1", borderRadius: "9999px", opacity: 0.5, filter: "blur(3px)" }} />
                  <div style={{ position: "absolute", top: "40%", left: "70%", width: "2.5rem", height: "2.5rem", background: "#cbd5e1", borderRadius: "9999px", opacity: 0.4, filter: "blur(2px)" }} />
                  <div style={{ position: "absolute", top: "75%", left: "65%", width: "1.5rem", height: "1.5rem", background: "#cbd5e1", borderRadius: "9999px", opacity: 0.55, filter: "blur(2px)" }} />

                  {/* Subtle crescent highlight */}
                  <div style={{ position: "absolute", inset: "-0.5rem", background: "linear-gradient(to top right, transparent, rgba(255,255,255,0.1), rgba(255,255,255,0.3))", borderRadius: "9999px" }} />
                </div>
              </>
            ) : (
              <>
                <div style={{ position: "absolute", inset: 0, background: "var(--color-fun-yellow)", opacity: 0.9, filter: "blur(90px)", borderRadius: "9999px", transform: "scale(1.25)" }} />
                <div style={{ position: "absolute", inset: 0, background: "var(--color-fun-orange)", opacity: 0.5, filter: "blur(130px)", borderRadius: "9999px", transform: "scale(1.5)" }} />
                <motion.div
                  animate={{ rotate: 360, scale: [1, 1.1, 1] }}
                  transition={{
                    rotate: { duration: 60, repeat: Infinity, ease: "linear" },
                    scale: { duration: 5, repeat: Infinity, ease: "easeInOut" },
                  }}
                  style={{ position: "absolute", inset: 0, zIndex: 0, display: "flex", alignItems: "center", justifyContent: "center" }}
                >
                  {[...Array(20)].map((_, i) => (
                    <div
                      key={i}
                      style={{
                        position: "absolute",
                        height: "250%",
                        width: "0.75rem",
                        background: "linear-gradient(to top, transparent, rgba(255,191,0,0.4), transparent)",
                        filter: "blur(2px)",
                        transform: `rotate(${i * 18}deg)`,
                      }}
                    />
                  ))}
                </motion.div>
                <div
                  style={{
                    width: "100%", height: "100%", borderRadius: "9999px",
                    background: "var(--color-fun-yellow)", position: "relative", zIndex: 10,
                    filter: "drop-shadow(0 0 50px rgba(255,191,0,0.8))",
                  }}
                />
              </>
            )}
          </div>
        </motion.div>
      )}

      {/* Day mode → clouds. Night mode → twinkling stars. (XOR like the prototype) */}
      {!dark ? (
        clouds.map((c, index) => (
        <motion.div
          key={`cloud-${index}`}
          initial={{ x: c.startX, y: 0 }}
          animate={{
            x: [c.startX, c.endX, c.startX],
            y: [0, 20, -20, 0],
            scale: [1, 1.05, 0.95, 1],
          }}
          transition={{
            x: { duration: c.duration, repeat: Infinity, ease: "easeInOut" },
            y: { duration: c.duration * 0.7, repeat: Infinity, ease: "easeInOut" },
            scale: { duration: c.duration * 0.5, repeat: Infinity, ease: "easeInOut" },
          }}
          style={{ position: "absolute", top: c.top, [c.side]: -150, zIndex: 5 }}
        >
          <div style={{ position: "relative", transform: `scale(${c.scale})` }}>
            <svg viewBox="0 0 100 60" style={{ width: "12rem", height: "8rem", overflow: "visible", filter: "drop-shadow(0 4px 6px rgba(0,0,0,0.1))" }}>
              <path
                d="M25,50 C15,50 5,42 5,30 C5,18 15,10 25,10 C27,10 29,11 31,12 C35,5 42,0 50,0 C60,0 68,8 70,18 C72,17 74,17 76,17 C85,17 92,24 92,33 C92,42 85,50 76,50 L25,50 Z"
                fill="white"
                stroke="black"
                strokeWidth="2.5"
                strokeLinejoin="round"
              />
              {c.hasEyes && (
                <foreignObject x="25" y="15" width="50" height="30">
                  <div style={{ display: "flex", alignItems: "center", justifyContent: "center", height: "100%", transform: "scale(0.5)" }}>
                    <div style={{ display: "flex", gap: "1rem" }}>
                      {[0, 1].map((eye) => (
                        <div key={eye} style={{ position: "relative", width: "1.5rem", height: "1.25rem" }}>
                          <svg viewBox="0 0 40 20" style={{ position: "absolute", inset: 0, width: "100%", height: "100%", color: "#000", fill: "none", stroke: "currentColor", strokeWidth: 2 }}>
                            <motion.path
                              d="M 5,15 Q 20,2 35,15"
                              strokeLinecap="round"
                              animate={{ d: ["M 5,15 Q 20,2 35,15", "M 5,15 Q 20,15 35,15", "M 5,15 Q 20,2 35,15"] }}
                              transition={{ duration: 6, repeat: Infinity, delay: eye * 0.4, times: [0, 0.05, 0.1] }}
                            />
                          </svg>
                          <motion.div
                            animate={{ scaleY: [1, 0, 1], opacity: [1, 0, 1] }}
                            transition={{ duration: 6, repeat: Infinity, delay: eye * 0.4, times: [0, 0.05, 0.1] }}
                            style={{
                              position: "absolute", top: "8px", left: "50%",
                              width: "0.5rem", height: "0.5rem",
                              background: "#000", borderRadius: "9999px",
                              transform: "translateX(-50%)",
                            }}
                          />
                        </div>
                      ))}
                    </div>
                  </div>
                </foreignObject>
              )}
            </svg>
          </div>
        </motion.div>
        ))
      ) : (
        stars.map((s) => (
          <motion.div
            key={`star-${s.key}`}
            initial={{ opacity: 0.2, scale: 0.5 }}
            animate={{
              opacity: [0.2, 1, 0.2],
              scale: [0.5, 1.2, 0.5],
              rotate: [0, 45, 0],
            }}
            transition={{
              duration: s.duration,
              repeat: Infinity,
              delay: s.delay,
              ease: "easeInOut",
            }}
            style={{
              position: "absolute", top: s.top, left: s.left,
              color: "rgba(255, 191, 0, 0.8)",
              filter: "drop-shadow(0 0 8px rgba(255,191,0,0.8))",
            }}
          >
            <Star size={s.size} fill="currentColor" />
          </motion.div>
        ))
      )}
    </div>
  );
};

// ─────────────────────────────────────────────────────────────────
// ResearchInput — clicking a mood appends to the free-text field.
// User can always edit / refine the text manually.
// ─────────────────────────────────────────────────────────────────
function ResearchInput({ onAnalyze, isLoading }) {
  const [freeText, setFreeText] = useState("");
  const [birthYear, setBirthYear] = useState("");

  // ─── Multimodal state ───────────────────────────────────────────
  const [imageB64, setImageB64]     = useState(null);
  const [imagePreview, setImagePreview] = useState(null);
  const [isListening, setIsListening] = useState(false);
  const [voiceError, setVoiceError]   = useState(null);

  // Drop mode: the text input morphs into a dropzone when active.
  // Triggered by clicking the image icon. Accepts drag-drop, paste, click.
  const [dropMode, setDropMode]       = useState(false);
  const [isDragOver, setIsDragOver]   = useState(false);

  const recognitionRef = React.useRef(null);
  const baseTextRef    = React.useRef("");
  const fileInputRef   = React.useRef(null);
  const dropZoneRef    = React.useRef(null);

  const appendMood = (m) => {
    const lower = m.toLowerCase();
    setFreeText((prev) => {
      const trimmed = prev.trim();
      if (!trimmed) return lower;
      const re = new RegExp(`\\b${lower}\\b`, "i");
      if (re.test(trimmed)) return prev;
      const endsWithPunct = /[.,;!?]$/.test(trimmed);
      const sep = endsWithPunct ? " " : " and ";
      return trimmed + sep + lower;
    });
  };

  // ─── Voice input via Web Speech API ─────────────────────────────
  const SpeechRecognitionImpl =
    (typeof window !== "undefined") &&
    (window.SpeechRecognition || window.webkitSpeechRecognition);

  const toggleVoice = () => {
    setVoiceError(null);
    if (!SpeechRecognitionImpl) {
      setVoiceError("Voice input needs Chrome, Edge, or Safari.");
      return;
    }
    if (isListening) {
      try { recognitionRef.current && recognitionRef.current.stop(); } catch (e) {}
      setIsListening(false);
      return;
    }
    const rec = new SpeechRecognitionImpl();
    rec.lang = (typeof document !== "undefined" && document.documentElement.lang) || "en-US";
    rec.interimResults = true;
    rec.continuous     = true;

    setFreeText((prev) => {
      const base = prev || "";
      baseTextRef.current = base;
      return base;
    });

    rec.onresult = (event) => {
      let fullTranscript = "";
      for (let i = 0; i < event.results.length; i++) {
        fullTranscript += event.results[i][0].transcript;
      }
      const live = fullTranscript.trim();
      const base = baseTextRef.current;
      const sep = base && live && !/\s$/.test(base) ? " " : "";
      setFreeText(base + sep + live);
    };

    rec.onerror = (e) => {
      setVoiceError(`Mic error: ${e.error || "unknown"}`);
      setIsListening(false);
    };
    rec.onend = () => setIsListening(false);

    recognitionRef.current = rec;
    try {
      rec.start();
      setIsListening(true);
    } catch (e) {
      setVoiceError("Could not start mic.");
    }
  };

  // ─── Image handling ─────────────────────────────────────────────
  const handleImageFile = (file) => {
    if (!file) return;
    if (!file.type.startsWith("image/")) {
      setVoiceError("That doesn't look like an image.");
      return;
    }
    if (file.size > 8 * 1024 * 1024) {
      setVoiceError("Image is too large (max 8 MB).");
      return;
    }
    const reader = new FileReader();
    reader.onload = (ev) => {
      setImageB64(ev.target.result);
      setImagePreview(ev.target.result);
      setVoiceError(null);
      setDropMode(false);
      setIsDragOver(false);
    };
    reader.readAsDataURL(file);
  };

  const onImageIconClick = () => {
    if (imageB64) {
      setImageB64(null);
      setImagePreview(null);
      setDropMode(false);
      return;
    }
    if (dropMode) {
      setDropMode(false);
      return;
    }
    setDropMode(true);
    setTimeout(() => { dropZoneRef.current && dropZoneRef.current.focus(); }, 50);
  };

  const onDropZoneClick = () => {
    if (!dropMode) return;
    fileInputRef.current && fileInputRef.current.click();
  };

  const onImageInputChange = (e) => {
    const f = e.target.files && e.target.files[0];
    handleImageFile(f);
    e.target.value = "";
  };

  const onDrop = (e) => {
    e.preventDefault();
    e.stopPropagation();
    setIsDragOver(false);
    const f = e.dataTransfer.files && e.dataTransfer.files[0];
    handleImageFile(f);
  };

  const onDragOver = (e) => {
    e.preventDefault();
    e.stopPropagation();
    if (dropMode) setIsDragOver(true);
  };

  const onDragLeave = (e) => {
    e.preventDefault();
    setIsDragOver(false);
  };

  const onPaste = (e) => {
    if (!dropMode) return;
    const items = (e.clipboardData || window.clipboardData)?.items || [];
    for (const item of items) {
      if (item.kind === "file" && item.type.startsWith("image/")) {
        e.preventDefault();
        const file = item.getAsFile();
        if (file) handleImageFile(file);
        return;
      }
    }
  };

  const onDropZoneKeyDown = (e) => {
    if (e.key === "Escape") {
      e.preventDefault();
      setDropMode(false);
      setIsDragOver(false);
    }
  };

  React.useEffect(() => {
    if (!dropMode) return;
    const handler = (e) => {
      const items = (e.clipboardData || window.clipboardData)?.items || [];
      for (const item of items) {
        if (item.kind === "file" && item.type.startsWith("image/")) {
          e.preventDefault();
          const file = item.getAsFile();
          if (file) handleImageFile(file);
          return;
        }
      }
    };
    window.addEventListener("paste", handler);
    return () => window.removeEventListener("paste", handler);
  }, [dropMode]);

  const clearImage = () => {
    setImageB64(null);
    setImagePreview(null);
  };

  const hasContent = !!freeText.trim() || !!imageB64;
  const canSubmit  = !!birthYear.trim() && hasContent;

  const handleSubmit = (e) => {
    if (e && e.preventDefault) e.preventDefault();
    if (!canSubmit || isLoading) return;
    if (isListening) {
      try { recognitionRef.current && recognitionRef.current.stop(); } catch (err) {}
      setIsListening(false);
    }
    onAnalyze({
      freeText:      freeText.trim(),
      selectedMoods: [],
      birthYear:     birthYear.trim(),
      imageB64:      imageB64,
    });
  };

  const suggestions = ["Lonely", "Anxious", "Empty", "Angry", "Nostalgic", "Grieving", "Discouraged", "Lost", "Sad", "Melancholic", "Exhausted", "Cynical"];
  const voiceSupported = !!SpeechRecognitionImpl;

  // ─── Visual: input vs. dropzone ─────────────────────────────────
  const inputBoxStyle = {
    width: "100%",
    minWidth: 0,
    background: dropMode
      ? (isDragOver ? "var(--color-fun-mint)" : "var(--color-fun-pink)")
      : "var(--color-fun-white)",
    border: dropMode
      ? "4px dashed var(--color-fun-ink)"
      : "4px solid var(--color-fun-ink)",
    padding: "2.5rem 9rem 2.5rem 3rem",
    borderRadius: "3rem",
    fontSize: "clamp(1.5rem, 4vw, 3rem)",
    fontFamily: "var(--font-display)",
    fontWeight: 900,
    outline: "none",
    boxShadow: "0 0 0 4px var(--shadow-stroke), 8px 8px 0 0 var(--shadow-color)",
    transition: "background-color 0.25s ease, border-color 0.25s ease",
    boxSizing: "border-box",
    /* textarea-specific — auto-expand vertically */
    resize: "none",
    overflow: "hidden",
    lineHeight: 1.3,
    display: "block",
  };

  return (
    <div style={{ width: "100%", transition: "all 0.5s", opacity: isLoading ? 0.5 : 1 }}>
      <div style={{ display: "flex", flexDirection: "column", gap: "1.5rem", position: "relative" }}>

        {/* ─── Free-text input OR dropzone (same shell, swapped content) ─── */}
        <div style={{ position: "relative" }}>
          {!dropMode ? (
            <>
              {/* "How are you, right now?" — floating label, offset right */}
              <motion.p
                animate={{ y: [0, -5, 2, 0], rotate: [0, 0.4, -0.4, 0] }}
                transition={{ duration: 9, repeat: Infinity, ease: "easeInOut" }}
                whileHover={{
                  y: -12,
                  rotate: (Math.random() - 0.5) * 10,
                  transition: { duration: 0.2, type: "spring", stiffness: 400, damping: 10 },
                }}
                whileTap={{ rotate: (Math.random() - 0.5) * 20, scale: 0.97 }}
                style={{
                  fontFamily: "var(--font-display)",
                  fontWeight: 900,
                  fontSize: "clamp(1.25rem, 2.5vw, 1.875rem)",
                  marginBottom: "0.75rem",
                  marginLeft: "2rem",
                  color: "var(--color-fun-ink)",
                  cursor: "default",
                  display: "inline-block",
                }}
              >
                How are you, right now?
              </motion.p>
              {/* Auto-grow textarea — expands vertically as user types */}
              <textarea
                value={freeText}
                onChange={(e) => {
                  setFreeText(e.target.value);
                  // Reset height then set to scrollHeight so it grows downward
                  e.target.style.height = "auto";
                  e.target.style.height = e.target.scrollHeight + "px";
                }}
                onKeyDown={(e) => {
                  // Submit on Enter (without Shift), allow Shift+Enter for newline
                  if (e.key === "Enter" && !e.shiftKey && canSubmit) {
                    e.preventDefault();
                    handleSubmit(e);
                  }
                }}
                placeholder={isListening ? "Listening… speak freely" : "a strange morning, I don't know why..."}
                rows={1}
                data-surface="white"
                style={inputBoxStyle}
              />
            </>
          ) : (
            <motion.div
              ref={dropZoneRef}
              tabIndex={0}
              role="button"
              aria-label="Drop, paste, or click to add an image"
              onClick={onDropZoneClick}
              onDragEnter={onDragOver}
              onDragOver={onDragOver}
              onDragLeave={onDragLeave}
              onDrop={onDrop}
              onPaste={onPaste}
              onKeyDown={onDropZoneKeyDown}
              initial={{ scale: 0.98, opacity: 0.6 }}
              animate={{ scale: 1, opacity: 1 }}
              transition={{ duration: 0.25 }}
              style={{
                ...inputBoxStyle,
                cursor: "pointer",
                display: "flex",
                alignItems: "center",
                gap: "1.25rem",
                userSelect: "none",
              }}
            >
              <motion.div
                animate={{ y: isDragOver ? -6 : 0, rotate: isDragOver ? -4 : 0 }}
                transition={{ duration: 0.2 }}
                style={{ display: "flex", alignItems: "center", justifyContent: "center" }}
              >
                <ImagePlus size={48} strokeWidth={3} />
              </motion.div>
              <div style={{ display: "flex", flexDirection: "column", gap: "0.25rem", flex: 1 }}>
                <div style={{ fontSize: "clamp(1.25rem, 2.8vw, 2rem)", lineHeight: 1.1 }}>
                  {isDragOver ? "Drop it!" : "Drop a meme or photo"}
                </div>
                <div style={{
                  fontSize: "clamp(0.85rem, 1.4vw, 1.05rem)",
                  fontWeight: 600, opacity: 0.75, letterSpacing: "0.02em",
                }}>
                  {isDragOver
                    ? "Release to attach"
                    : "or paste with Cmd/Ctrl + V · or click to browse"}
                </div>
              </div>
            </motion.div>
          )}

          {/* Mic button */}
          <motion.button
            type="button"
            onClick={toggleVoice}
            disabled={isLoading || !voiceSupported || dropMode}
            title={
              dropMode ? "Cancel image first" :
              voiceSupported ? (isListening ? "Stop listening" : "Speak instead") :
              "Voice not supported in this browser"
            }
            whileTap={{ scale: 0.92 }}
            animate={isListening ? { scale: [1, 1.12, 1] } : { scale: 1 }}
            transition={isListening
              ? { duration: 1, repeat: Infinity, ease: "easeInOut" }
              : { duration: 0.2 }}
            style={{
              position: "absolute", right: "5.25rem", top: "50%", transform: "translateY(-50%)",
              width: "3.5rem", height: "3.5rem", borderRadius: "9999px",
              background: isListening ? "var(--color-fun-pink)" : "var(--color-fun-mint)",
              border: "3px solid var(--color-fun-ink)", color: "var(--color-fun-ink)",
              display: "flex", alignItems: "center", justifyContent: "center",
              cursor: (voiceSupported && !dropMode) ? "pointer" : "not-allowed",
              opacity: (voiceSupported && !dropMode) ? 1 : 0.3,
              boxShadow: "0 0 0 3px var(--shadow-stroke), 4px 4px 0 0 var(--shadow-color)",
              zIndex: 2,
            }}
          >
            {isListening ? <MicOff size={24} strokeWidth={3} /> : <Mic size={24} strokeWidth={3} />}
          </motion.button>

          {/* Image icon: opens dropzone OR closes/clears it */}
          <motion.button
            type="button"
            onClick={onImageIconClick}
            disabled={isLoading}
            title={
              imageB64 ? "Remove image" :
              dropMode ? "Cancel" :
              "Drop, paste, or browse an image"
            }
            whileHover={{ rotate: dropMode ? 0 : 6 }}
            whileTap={{ scale: 0.92 }}
            animate={dropMode ? { rotate: 90 } : { rotate: 0 }}
            transition={{ duration: 0.25 }}
            style={{
              position: "absolute", right: "1.25rem", top: "50%", transform: "translateY(-50%)",
              width: "3.5rem", height: "3.5rem", borderRadius: "9999px",
              background: imageB64
                ? "var(--color-fun-pink)"
                : (dropMode ? "var(--color-fun-pink)" : "var(--color-fun-yellow, #FFE066)"),
              border: "3px solid var(--color-fun-ink)", color: "var(--color-fun-ink)",
              display: "flex", alignItems: "center", justifyContent: "center", cursor: "pointer",
              boxShadow: "0 0 0 3px var(--shadow-stroke), 4px 4px 0 0 var(--shadow-color)",
              zIndex: 2,
            }}
          >
            {(imageB64 || dropMode)
              ? <X size={24} strokeWidth={3} />
              : <ImagePlus size={24} strokeWidth={3} />}
          </motion.button>

          <input
            ref={fileInputRef}
            type="file"
            accept="image/*"
            onChange={onImageInputChange}
            style={{ display: "none" }}
          />
        </div>

        {/* ─── Image preview ─── */}
        {imagePreview && (
          <motion.div
            initial={{ opacity: 0, y: -10 }}
            animate={{ opacity: 1, y: 0 }}
            style={{
              display: "flex", alignItems: "center", gap: "1rem",
              padding: "1rem 1.25rem", borderRadius: "1.5rem",
              background: "var(--color-fun-white)",
              border: "3px solid var(--color-fun-ink)",
              boxShadow: "0 0 0 3px var(--shadow-stroke), 4px 4px 0 0 var(--shadow-color)",
            }}
          >
            <img
              src={imagePreview}
              alt="Uploaded"
              style={{
                width: "5rem", height: "5rem", objectFit: "cover",
                borderRadius: "1rem", border: "3px solid var(--color-fun-ink)",
              }}
            />
            <div style={{ flex: 1, fontFamily: "var(--font-display)", fontWeight: 700 }}>
              <div style={{ fontSize: "1.1rem" }}>Image attached</div>
              <div style={{ fontSize: "0.9rem", opacity: 0.7 }}>
                Ami will read its emotional subtext alongside your words.
              </div>
            </div>
            <motion.button
              type="button"
              onClick={clearImage}
              whileHover={{ rotate: 90 }}
              style={{
                width: "2.5rem", height: "2.5rem", borderRadius: "9999px",
                background: "var(--color-fun-pink)", border: "3px solid var(--color-fun-ink)",
                cursor: "pointer", display: "flex", alignItems: "center", justifyContent: "center",
              }}
              title="Remove image"
            >
              <X size={18} strokeWidth={3} />
            </motion.button>
          </motion.div>
        )}

        {voiceError && (
          <div style={{
            padding: "0.75rem 1rem", borderRadius: "1rem",
            background: "var(--color-fun-pink)", border: "2px solid var(--color-fun-ink)",
            fontFamily: "var(--font-display)", fontWeight: 700, fontSize: "0.95rem",
          }}>
            {voiceError}
          </div>
        )}

        {/* ─── Birth year + submit ─── */}
        <div style={{ display: "flex", flexWrap: "wrap", gap: "1.5rem", alignItems: "center" }}>
          <div style={{ position: "relative", flex: 1, minWidth: "200px" }}>
            <input
              type="number"
              value={birthYear}
              onChange={(e) => setBirthYear(e.target.value)}
              onKeyDown={(e) => { if (e.key === "Enter" && canSubmit) handleSubmit(e); }}
              placeholder="Your birth year (e.g. 1995)"
              min="1940" max="2020"
              data-surface="white"
              style={{
                width: "100%", background: "var(--color-fun-white)",
                border: "4px solid var(--color-fun-ink)", padding: "1.5rem 2.5rem",
                borderRadius: "2rem", fontSize: "clamp(1.25rem, 2.5vw, 1.875rem)",
                fontFamily: "var(--font-display)", fontWeight: 700,
                outline: "none", boxShadow: "0 0 0 4px var(--shadow-stroke), 8px 8px 0 0 var(--shadow-color)",
              }}
            />
          </div>

          <motion.button
            type="button"
            onClick={handleSubmit}
            disabled={isLoading || !canSubmit}
            animate={!isLoading && canSubmit ? { y: [0, -8, 0], scale: [1, 1.05, 1] } : {}}
            transition={{ duration: 4, repeat: Infinity, ease: "easeInOut" }}
            whileHover={{ rotate: 6 }}
            style={{
              background: "var(--color-fun-mint)", color: "black",
              padding: "1.5rem 3rem", border: "4px solid black", borderRadius: "9999px",
              display: "flex", alignItems: "center", justifyContent: "center",
              gap: "1rem", cursor: "pointer",
              opacity: (!isLoading && !canSubmit) ? 0.3 : 1,
              boxShadow: "0 0 0 4px var(--shadow-stroke), 8px 8px 0 0 var(--shadow-color)",
              fontFamily: "var(--font-display)", fontWeight: 900, fontSize: "1.5rem",
            }}
          >
            <span>{isLoading ? "AMI IS THINKING..." : "FIND MY WELL-BEING"}</span>
            {isLoading ? (
              <motion.div animate={{ rotate: 360 }} transition={{ duration: 1, repeat: Infinity, ease: "linear" }}>
                <Loader2 size={36} />
              </motion.div>
            ) : (
              <Sparkles size={36} strokeWidth={3} />
            )}
          </motion.button>
        </div>

        {/* ─── Mood suggestion chips ─── */}
        <div style={{ display: "flex", flexWrap: "wrap", gap: "0.75rem", justifyContent: "center", marginTop: "0.5rem" }}>
          {suggestions.map((suggestion, si) => (
            <motion.button
              key={suggestion}
              type="button"
              onClick={() => appendMood(suggestion)}
              animate={{ y: [0, -6 - (si % 3) * 2, 0], rotate: [0, (si % 2 === 0 ? 1 : -1), 0] }}
              transition={{ duration: 5 + si * 0.4, repeat: Infinity, ease: "easeInOut", delay: si * 0.15 }}
              whileHover={{
                y: -14,
                rotate: (Math.random() - 0.5) * 14,
                scale: 1.1,
                zIndex: 10,
                transition: { duration: 0.2, type: "spring", stiffness: 400, damping: 10 },
              }}
              whileTap={{ scale: 0.93, rotate: (Math.random() - 0.5) * 25 }}
              style={{
                background: "var(--color-fun-white)", color: "var(--color-fun-ink)",
                padding: "0.75rem 1.5rem", border: "3px solid var(--color-fun-ink)",
                borderRadius: "9999px", fontFamily: "var(--font-display)", fontWeight: 700,
                fontSize: "1.5rem", cursor: "pointer",
                boxShadow: "0 0 0 4px var(--shadow-stroke), 8px 8px 0 0 var(--shadow-color)",
                transition: "background-color 0.2s ease",
              }}
            >
              <FloatingWords text={suggestion} intensity={0.5} />
            </motion.button>
          ))}
        </div>
      </div>
    </div>
  );
}

// ─────────────────────────────────────────────────────────────────
// RecommendationCard
// ─────────────────────────────────────────────────────────────────
// ─────────────────────────────────────────────────────────────────
// AnecdoteOrDescription — splits the anecdote into body + question card
// ─────────────────────────────────────────────────────────────────
function splitAnecdote(text) {
  if (!text) return { body: "", question: "" };
  // The engaging question is always the last sentence ending with ?
  const lastQ = text.lastIndexOf("?");
  if (lastQ === -1 || lastQ < text.length * 0.4) return { body: text, question: "" };
  // Walk back from the ? to find the sentence start
  const before = text.slice(0, lastQ);
  const lastStop = Math.max(
    before.lastIndexOf(". "),
    before.lastIndexOf("! "),
    before.lastIndexOf("\n")
  );
  const questionStart = lastStop === -1 ? 0 : lastStop + 2;
  const body = text.slice(0, questionStart).trim();
  const question = text.slice(questionStart, lastQ + 1).trim();
  return { body: body || text, question };
}

function AnecdoteOrDescription({ recommendation }) {
  if (recommendation.type !== "anecdote") {
    return (
      <p data-secondary-text style={{
        fontSize: "1.5rem",
        fontFamily: "var(--font-body)",
        fontWeight: 500,
        color: "var(--text-secondary)",
        marginBottom: "3rem",
        lineHeight: 1.6,
        fontStyle: "italic",
      }}>
        <FloatingWords text={recommendation.description} intensity={0.4} />
      </p>
    );
  }
  const { body, question } = splitAnecdote(recommendation.description);
  return (
    <>
      <p data-secondary-text style={{
        fontSize: "1.4rem",
        fontFamily: "var(--font-body)",
        fontWeight: 500,
        color: "var(--text-secondary)",
        marginBottom: question ? "1.5rem" : "3rem",
        lineHeight: 1.6,
        fontStyle: "italic",
      }}>
        <FloatingWords text={body} intensity={0.4} />
      </p>
      {question && (
        <motion.div
          animate={{ y: [0, -5, 0], rotate: [0, 0.5, -0.5, 0] }}
          transition={{ duration: 6, repeat: Infinity, ease: "easeInOut" }}
          style={{
            background: "var(--color-fun-pink)",
            border: "4px solid var(--color-fun-ink)",
            borderRadius: "2rem",
            padding: "1.5rem 2rem",
            marginBottom: "2rem",
            transform: "rotate(-1deg)",
            boxShadow: "0 0 0 4px var(--shadow-stroke), 6px 6px 0 0 var(--shadow-color)",
          }}
        >
          <span style={{
            fontFamily: "var(--font-sketch)",
            fontSize: "0.75rem",
            letterSpacing: "0.2em",
            textTransform: "uppercase",
            color: "rgba(0,0,0,0.5)",
            display: "block",
            marginBottom: "0.5rem",
          }}>And you?</span>
          <p style={{
            fontFamily: "var(--font-display)",
            fontWeight: 900,
            fontSize: "clamp(1.25rem, 2.5vw, 1.75rem)",
            color: "var(--color-fun-ink)",
            lineHeight: 1.25,
            margin: 0,
          }}>
            <FloatingWords text={question} intensity={0.7} />
          </p>
        </motion.div>
      )}
    </>
  );
}

function RecommendationCard({ recommendation, index, typeIndex = 0, onOpenDetails }) {
  const bgColor = index % 3 === 0 ? "var(--color-fun-blue)"
    : index % 3 === 1 ? "var(--color-fun-yellow)"
    : "var(--color-fun-mint)";
  const Icon = recommendation.type === "film" ? Video
    : recommendation.type === "series" ? Tv
    : Book;

  const url = externalUrlFor(recommendation);
  const details = recommendation.details || null;

  const detailEntries = useMemo(() => {
    if (!details) return [];
    const out = [];
    const fmt = (k) => k.replace(/([A-Z])/g, " $1").replace(/^./, (c) => c.toUpperCase());
    Object.entries(details).forEach(([k, v]) => {
      if (v == null || v === "") return;
      const value = Array.isArray(v) ? v.join(", ") : String(v);
      out.push([fmt(k), value]);
    });
    if (recommendation.creator && recommendation.creator !== "Memory") {
      out.unshift([recommendation.type === "novel" ? "Author" : recommendation.type === "film" ? "Director" : recommendation.type === "series" ? "Creator" : "By", recommendation.creator]);
    }
    if (recommendation.year && recommendation.year !== "Memory") {
      out.unshift(["Year", recommendation.year]);
    }
    return out;
  }, [details, recommendation]);

  return (
    <motion.div
      initial={{ opacity: 0, scale: 0.9 }}
      whileInView={{ opacity: 1, scale: 1 }}
      viewport={{ once: true }}
      transition={{ type: "spring", stiffness: 100, damping: 12 }}
      style={{ width: "100%" }}
    >
      <motion.div
        animate={{ y: [0, -10, 0] }}
        transition={{ duration: 7 + index, repeat: Infinity, ease: "easeInOut" }}
        className="sticker-shadow sticker-shadow-hover"
        data-surface="white"
        style={{
          position: "relative", padding: "3rem", borderRadius: "3.5rem",
          border: "4px solid var(--color-fun-ink)",
          transition: "all 0.3s",
          background: "var(--color-fun-white)",
        }}
      >
        <div style={{ position: "absolute", top: "1rem", right: "1rem", width: "3rem", height: "3rem", borderTop: "4px solid var(--border-subtle)", borderRight: "4px solid var(--border-subtle)", borderTopRightRadius: "1.5rem" }} />

        <div style={{ display: "flex", alignItems: "center", gap: "1rem", marginBottom: "2rem" }}>
          <motion.div
            animate={{ y: [0, -6, 0], rotate: [3, -3, 3] }}
            transition={{ duration: 3 + index % 2, repeat: Infinity, ease: "easeInOut" }}
            whileHover={{ scale: 1.2, rotate: -10, y: -10 }}
            whileTap={{ scale: 0.9 }}
            style={{
              width: "3.5rem", height: "3.5rem", borderRadius: "9999px",
              border: "4px solid var(--color-fun-ink)",
              display: "flex", alignItems: "center", justifyContent: "center",
              background: bgColor,
            }}
          >
            <span style={{ color: "var(--color-fun-ink)" }}><Icon size={32} /></span>
          </motion.div>
          <span style={{ fontFamily: "var(--font-sketch)", fontSize: "1.875rem", color: "var(--text-secondary)" }}>
            <FloatingWords
              text={typeLabel(recommendation.type, typeIndex)}
              intensity={0.6}
            />
          </span>
        </div>

        {recommendation.intro && (
          <motion.p
            initial={{ opacity: 0, y: -6 }}
            whileInView={{ opacity: 1, y: 0 }}
            viewport={{ once: true }}
            transition={{ delay: 0.15 + index * 0.05, duration: 0.5 }}
            style={{
              fontFamily: "var(--font-sketch)",
              fontSize: "clamp(1.1rem, 1.6vw, 1.5rem)",
              fontStyle: "italic",
              color: "var(--text-secondary)",
              marginBottom: "1rem",
              lineHeight: 1.4,
              maxWidth: "90%",
            }}
          >
            {recommendation.intro}
          </motion.p>
        )}

        <h3
          className="storybook-title"
          style={{
            fontSize: "clamp(2rem, 5vw, 4.5rem)",
            lineHeight: 1.1,
            color: "white",
            marginBottom: "2rem",
            filter: "drop-shadow(0 10px 8px rgba(0,0,0,0.04)) drop-shadow(0 4px 3px rgba(0,0,0,0.1))",
          }}
        >
          <FloatingWords text={recommendation.title} />
        </h3>

        {recommendation.poster && (
          <motion.div
            whileHover={{
              scale: 1.04,
              rotate: 0.8,
              y: -6,
              transition: { type: "spring", stiffness: 300, damping: 18 },
            }}
            whileTap={{ scale: 0.96, rotate: -1.2 }}
            style={{
              width: "100%",
              marginBottom: "2rem",
              borderRadius: "1.5rem",
              overflow: "hidden",
              border: "3px solid var(--color-fun-ink)",
              boxShadow: "0 0 0 3px var(--shadow-stroke), 6px 6px 0 0 var(--shadow-color)",
              transform: "rotate(-0.6deg)",
              background: "var(--color-fun-white)",
              cursor: "pointer",
            }}
          >
            <img
              src={upgradePosterQuality(recommendation.poster)}
              alt={recommendation.title}
              loading="lazy"
              style={{
                display: "block",
                width: "100%",
                height: "auto",
                maxHeight: "320px",
                objectFit: "cover",
                objectPosition: "top center",
              }}
              onError={(e) => { e.currentTarget.parentElement.style.display = "none"; }}
            />
          </motion.div>
        )}

        <AnecdoteOrDescription recommendation={recommendation} />

        <div className="ami-why-block" style={{
          background: "var(--color-fun-yellow)",
          padding: "2.5rem",
          border: "4px solid var(--color-fun-ink)",
          position: "relative",
          overflow: "hidden",
          borderRadius: "2.5rem",
          boxShadow: "inset 0 2px 4px rgba(0,0,0,0.06)",
          transform: "rotate(1deg)",
        }}>
          <div style={{ display: "flex", alignItems: "baseline", gap: "0.75rem", flexWrap: "wrap", marginBottom: "1rem" }}>
            <p style={{
              fontFamily: "var(--font-sketch)",
              fontSize: "1.5rem",
              color: "var(--text-muted)",
              textDecoration: "underline",
              margin: 0,
            }}>
              <FloatingWords text="A little secret..." intensity={0.5} />
            </p>
            {recommendation.mechanism && <MechanismPill mechanismId={recommendation.mechanism} color={undefined} />}
          </div>
          <p style={{
            fontSize: "1.5rem",
            fontFamily: "var(--font-body)",
            fontWeight: 600,
            color: "var(--color-fun-ink)",
            lineHeight: 1.3,
          }}>
            <FloatingWords text={recommendation.artisticConnection} intensity={0.5} />
          </p>
          {recommendation.paper && (
            <p style={{
              marginTop: "1rem",
              fontSize: "0.85rem",
              fontFamily: "var(--font-body)",
              fontStyle: "italic",
              color: "rgba(0,0,0,0.55)",
              lineHeight: 1.45,
              borderTop: "1.5px dashed rgba(0,0,0,0.18)",
              paddingTop: "0.75rem",
            }}>
              {recommendation.paper}
            </p>
          )}
        </div>

        {(url || detailEntries.length > 0) && (
          <div style={{ display: "flex", gap: "1rem", marginTop: "3rem" }}>
            {url && (
              <motion.a
                href={url}
                target="_blank"
                rel="noopener noreferrer"
                whileHover={{ y: -4 }}
                whileTap={{ y: 0, scale: 0.98 }}
                className="sticker-shadow"
                style={{
                  flex: 1,
                  padding: "1.25rem 2rem",
                  background: "var(--color-fun-orange)",
                  color: "white",
                  border: "4px solid var(--color-fun-ink)",
                  borderRadius: "1.5rem",
                  fontFamily: "var(--font-display)",
                  fontWeight: 900,
                  fontSize: "1.5rem",
                  textAlign: "center",
                  textDecoration: "none",
                  cursor: "pointer",
                  transition: "transform 0.2s",
                }}
              >
                <FloatingWords text="LET'S GO!" />
              </motion.a>
            )}
            {detailEntries.length > 0 && (
              <motion.button
                onClick={() => onOpenDetails && onOpenDetails({ recommendation, detailEntries, typeIndex })}
                whileHover={{ rotate: 10 }}
                whileTap={{ scale: 0.95 }}
                aria-label="Show details"
                className="sticker-shadow"
                style={{
                  padding: "1.25rem",
                  background: "var(--color-fun-white)",
                  color: "var(--color-fun-ink)",
                  border: "4px solid var(--color-fun-ink)",
                  borderRadius: "1.5rem",
                  cursor: "pointer",
                  transition: "transform 0.2s",
                  display: "flex", alignItems: "center", justifyContent: "center",
                }}
              >
                <ArrowUpRight size={36} strokeWidth={4} />
              </motion.button>
            )}
          </div>
        )}
      </motion.div>
    </motion.div>
  );
}

// ─────────────────────────────────────────────────────────────────
// MagazineView
// ─────────────────────────────────────────────────────────────────
const FlipPage = React.forwardRef(function FlipPage(props, ref) {
  const isLeftPage = (props.number % 2) === 0;
  const stop = (e) => { e.stopPropagation(); };

  return (
    <div
      ref={ref}
      className="ami-flip-page page"
      style={{
        width: props.pageW,
        height: props.pageH,
        background: "var(--color-fun-white)",
        overflow: "hidden",
        position: "relative",
      }}
      data-density="soft"
    >
      <div
        aria-hidden="true"
        style={{
          position: "absolute",
          top: 0, bottom: 0,
          [isLeftPage ? "right" : "left"]: 0,
          width: "9%",
          maxWidth: "60px",
          pointerEvents: "none",
          zIndex: 2,
          background: isLeftPage
            ? "linear-gradient(to right, rgba(0,0,0,0) 0%, rgba(0,0,0,0.05) 55%, rgba(0,0,0,0.28) 100%)"
            : "linear-gradient(to left,  rgba(0,0,0,0) 0%, rgba(0,0,0,0.05) 55%, rgba(0,0,0,0.28) 100%)",
        }}
      />
      <div
        className="ami-flip-page-content"
        onPointerDownCapture={stop}
        onMouseDownCapture={stop}
        onTouchStartCapture={stop}
        style={{
          position: "absolute",
          top: 0, left: 0, right: 0, bottom: 0,
          padding: 0,
          zIndex: 1,
        }}
      >
        {props.children}
      </div>
    </div>
  );
});

function MagazineView({ result, currentMood, onSurpriseMe, onReturn, loading }) {
  const recs = result.recommendations || [];
  const totalSpreads = 1 + recs.length;
  const totalPages = totalSpreads * 2;

  const [spread, setSpread] = useState(0);
  const [detailsOpenFor, setDetailsOpenFor] = useState(null);
  const bookRef = React.useRef(null);

  const THEMES = [
    { hex: "#38BDF8", soft: "rgba(56,189,248,0.18)" },
    { hex: "#E8B4AA", soft: "rgba(232,180,170,0.22)" },
    { hex: "#FFBF00", soft: "rgba(255,191,0,0.20)" },
    { hex: "#FFB347", soft: "rgba(255,179,71,0.22)" },
  ];
  const accent = THEMES[spread % THEMES.length];

  const typeIndexes = useMemo(() => {
    const counters = {};
    return recs.map((r) => {
      const t = r.type || "discovery";
      counters[t] = (counters[t] || 0) + 1;
      return counters[t] - 1;
    });
  }, [recs]);

  const measure = () => {
    const margin = 32;
    const reserved = 140;
    const ASPECT = 17 / 11;
    const availW = Math.max(320, window.innerWidth  - margin * 2);
    const availH = Math.max(280, window.innerHeight - reserved - margin);
    let w = availW, h = w / ASPECT;
    if (h > availH) { h = availH; w = h * ASPECT; }
    w = Math.floor(w); h = Math.floor(h);
    return { w, h, pw: Math.floor(w / 2), ph: h };
  };
  const [bookSize, setBookSize] = useState(() =>
    (typeof window !== "undefined" ? measure() : { w: 900, h: 580, pw: 450, ph: 580 })
  );
  useEffect(() => {
    const recalc = () => setBookSize(measure());
    recalc();
    window.addEventListener("resize", recalc);
    return () => window.removeEventListener("resize", recalc);
  }, []);

  const flipBookAvailable =
    typeof HTMLFlipBook === "function" ||
    (HTMLFlipBook && typeof HTMLFlipBook === "object");

  const [mountTick, setMountTick] = useState(0);
  useEffect(() => {
    const id = requestAnimationFrame(() => setMountTick(1));
    return () => cancelAnimationFrame(id);
  }, []);

  const renderOvertureLeft = (theme) => (
    <div style={{
      height: "100%",
      display: "flex", alignItems: "center", justifyContent: "center",
      background: "linear-gradient(135deg, var(--color-fun-white) 0%, var(--color-fun-white) 60%, " + theme.soft + " 100%)",
    }}>
      <motion.div
        initial={{ scale: 0.7, opacity: 0 }}
        animate={{ scale: 1.2, opacity: 1 }}
        transition={{ duration: 0.9, delay: 0.1 }}
        style={{ display: "flex", alignItems: "center", justifyContent: "center" }}
      >
        <MoodShape mood={currentMood} />
      </motion.div>
    </div>
  );

  const renderOvertureRight = (theme) => (
    <div style={{
      height: "100%",
      background: "var(--color-fun-white)",
      display: "grid", gridTemplateRows: "auto 1fr auto",
      padding: "7% 8% 5%", gap: "1.75rem",
      position: "relative",
    }}>
      <div style={{
        display: "flex", justifyContent: "space-between", alignItems: "baseline",
        paddingBottom: "0.85rem",
        borderBottom: "1px solid var(--border-soft)",
      }}>
        <span style={{
          fontFamily: "var(--font-display)", fontWeight: 900,
          fontSize: "clamp(1.1rem, 1.7vw, 1.7rem)",
          letterSpacing: "-0.04em", color: "var(--color-fun-ink)",
        }}>AMI.</span>
        <span style={{
          fontFamily: "var(--font-body)", fontSize: "0.6rem",
          letterSpacing: "0.3em", textTransform: "uppercase",
          color: "var(--text-faint)", fontWeight: 700,
        }}>Issue One</span>
      </div>
      <div style={{
        position: "relative", paddingLeft: "2.25rem",
        display: "flex", alignItems: "center", minHeight: 0,
      }}>
        <Quote size={44} style={{
          position: "absolute", left: "-0.3rem", top: "-0.2rem",
          color: theme.hex, opacity: 0.4,
        }} />
        <p style={{
          fontFamily: "var(--font-display)", fontWeight: 900, fontStyle: "italic",
          fontSize: "clamp(1.15rem, 1.75vw, 1.9rem)",
          lineHeight: 1.3, letterSpacing: "-0.015em",
          color: "var(--color-fun-ink)", margin: 0,
        }}>{result.thesis}</p>
      </div>
      <div style={{
        display: "flex", justifyContent: "flex-end", alignItems: "flex-end",
        paddingTop: "0.5rem", borderTop: "1px solid var(--border-soft)",
      }}>
        <span style={{
          fontFamily: "var(--font-display)", fontWeight: 900,
          fontSize: "1.6rem", letterSpacing: "-0.04em",
          lineHeight: 1, color: "var(--text-faint)",
        }}>02</span>
      </div>
    </div>
  );

  const renderRecLeft = (rec, recIdx, theme) => {
    const iconFor = (t) =>
      t === "film"     ? Video :
      t === "series"   ? Tv    :
      t === "novel"    ? Book  :
      t === "anecdote" ? Heart :
      Sparkles;
    const Icon = iconFor(rec.type);
    const label = typeLabel(rec.type, typeIndexes[recIdx]);
    const detailsOpen = detailsOpenFor === recIdx;

    return (
      <div style={{
        height: "100%",
        background: "var(--color-fun-white)",
        display: "grid", gridTemplateRows: "auto 1fr auto",
        padding: "6% 7% 4%", gap: "1rem",
        position: "relative",
      }}>
        <div style={{
          display: "flex", alignItems: "center", gap: "0.5rem",
          color: theme.hex, fontFamily: "var(--font-body)",
          fontSize: "0.65rem", fontWeight: 800,
          letterSpacing: "0.35em", textTransform: "uppercase",
        }}>
          <Icon size={14} /> {label}
        </div>
        <div style={{
          minHeight: 0,
          display: "flex", alignItems: "center", justifyContent: "center",
          position: "relative",
        }}>
          <div style={{
            position: "absolute", top: "-1%", left: "50%",
            transform: "translateX(-50%) rotate(3deg)",
            width: "18%", height: "1.6rem",
            background: theme.soft, border: "1px solid " + theme.hex,
            borderRadius: "0.15rem",
            zIndex: 3, opacity: 0.9,
            boxShadow: "0 1px 3px rgba(0,0,0,0.06)",
          }} />
          {rec.poster ? (
            <motion.div
              animate={{ rotate: [-0.5, 0.5, -0.5] }}
              transition={{ duration: 8, repeat: Infinity, ease: "easeInOut" }}
              style={{
                height: "100%", maxHeight: "100%",
                aspectRatio: "2 / 3", maxWidth: "85%",
                border: "10px solid var(--color-fun-white)",
                background: "var(--color-fun-white)",
                boxShadow: "0 12px 32px rgba(0,0,0,0.22), 0 0 0 1px var(--border-soft)",
                overflow: "hidden", position: "relative",
              }}
            >
              <img
                src={upgradePosterQuality(rec.poster)} alt={rec.title}
                style={{ width: "100%", height: "100%", objectFit: "cover", display: "block" }}
                referrerPolicy="no-referrer"
                onError={(e) => { e.currentTarget.style.display = "none"; }}
              />
            </motion.div>
          ) : (
            <div style={{
              height: "100%", aspectRatio: "2 / 3", maxWidth: "85%",
              border: "3px solid var(--color-fun-ink)",
              background: theme.soft,
              display: "flex", flexDirection: "column",
              alignItems: "center", justifyContent: "center",
              padding: "1.5rem", textAlign: "center",
              boxShadow: "0 12px 32px rgba(0,0,0,0.16)",
            }}>
              <Icon size={56} style={{ color: theme.hex, marginBottom: "1rem" }} />
              <div style={{
                fontFamily: "var(--font-sketch)",
                fontSize: "clamp(1rem, 1.6vw, 1.5rem)",
                color: "var(--color-fun-ink)", lineHeight: 1.15,
              }}>{rec.title}</div>
            </div>
          )}
          <button
            type="button"
            onPointerDown={(e) => e.stopPropagation()}
            onMouseDown={(e) => e.stopPropagation()}
            onClick={(e) => { e.stopPropagation(); setDetailsOpenFor(detailsOpen ? null : recIdx); }}
            aria-label={detailsOpen ? "Hide details" : "Show the small print"}
            style={{
              position: "absolute", top: "1rem", right: "1rem",
              width: "2.25rem", height: "2.25rem", borderRadius: "9999px",
              background: "var(--color-fun-ink)", color: "var(--color-fun-white)",
              border: "none", cursor: "pointer", padding: 0,
              display: "flex", alignItems: "center", justifyContent: "center",
              boxShadow: "0 4px 10px rgba(0,0,0,0.2)",
              transition: "transform 0.2s", zIndex: 6,
              transform: detailsOpen ? "rotate(180deg)" : "rotate(0deg)",
            }}
          ><ChevronRight size={16} /></button>
          <AnimatePresence>
            {detailsOpen && (
              <motion.div
                initial={{ opacity: 0, x: 12, scale: 0.95 }}
                animate={{ opacity: 1, x: 0, scale: 1 }}
                exit={{ opacity: 0, x: 12, scale: 0.95 }}
                transition={{ duration: 0.18 }}
                style={{
                  position: "absolute", top: "3.25rem", right: "0.75rem",
                  width: "min(60%, 14rem)", maxHeight: "80%",
                  overflowY: "auto",
                  background: "var(--color-fun-white)",
                  border: "1px solid var(--border-subtle)",
                  borderRadius: "0.75rem",
                  padding: "0.9rem 1.05rem",
                  fontSize: "0.72rem",
                  fontFamily: "var(--font-body)",
                  color: "var(--color-fun-ink)",
                  boxShadow: "0 12px 30px rgba(0,0,0,0.15)",
                  zIndex: 7,
                }}
              >
                <div style={{
                  fontStyle: "italic", color: "var(--text-faint)",
                  borderBottom: "1px solid var(--border-soft)",
                  paddingBottom: "0.4rem", marginBottom: "0.5rem",
                  fontSize: "0.62rem",
                }}>…the small print</div>
                {rec.year && rec.year !== "Memory" && <Row k="Year" v={rec.year} accent={theme.hex} />}
                {rec.creator && rec.creator !== "Memory" && <Row k="By" v={rec.creator} accent={theme.hex} />}
                {rec.details && Object.entries(rec.details).map(([k, v]) => {
                  if (v == null || v === "") return null;
                  const lab = k.replace(/([A-Z])/g, " $1").replace(/^./, (c) => c.toUpperCase());
                  const val = Array.isArray(v) ? v.join(", ") : String(v);
                  return <Row key={k} k={lab} v={val} accent={theme.hex} />;
                })}
              </motion.div>
            )}
          </AnimatePresence>
        </div>
        <div style={{
          display: "flex", justifyContent: "space-between", alignItems: "flex-end",
        }}>
          <span style={{
            fontFamily: "var(--font-body)", fontSize: "0.6rem",
            letterSpacing: "0.3em", textTransform: "uppercase",
            color: "var(--text-faint)", fontWeight: 700,
          }}>{rec.creator && rec.creator !== "Memory" ? rec.creator : "Ami"}</span>
          <span style={{
            fontFamily: "var(--font-display)", fontWeight: 900,
            fontSize: "1.6rem", letterSpacing: "-0.04em",
            lineHeight: 1, color: "var(--text-faint)",
          }}>{String((recIdx + 1) * 2 + 1).padStart(2, "0")}</span>
        </div>
      </div>
    );
  };

  const renderRecRight = (rec, recIdx, theme) => {
    const url = externalUrlFor(rec);
    return (
      <div style={{
        height: "100%",
        background: "var(--color-fun-white)",
        display: "grid",
        gridTemplateRows: "auto auto 1fr auto",
        padding: "6% 7% 4%", gap: "0.75rem",
        position: "relative",
      }}>
        {rec.intro && (
          <p style={{
            fontFamily: "var(--font-sketch)",
            fontSize: "clamp(0.95rem, 1.25vw, 1.2rem)",
            fontStyle: "italic", color: "var(--text-secondary)",
            margin: 0, lineHeight: 1.35,
          }}>{rec.intro}</p>
        )}
        <h2 style={{
          fontFamily: "var(--font-display)", fontWeight: 900,
          textTransform: "uppercase", letterSpacing: "-0.035em",
          lineHeight: 0.92,
          fontSize: "clamp(2rem, 4vw, 3.6rem)",
          color: "var(--color-fun-ink)", margin: 0,
          overflowWrap: "break-word", wordBreak: "break-word", hyphens: "auto",
          display: "-webkit-box", WebkitLineClamp: 4, WebkitBoxOrient: "vertical", overflow: "hidden",
        }}>{rec.title}</h2>
        <div style={{
          minHeight: 0, overflowY: "auto",
          display: "flex", flexDirection: "column",
          gap: "1.1rem", paddingRight: "0.25rem",
        }}>
          <p style={{
            fontFamily: "var(--font-body)",
            fontSize: "clamp(0.9rem, 1.15vw, 1.15rem)",
            fontWeight: 500, lineHeight: 1.55,
            color: "var(--color-fun-ink)", margin: 0,
          }}>{rec.description}</p>
          <div className="ami-why-block" style={{
            background: theme.hex, borderRadius: "1.25rem",
            padding: "1.1rem 1.25rem",
            border: "2px solid var(--color-fun-ink)",
            transform: "rotate(-0.6deg)", position: "relative",
            boxShadow: "0 6px 14px rgba(0,0,0,0.08)",
          }}>
            <div style={{ display: "flex", alignItems: "center", gap: "0.5rem", marginBottom: "0.4rem", flexWrap: "wrap" }}>
              <span className="ami-why-label" style={{
                fontFamily: "var(--font-sketch)", fontSize: "0.85rem",
                fontWeight: 700, textTransform: "uppercase", letterSpacing: "0.2em",
                color: "rgba(0,0,0,0.55)",
              }}>Why this · for you</span>
              {rec.mechanism && <MechanismPill mechanismId={rec.mechanism} color={undefined} />}
            </div>
            <p style={{
              fontFamily: "var(--font-body)",
              fontSize: "clamp(0.85rem, 1.1vw, 1.05rem)",
              fontWeight: 700, lineHeight: 1.45,
              color: "#000", margin: 0,
            }}>{rec.artisticConnection}</p>
            {rec.paper && (
              <p style={{
                marginTop: "0.6rem", paddingTop: "0.5rem",
                borderTop: "1px dashed rgba(0,0,0,0.25)",
                fontFamily: "var(--font-body)", fontStyle: "italic",
                fontSize: "0.7rem", lineHeight: 1.35,
                color: "rgba(0,0,0,0.62)", margin: "0.6rem 0 0 0",
              }}>{rec.paper}</p>
            )}
          </div>
        </div>
        <div style={{
          display: "flex", justifyContent: "space-between", alignItems: "center",
          gap: "0.75rem",
        }}>
          {url ? (
            <motion.a
              href={url} target="_blank" rel="noopener noreferrer"
              whileHover={{ y: -2, scale: 1.03 }} whileTap={{ scale: 0.97 }}
              onPointerDown={(e) => e.stopPropagation()}
              onMouseDown={(e) => e.stopPropagation()}
              style={{
                display: "inline-flex", alignItems: "center", gap: "0.5rem",
                padding: "0.65rem 1.25rem",
                background: "var(--color-fun-ink)", color: "var(--color-fun-white)",
                fontFamily: "var(--font-display)", fontWeight: 900,
                fontSize: "0.85rem", letterSpacing: "0.08em", textTransform: "uppercase",
                borderRadius: "9999px", textDecoration: "none",
                boxShadow: "0 4px 0 0 " + theme.hex,
              }}
            >Let's go <ArrowUpRight size={14} /></motion.a>
          ) : (
            <span style={{
              fontFamily: "var(--font-sketch)", fontSize: "1rem",
              color: "var(--text-muted)", fontStyle: "italic",
            }}>a memory to carry with you</span>
          )}
          <span style={{
            fontFamily: "var(--font-display)", fontWeight: 900,
            fontSize: "1.6rem", letterSpacing: "-0.04em",
            lineHeight: 1, color: "var(--text-faint)",
          }}>{String((recIdx + 1) * 2 + 2).padStart(2, "0")}</span>
        </div>
      </div>
    );
  };


  // ── Anecdote spread: text flows across both pages, no poster ──────
  const renderAnecdoteLeft = (rec, recIdx, theme) => {
    const { body } = splitAnecdote(rec.description || "");
    const iconFor = (t) => t === "anecdote" ? Heart : Sparkles;
    const Icon = iconFor(rec.type);
    const label = typeLabel(rec.type, typeIndexes[recIdx]);
    return (
      <div style={{
        height: "100%",
        background: "var(--color-fun-white)",
        display: "grid", gridTemplateRows: "auto auto 1fr auto",
        padding: "6% 7% 4%", gap: "0.9rem",
        position: "relative",
      }}>
        {/* Type label */}
        <div style={{
          display: "flex", alignItems: "center", gap: "0.5rem",
          color: theme.hex, fontFamily: "var(--font-body)",
          fontSize: "0.65rem", fontWeight: 800,
          letterSpacing: "0.35em", textTransform: "uppercase",
        }}>
          <Icon size={14} /> {label}
        </div>
        {/* Title */}
        <h2 style={{
          fontFamily: "var(--font-display)", fontWeight: 900,
          textTransform: "uppercase", letterSpacing: "-0.035em",
          lineHeight: 0.95, fontSize: "clamp(1.6rem, 3.2vw, 2.8rem)",
          color: "var(--color-fun-ink)", margin: 0,
          overflowWrap: "break-word", wordBreak: "break-word",
        }}>
          {rec.intro && (
            <span style={{
              display: "block",
              fontFamily: "var(--font-sketch)",
              fontSize: "clamp(0.85rem, 1.1vw, 1.05rem)",
              fontWeight: 400, textTransform: "none", letterSpacing: "0",
              color: "var(--text-secondary)", marginBottom: "0.4rem",
              fontStyle: "italic",
            }}>{rec.intro}</span>
          )}
          {rec.title}
        </h2>
        {/* Anecdote body text — no scroll, fits the page */}
        <div style={{ minHeight: 0, overflowY: "auto" }}>
          <p style={{
            fontFamily: "var(--font-body)",
            fontSize: "clamp(0.82rem, 1.05vw, 1rem)",
            fontWeight: 500, lineHeight: 1.65,
            color: "var(--color-fun-ink)", margin: 0,
          }}>{body}</p>
        </div>
        {/* Page number */}
        <div style={{ display: "flex", justifyContent: "space-between", alignItems: "flex-end" }}>
          <span style={{
            fontFamily: "var(--font-body)", fontSize: "0.6rem",
            letterSpacing: "0.3em", textTransform: "uppercase",
            color: "var(--text-faint)", fontWeight: 700,
          }}>A Little Story</span>
          <span style={{
            fontFamily: "var(--font-display)", fontWeight: 900,
            fontSize: "1.6rem", letterSpacing: "-0.04em",
            lineHeight: 1, color: "var(--text-faint)",
          }}>{String((recIdx + 1) * 2 + 1).padStart(2, "0")}</span>
        </div>
      </div>
    );
  };

  const renderAnecdoteRight = (rec, recIdx, theme) => {
    const { question } = splitAnecdote(rec.description || "");
    return (
      <div style={{
        height: "100%",
        background: theme.soft,
        display: "grid", gridTemplateRows: "1fr auto auto",
        padding: "8% 8% 5%", gap: "1.5rem",
        position: "relative",
      }}>
        {/* Engaging question — the star of the right page */}
        <div style={{
          display: "flex", flexDirection: "column",
          justifyContent: "center", gap: "1.25rem",
        }}>
          <span style={{
            fontFamily: "var(--font-sketch)",
            fontSize: "clamp(0.75rem, 1vw, 0.9rem)",
            letterSpacing: "0.25em", textTransform: "uppercase",
            color: "rgba(0,0,0,0.4)",
          }}>And you?</span>
          {question && (
            <p style={{
              fontFamily: "var(--font-display)", fontWeight: 900,
              fontSize: "clamp(1.4rem, 2.6vw, 2.4rem)",
              lineHeight: 1.15, color: "var(--color-fun-ink)",
              margin: 0,
            }}>{question}</p>
          )}
        </div>
        {/* Why this helps — nostalgia mechanism */}
        <div className="ami-why-block ami-why-anecdote" style={{
          background: "var(--color-fun-white)", borderRadius: "1.25rem",
          padding: "1rem 1.25rem",
          border: "2px solid var(--color-fun-ink)",
          transform: "rotate(0.5deg)",
          boxShadow: "0 6px 14px rgba(0,0,0,0.08)",
        }}>
          <div style={{ display: "flex", alignItems: "center", gap: "0.5rem", marginBottom: "0.4rem", flexWrap: "wrap" }}>
            <span className="ami-why-label" style={{
              fontFamily: "var(--font-sketch)", fontSize: "0.75rem",
              fontWeight: 700, textTransform: "uppercase", letterSpacing: "0.2em",
              color: "rgba(0,0,0,0.45)",
            }}>Why this · for you</span>
            <MechanismPill mechanismId="nostalgia" color={undefined} />
          </div>
          <p style={{
            fontFamily: "var(--font-body)",
            fontSize: "clamp(0.8rem, 1vw, 0.95rem)",
            fontWeight: 700, lineHeight: 1.45,
            color: "#000", margin: 0,
          }}>{rec.artisticConnection}</p>
          {rec.paper && (
            <p style={{
              marginTop: "0.5rem", paddingTop: "0.4rem",
              borderTop: "1px dashed rgba(0,0,0,0.2)",
              fontFamily: "var(--font-body)", fontStyle: "italic",
              fontSize: "0.65rem", lineHeight: 1.35,
              color: "rgba(0,0,0,0.55)", margin: "0.5rem 0 0 0",
            }}>{rec.paper}</p>
          )}
        </div>
        {/* Page number */}
        <div style={{ display: "flex", justifyContent: "flex-end" }}>
          <span style={{
            fontFamily: "var(--font-display)", fontWeight: 900,
            fontSize: "1.6rem", letterSpacing: "-0.04em",
            lineHeight: 1, color: "rgba(0,0,0,0.15)",
          }}>{String((recIdx + 1) * 2 + 2).padStart(2, "0")}</span>
        </div>
      </div>
    );
  };

  const pages = [];
  const theme0 = THEMES[0];
  pages.push(renderOvertureLeft(theme0));
  pages.push(renderOvertureRight(theme0));
  recs.forEach((rec, recIdx) => {
    const t = THEMES[(recIdx + 1) % THEMES.length];
    if (rec.type === "anecdote") {
      pages.push(renderAnecdoteLeft(rec, recIdx, t));
      pages.push(renderAnecdoteRight(rec, recIdx, t));
    } else {
      pages.push(renderRecLeft(rec, recIdx, t));
      pages.push(renderRecRight(rec, recIdx, t));
    }
  });

  const handleFlip = (e) => {
    setSpread(Math.floor(e.data / 2));
    setDetailsOpenFor(null);
  };
  const goNext = () => bookRef.current && bookRef.current.pageFlip().flipNext();
  const goPrev = () => bookRef.current && bookRef.current.pageFlip().flipPrev();
  const goTo   = (i) => bookRef.current && bookRef.current.pageFlip().flip(i * 2);

  useEffect(() => {
    const onKey = (e) => {
      if (e.key === "ArrowRight" && spread < totalSpreads - 1) goNext();
      else if (e.key === "ArrowLeft" && spread > 0) goPrev();
    };
    window.addEventListener("keydown", onKey);
    return () => window.removeEventListener("keydown", onKey);
  }, [spread, totalSpreads]);

  return (
    <div style={{
      width: "100vw",
      marginLeft: "calc(50% - 50vw)",
      marginRight: "calc(50% - 50vw)",
      marginTop: "3rem",
      display: "flex", flexDirection: "column", alignItems: "center",
      gap: "1.25rem",
    }}>
      <div style={{
        position: "relative",
        width: bookSize.w, height: bookSize.h,
        filter: "drop-shadow(0 30px 50px rgba(0,0,0,0.22))",
      }}>
        {!flipBookAvailable ? (
          <div style={{
            width: "100%", height: "100%",
            display: "flex", flexDirection: "column",
            alignItems: "center", justifyContent: "center",
            gap: "1rem", textAlign: "center", padding: "2rem",
            background: "var(--color-fun-white)",
            border: "3px solid var(--color-fun-ink)",
            borderRadius: "1rem",
          }}>
            <Book size={48} style={{ color: "var(--text-faint)" }} />
            <p style={{
              fontFamily: "var(--font-display)", fontWeight: 900,
              fontSize: "1.1rem", color: "var(--color-fun-ink)", margin: 0,
            }}>The flipbook engine didn't load</p>
            <p style={{
              fontFamily: "var(--font-body)", fontSize: "0.85rem",
              color: "var(--text-muted)", margin: 0, maxWidth: "28rem",
            }}>react-pageflip couldn't be fetched from the CDN. Re-run this
            cell — esm.sh sometimes needs a moment to build a package on
            first request.</p>
          </div>
        ) : (
          <HTMLFlipBook
            ref={bookRef}
            key={"book-" + bookSize.pw + "x" + bookSize.ph + "-" + totalPages + "-" + mountTick}
            width={bookSize.pw}
            height={bookSize.ph}
            size="fixed"
            minWidth={280}
            maxWidth={2000}
            minHeight={360}
            maxHeight={2400}
            flippingTime={800}
            maxShadowOpacity={0.55}
            drawShadow={true}
            useMouseEvents={true}
            mobileScrollSupport={true}
            swipeDistance={20}
            usePortrait={false}
            showCover={false}
            disableFlipByClick={true}
            clickEventForward={true}
            showPageCorners={true}
            onFlip={handleFlip}
            className="ami-magazine-book"
          >
            {pages.map((content, i) => (
              <FlipPage
                key={"page-" + i}
                number={i}
                pageW={bookSize.pw}
                pageH={bookSize.ph}
              >
                {content}
              </FlipPage>
            ))}
          </HTMLFlipBook>
        )}

        {flipBookAvailable && spread > 0 && (
          <button
            type="button" onClick={goPrev} aria-label="Previous spread"
            style={navBtnStyle(false)}
          ><ChevronLeft size={22} /></button>
        )}
        {flipBookAvailable && spread < totalSpreads - 1 && (
          <button
            type="button" onClick={goNext} aria-label="Next spread"
            style={navBtnStyle(true)}
          ><ChevronRight size={22} /></button>
        )}
      </div>

      <div style={{
        display: "flex", alignItems: "center", gap: "0.5rem",
        padding: "0.45rem 0.9rem",
        background: "var(--border-soft)",
        borderRadius: "9999px",
      }}>
        {Array.from({ length: totalSpreads }).map((_, i) => {
          const t = THEMES[i % THEMES.length];
          const active = i === spread;
          return (
            <button
              key={i} type="button"
              onClick={() => goTo(i)}
              aria-label={i === 0 ? "Overture" : "Selection " + i}
              style={{
                width: active ? "2.5rem" : "0.7rem",
                height: "0.7rem", borderRadius: "9999px",
                border: "none", padding: 0, cursor: "pointer",
                background: active ? t.hex : "var(--text-faint)",
                transition: "width 0.4s ease, background 0.4s ease",
              }}
            />
          );
        })}
      </div>

      <div style={{
        display: "flex", flexWrap: "wrap", gap: "1rem",
        justifyContent: "center", alignItems: "center",
      }}>
        <span style={{
          fontFamily: "var(--font-sketch)",
          fontSize: "clamp(1.1rem, 1.6vw, 1.5rem)",
          color: accent.hex, textTransform: "lowercase", fontStyle: "italic",
        }}>{currentMood || "Ami"}</span>
        <span style={{ width: "2.5rem", height: "1px", background: "var(--text-faint)" }} />
        <span style={{
          fontSize: "0.6rem", fontWeight: 800,
          letterSpacing: "0.45em", textTransform: "uppercase",
          color: "var(--text-muted)",
        }}>
          {spread === 0 ? "Overture" : "Selection " + spread + " of " + (totalSpreads - 1)}
        </span>
        <span style={{ width: "2.5rem", height: "1px", background: "var(--text-faint)" }} />
        <motion.button
          onClick={onSurpriseMe} disabled={loading}
          animate={!loading ? { y: [0, -7, 0], rotate: [0, -1, 1, 0] } : {}}
          transition={{ duration: 5, repeat: Infinity, ease: "easeInOut" }}
          whileHover={{ scale: loading ? 1 : 1.08, rotate: loading ? 0 : -3, y: loading ? 0 : -12, transition: { duration: 0.2, type: "spring", stiffness: 400, damping: 10 } }}
          whileTap={{ scale: 0.95, rotate: (Math.random() - 0.5) * 15 }}
          style={{
            display: "inline-flex", alignItems: "center", gap: "0.5rem",
            padding: "0.75rem 1.6rem", borderRadius: "9999px",
            background: "var(--color-fun-yellow)", color: "var(--color-fun-ink)",
            fontFamily: "var(--font-display)", fontWeight: 900,
            fontSize: "0.85rem", letterSpacing: "0.02em",
            border: "2px solid var(--color-fun-ink)",
            cursor: loading ? "wait" : "pointer",
            opacity: loading ? 0.6 : 1,
            boxShadow: "0 6px 0 -1px var(--color-fun-ink)",
            transition: "opacity 0.2s",
          }}
        >
          {loading ? (
            <Loader2 size={14} style={{ animation: "spin 1s linear infinite" }} />
          ) : (
            <Sparkles size={14} strokeWidth={2.5} />
          )}
          <span>{loading ? "AMI IS THINKING…" : "SURPRISE ME"}</span>
        </motion.button>
        <motion.button
          onClick={onReturn}
          animate={{ y: [0, -5, 0], rotate: [0, 0.8, -0.8, 0] }}
          transition={{ duration: 6, repeat: Infinity, ease: "easeInOut", delay: 0.5 }}
          whileHover={{ scale: 1.08, y: -12, rotate: 3, transition: { duration: 0.2, type: "spring", stiffness: 400, damping: 10 } }}
          whileTap={{ scale: 0.95, rotate: (Math.random() - 0.5) * 15 }}
          style={{
            padding: "0.75rem 1.6rem", borderRadius: "9999px",
            background: "var(--color-fun-ink)", color: "var(--color-fun-white)",
            fontFamily: "var(--font-display)", fontWeight: 900,
            fontSize: "0.85rem", border: "none", cursor: "pointer",
            boxShadow: "0 8px 20px -5px rgba(0,0,0,0.25)",
          }}
        >DONE FOR TODAY</motion.button>
      </div>
    </div>
  );
}

function Row({ k, v, accent }) {
  return (
    <div style={{
      display: "flex", justifyContent: "space-between", gap: "0.5rem",
      padding: "0.2rem 0",
    }}>
      <span style={{ color: accent, opacity: 0.8, fontSize: "0.62rem", textTransform: "uppercase", letterSpacing: "0.18em", fontWeight: 800 }}>{k}</span>
      <span style={{ color: "var(--color-fun-ink)", textAlign: "right", maxWidth: "60%", fontSize: "0.72rem", lineHeight: 1.25 }}>{v}</span>
    </div>
  );
}

function navBtnStyle(rightSide) {
  return {
    position: "absolute",
    top: "50%",
    [rightSide ? "right" : "left"]: "-1.6rem",
    transform: "translateY(-50%)",
    width: "2.8rem", height: "2.8rem", borderRadius: "9999px",
    background: "var(--color-fun-white)",
    color: "var(--color-fun-ink)",
    border: "2px solid var(--color-fun-ink)",
    display: "flex", alignItems: "center", justifyContent: "center",
    cursor: "pointer", padding: 0,
    boxShadow: "0 6px 16px rgba(0,0,0,0.18)",
    zIndex: 30,
  };
}

// ─────────────────────────────────────────────────────────────────
// CrisisPanel
// ─────────────────────────────────────────────────────────────────
function CrisisPanel({ onContinue, onClose }) {
  const helplines = [
    { region: "International", line: "befrienders.org — find a helpline near you" },
    { region: "USA",    line: "988 — Suicide & Crisis Lifeline" },
    { region: "UK",     line: "116 123 — Samaritans" },
    { region: "France", line: "3114 — Numéro national de prévention du suicide" },
  ];

  return (
    <motion.div
      initial={{ opacity: 0, scale: 0.95 }}
      animate={{ opacity: 1, scale: 1 }}
      exit={{ opacity: 0 }}
      style={{
        position: "fixed", inset: 0, zIndex: 60,
        display: "flex",
        alignItems: "flex-start",
        justifyContent: "center",
        padding: "1.5rem",
        background: "rgba(0,0,0,0.5)", backdropFilter: "blur(4px)",
        overflowY: "auto",
        overscrollBehavior: "contain",
      }}
    >
      <motion.div
        initial={{ y: 40, rotate: -1 }}
        animate={{ y: 0, rotate: 0 }}
        className="sticker-shadow"
        style={{
          background: "var(--color-fun-white)",
          maxWidth: "42rem", width: "100%",
          padding: "3.5rem", borderRadius: "3rem",
          border: "4px solid var(--color-fun-ink)",
          position: "relative",
          margin: "auto",
          flexShrink: 0,
        }}
      >
        <button
          onClick={onClose}
          aria-label="Close"
          style={{
            position: "absolute", top: "1.5rem", right: "1.5rem",
            width: "2.5rem", height: "2.5rem", borderRadius: "9999px",
            border: "2px solid var(--color-fun-ink)",
            display: "flex", alignItems: "center", justifyContent: "center",
            background: "transparent", cursor: "pointer",
          }}
        >
          <X size={20} />
        </button>

        <motion.div
          animate={{ y: [0, -8, 0], rotate: [-2, 2, -2] }}
          transition={{ duration: 4, repeat: Infinity, ease: "easeInOut" }}
          style={{
            width: "5rem", height: "5rem",
            background: "var(--color-fun-pink)",
            borderRadius: "1.5rem",
            border: "4px solid var(--color-fun-ink)",
            display: "flex", alignItems: "center", justifyContent: "center",
            marginBottom: "2rem", transform: "rotate(-3deg)",
          }}
        >
          <Heart size={40} fill="currentColor" />
        </motion.div>

        <h2 className="font-display" style={{ fontSize: "clamp(2rem, 5vw, 3rem)", fontWeight: 900, marginBottom: "1.5rem", lineHeight: 1.1 }}>
          <FloatingWords text="Take a breath." intensity={0.6} />
          <br />
          <FloatingWords text="You are not alone." style={{ color: "var(--color-fun-pink)" }} intensity={0.8} />
        </h2>

        <p style={{ fontSize: "1.125rem", color: "var(--text-secondary)", marginBottom: "2rem", lineHeight: 1.6 }}>
          What you've shared sounds heavy. Before Ami hands you anything, please know that real people are ready to listen, right now.
        </p>

        <div style={{ display: "flex", flexDirection: "column", gap: "1rem", marginBottom: "2.5rem" }}>
          {helplines.map((h, i) => (
            <div key={i} style={{
              display: "flex", gap: "1rem", alignItems: "center",
              padding: "1rem", background: "rgba(255,191,0,0.3)",
              border: "2px solid var(--color-fun-ink)", borderRadius: "1rem",
            }}>
              <div style={{
                width: "2.5rem", height: "2.5rem",
                background: "var(--color-fun-yellow)",
                borderRadius: "9999px", border: "2px solid var(--color-fun-ink)",
                display: "flex", alignItems: "center", justifyContent: "center",
                flexShrink: 0,
              }}>
                <Phone size={18} />
              </div>
              <div>
                <span style={{ fontFamily: "var(--font-display)", fontWeight: 700, fontSize: "0.875rem", textTransform: "uppercase", letterSpacing: "0.05em", color: "var(--text-secondary)" }}>{h.region}</span>
                <p style={{ fontFamily: "var(--font-body)" }}>{h.line}</p>
              </div>
            </div>
          ))}
        </div>

        <div style={{ display: "flex", flexDirection: "column", gap: "0.75rem" }}>
          <button
            onClick={onClose}
            style={{
              width: "100%", padding: "1.25rem 0",
              background: "var(--color-fun-ink)", color: "var(--color-fun-white)",
              borderRadius: "9999px", border: "none", cursor: "pointer",
              fontFamily: "var(--font-display)", fontWeight: 900, fontSize: "1.125rem",
            }}
          >
            I'll reach out for help
          </button>
          <button
            onClick={onContinue}
            style={{
              width: "100%", padding: "0.75rem 0",
              color: "var(--text-secondary)", background: "transparent", border: "none",
              fontFamily: "var(--font-body)", fontSize: "0.875rem",
              textDecoration: "underline", cursor: "pointer",
            }}
          >
            Continue gently anyway
          </button>
        </div>
      </motion.div>
    </motion.div>
  );
}

// ─────────────────────────────────────────────────────────────────
// DetailsModal
// ─────────────────────────────────────────────────────────────────
function DetailsModal({ payload, onClose }) {
  if (!payload) return null;
  const { recommendation, detailEntries, typeIndex } = payload;

  const Icon = recommendation.type === "film" ? Video
    : recommendation.type === "series" ? Tv
    : Book;

  const url = externalUrlFor(recommendation);

  useEffect(() => {
    const prev = document.body.style.overflow;
    document.body.style.overflow = "hidden";
    const onKey = (e) => { if (e.key === "Escape") onClose(); };
    window.addEventListener("keydown", onKey);
    return () => {
      document.body.style.overflow = prev;
      window.removeEventListener("keydown", onKey);
    };
  }, [onClose]);

  return (
    <motion.div
      initial={{ opacity: 0 }}
      animate={{ opacity: 1 }}
      exit={{ opacity: 0 }}
      onClick={onClose}
      style={{
        position: "fixed", inset: 0, zIndex: 70,
        background: "rgba(0,0,0,0.6)",
        backdropFilter: "blur(8px)",
        display: "flex",
        alignItems: "flex-start",
        justifyContent: "center",
        padding: "2rem 1rem",
        overflowY: "auto",
        overscrollBehavior: "contain",
      }}
    >
      <motion.div
        initial={{ scale: 0.92, y: 30, opacity: 0 }}
        animate={{ scale: 1, y: 0, opacity: 1 }}
        exit={{ scale: 0.92, y: 30, opacity: 0 }}
        transition={{ type: "spring", stiffness: 260, damping: 26 }}
        onClick={(e) => e.stopPropagation()}
        className="sticker-shadow"
        style={{
          background: "var(--color-fun-white)",
          color: "var(--color-fun-ink)",
          maxWidth: "64rem", width: "100%",
          borderRadius: "3rem",
          border: "4px solid var(--color-fun-ink)",
          padding: "3rem clamp(1.5rem, 4vw, 3.5rem)",
          margin: "auto",
          flexShrink: 0,
          position: "relative",
        }}
      >
        <button
          onClick={onClose}
          aria-label="Close details"
          style={{
            position: "absolute", top: "1.5rem", right: "1.5rem",
            width: "3rem", height: "3rem", borderRadius: "9999px",
            border: "3px solid var(--color-fun-ink)",
            background: "var(--color-fun-white)",
            color: "var(--color-fun-ink)",
            display: "flex", alignItems: "center", justifyContent: "center",
            cursor: "pointer", zIndex: 2,
          }}
        >
          <X size={20} strokeWidth={3} />
        </button>

        <div className="ami-modal-grid" style={{
          display: "grid",
          gridTemplateColumns: payload.recommendation.poster ? "minmax(0, 18rem) minmax(0, 1fr)" : "1fr",
          gap: "2.5rem",
          alignItems: "start",
        }}>
          {payload.recommendation.poster && (
            <motion.div
              whileHover={{ scale: 1.03, rotate: 1, y: -4, transition: { type: "spring", stiffness: 300, damping: 18 } }}
              whileTap={{ scale: 0.97, rotate: -1 }}
              style={{
                position: "sticky",
                top: "1rem",
                width: "100%",
                borderRadius: "1.75rem",
                overflow: "hidden",
                border: "3px solid var(--color-fun-ink)",
                boxShadow: "0 0 0 3px var(--shadow-stroke), 6px 6px 0 0 var(--shadow-color)",
                transform: "rotate(-1deg)",
                background: "var(--color-fun-white)",
                cursor: "pointer",
              }}
            >
              <img
                src={upgradePosterQuality(payload.recommendation.poster)}
                alt={payload.recommendation.title}
                loading="lazy"
                style={{
                  display: "block",
                  width: "100%",
                  height: "auto",
                  maxHeight: "26rem",
                  objectFit: "cover",
                  objectPosition: "top center",
                }}
                onError={(e) => { e.currentTarget.parentElement.style.display = "none"; }}
              />
            </motion.div>
          )}

          <div>
            <div style={{ display: "flex", alignItems: "center", gap: "1rem", marginBottom: "2rem", paddingRight: "3rem" }}>
              <motion.div
                animate={{ y: [0, -6, 0], rotate: [3, -3, 3] }}
                transition={{ duration: 4, repeat: Infinity, ease: "easeInOut" }}
                style={{
                  width: "3.5rem", height: "3.5rem", borderRadius: "9999px",
                  border: "4px solid var(--color-fun-ink)",
                  background: "var(--color-fun-pink)",
                  display: "flex", alignItems: "center", justifyContent: "center",
                  flexShrink: 0,
                }}
              >
                <Icon size={28} />
              </motion.div>
              <span style={{ fontFamily: "var(--font-sketch)", fontSize: "1.5rem", color: "var(--text-secondary)" }}>
                {typeLabel(recommendation.type, typeIndex)}
              </span>
            </div>

            <h2
              className="storybook-title"
              style={{
                fontSize: "clamp(2.25rem, 6vw, 4.5rem)",
                lineHeight: 1.05,
                color: "white",
                marginBottom: "1.5rem",
                filter: "drop-shadow(0 10px 8px rgba(0,0,0,0.05))",
              }}
            >
              {recommendation.title}
            </h2>

            <p style={{
              fontSize: "1.25rem",
              fontFamily: "var(--font-body)",
              color: "var(--text-secondary)",
              fontStyle: "italic",
              lineHeight: 1.6,
              marginBottom: "2.5rem",
            }}>
              {recommendation.description}
            </p>

            <div style={{
              background: "var(--color-fun-pink)",
              padding: "2rem 2.5rem",
              border: "4px solid var(--color-fun-ink)",
              borderRadius: "2rem",
              marginBottom: "2rem",
              transform: "rotate(-0.4deg)",
            }}>
              <p style={{
                fontFamily: "var(--font-sketch)",
                fontSize: "1.5rem",
                marginBottom: "1rem",
                color: "rgba(0,0,0,0.5)",
                textDecoration: "underline",
              }}>
                ...the small print
              </p>
              <dl style={{
                display: "grid",
                gridTemplateColumns: "auto 1fr",
                rowGap: "0.75rem",
                columnGap: "1.5rem",
                fontFamily: "var(--font-body)",
                fontSize: "1.05rem",
                margin: 0,
                color: "#000",
              }}>
                {detailEntries.map(([label, value]) => (
                  <React.Fragment key={label}>
                    <dt style={{
                      fontFamily: "var(--font-display)",
                      fontWeight: 700,
                      textTransform: "uppercase",
                      letterSpacing: "0.05em",
                      fontSize: "0.85rem",
                      alignSelf: "center",
                      color: "rgba(0,0,0,0.7)",
                    }}>{label}</dt>
                    <dd style={{ margin: 0, fontWeight: 500 }}>{value}</dd>
                  </React.Fragment>
                ))}
              </dl>
            </div>

            <div style={{
              background: "var(--color-fun-yellow)",
              padding: "1.5rem 2rem",
              border: "4px solid var(--color-fun-ink)",
              borderRadius: "1.75rem",
              marginBottom: url ? "2rem" : 0,
              transform: "rotate(0.4deg)",
            }}>
              <p style={{
                fontFamily: "var(--font-sketch)",
                fontSize: "1.25rem",
                marginBottom: "0.5rem",
                color: "rgba(0,0,0,0.5)",
                textDecoration: "underline",
              }}>
                why it might help...
              </p>
              <p style={{
                fontSize: "1.125rem",
                fontFamily: "var(--font-body)",
                fontWeight: 600,
                color: "#000",
                lineHeight: 1.4,
              }}>
                {recommendation.artisticConnection}
              </p>
              {recommendation.paper && (
                <p style={{
                  marginTop: "0.875rem",
                  fontSize: "0.85rem",
                  fontFamily: "var(--font-body)",
                  fontStyle: "italic",
                  color: "rgba(0,0,0,0.65)",
                  lineHeight: 1.45,
                  borderTop: "1.5px dashed rgba(0,0,0,0.25)",
                  paddingTop: "0.625rem",
                }}>
                  {recommendation.paper}
                </p>
              )}
            </div>

            {url && (
              <motion.a
                href={url}
                target="_blank"
                rel="noopener noreferrer"
                whileHover={{ y: -4 }}
                whileTap={{ y: 0, scale: 0.98 }}
                className="sticker-shadow"
                style={{
                  display: "block",
                  width: "100%",
                  padding: "1.25rem 2rem",
                  background: "var(--color-fun-orange)",
                  color: "white",
                  border: "4px solid var(--color-fun-ink)",
                  borderRadius: "1.5rem",
                  fontFamily: "var(--font-display)",
                  fontWeight: 900,
                  fontSize: "1.5rem",
                  textAlign: "center",
                  textDecoration: "none",
                  cursor: "pointer",
                  transition: "transform 0.2s",
                }}
              >
                LET'S GO!
              </motion.a>
            )}
          </div>
        </div>
      </motion.div>
    </motion.div>
  );
}

// ─────────────────────────────────────────────────────────────────
// ThesisModal
// ─────────────────────────────────────────────────────────────────
function ThesisModal({ thesis, thematicChart, currentMood, onClose }) {
  useEffect(() => {
    const prev = document.body.style.overflow;
    document.body.style.overflow = "hidden";
    const onKey = (e) => { if (e.key === "Escape") onClose(); };
    window.addEventListener("keydown", onKey);
    return () => {
      document.body.style.overflow = prev;
      window.removeEventListener("keydown", onKey);
    };
  }, [onClose]);

  return (
    <motion.div
      initial={{ opacity: 0 }}
      animate={{ opacity: 1 }}
      exit={{ opacity: 0 }}
      onClick={onClose}
      style={{
        position: "fixed", inset: 0, zIndex: 70,
        background: "rgba(0,0,0,0.6)",
        backdropFilter: "blur(8px)",
        display: "flex",
        alignItems: "flex-start",
        justifyContent: "center",
        padding: "2rem 1rem",
        overflowY: "auto",
        overscrollBehavior: "contain",
      }}
    >
      <motion.div
        initial={{ scale: 0.92, y: 30, opacity: 0 }}
        animate={{ scale: 1, y: 0, opacity: 1 }}
        exit={{ scale: 0.92, y: 30, opacity: 0 }}
        transition={{ type: "spring", stiffness: 260, damping: 26 }}
        onClick={(e) => e.stopPropagation()}
        className="sticker-shadow"
        style={{
          background: "var(--color-fun-white)",
          color: "var(--color-fun-ink)",
          maxWidth: "56rem", width: "100%",
          borderRadius: "3rem",
          border: "4px solid var(--color-fun-ink)",
          padding: "clamp(2.5rem, 5vw, 4.5rem)",
          margin: "auto",
          flexShrink: 0,
          position: "relative",
        }}
      >
        <button
          onClick={onClose}
          aria-label="Close thesis"
          style={{
            position: "absolute", top: "1.5rem", right: "1.5rem",
            width: "3rem", height: "3rem", borderRadius: "9999px",
            border: "3px solid var(--color-fun-ink)",
            background: "var(--color-fun-white)",
            color: "var(--color-fun-ink)",
            display: "flex", alignItems: "center", justifyContent: "center",
            cursor: "pointer", zIndex: 2,
          }}
        >
          <X size={20} strokeWidth={3} />
        </button>

        <div style={{
          display: "flex", justifyContent: "space-between", alignItems: "flex-start",
          gap: "1.5rem", flexWrap: "wrap",
          marginBottom: "2.5rem", paddingRight: "3rem",
        }}>
          <motion.div
            animate={{ y: [0, -6, 0], rotate: [6, -6, 6] }}
            transition={{ duration: 4, repeat: Infinity, ease: "easeInOut" }}
            style={{
              width: "5rem", height: "5rem",
              background: "var(--color-fun-yellow)",
              borderRadius: "1.25rem",
              border: "3px solid var(--color-fun-ink)",
              display: "flex", alignItems: "center", justifyContent: "center",
              flexShrink: 0,
            }}
          >
            <Quote size={42} />
          </motion.div>
          <MoodShape mood={currentMood} />
        </div>

        <p style={{
          fontSize: "clamp(1.75rem, 3.6vw, 2.75rem)",
          fontFamily: "var(--font-display)",
          fontWeight: 900,
          lineHeight: 1.25,
          color: "var(--color-fun-ink)",
          margin: 0,
          marginBottom: "3rem",
        }}>
          {thesis}
        </p>

        <div style={{
          height: "280px",
          background: "rgba(162, 255, 0, 0.18)",
          borderRadius: "1.5rem",
          padding: "1.25rem",
          border: "2px solid var(--border-soft)",
          marginBottom: "2.5rem",
        }}>
          <ResponsiveContainer width="100%" height="100%">
            <AreaChart data={thematicChart}>
              <Area type="stepAfter" dataKey="value" stroke="#101112" strokeWidth={5} fill="#FFC0CB" fillOpacity={0.8} />
            </AreaChart>
          </ResponsiveContainer>
        </div>

        <p style={{
          fontFamily: "var(--font-sketch)",
          fontSize: "1.25rem",
          color: "var(--text-muted)",
          textAlign: "center",
          margin: 0,
        }}>
          your emotional landscape, woven into research
        </p>
      </motion.div>
    </motion.div>
  );
}

// ─────────────────────────────────────────────────────────────────
//
// ─────────────────────────────────────────────────────────────────
async function fetchRecommendations({
  freeText, selectedMoods, birthYear,
  continueAnyway = false, nonce = null, imageB64 = null,
  onProgress = null,
}) {
  const response = await fetch("/api/recommend", {
    method: "POST",
    headers: {
      "Content-Type": "application/json",
      "Accept":       "text/event-stream",
    },
    body: JSON.stringify({
      free_text:       freeText,
      selected_moods:  (selectedMoods || []).map((m) => m.toLowerCase()),
      birth_year:      parseInt(birthYear, 10),
      lang:            "en",
      continue_anyway: continueAnyway,
      agentic:         false,
      nonce:           nonce,
      image_b64:       imageB64,
    }),
  });

  if (!response.ok || !response.body) {
    throw new Error(`API ${response.status}: ${await response.text()}`);
  }

  const reader  = response.body.getReader();
  const decoder = new TextDecoder();
  let buffer    = "";
  const collectedItems = [];
  let crisis = false;
  let crisisPayload = null;
  let donePayload = null;
  let serverError = null;

  const dispatch = (eventName, dataObj) => {
    if (eventName === "status" && onProgress) {
      onProgress({ step: dataObj.step, msg: dataObj.msg, elapsed: dataObj.elapsed });
    } else if (eventName === "item") {
      collectedItems.push(dataObj.item);
      if (onProgress) onProgress({ step: "item", msg: `Found ${collectedItems.length} works`, item: dataObj.item });
    } else if (eventName === "done") {
      donePayload = dataObj;
    } else if (eventName === "crisis") {
      crisis = true;
      crisisPayload = dataObj;
    } else if (eventName === "vision_insight") {
      if (onProgress) onProgress({ step: "vision_insight", insight: dataObj });
    } else if (eventName === "error") {
      serverError = dataObj.error || "unknown error";
    }
  };

  const processBuffer = () => {
    let idx;
    while ((idx = buffer.indexOf("\n\n")) !== -1 || (idx = buffer.indexOf("\r\n\r\n")) !== -1) {
      const sepLen = buffer.slice(idx, idx + 4) === "\r\n\r\n" ? 4 : 2;
      const frame = buffer.slice(0, idx);
      buffer = buffer.slice(idx + sepLen);
      let eventName = "message";
      let dataLines = [];
      for (const line of frame.split(/\r?\n/)) {
        if (line.startsWith("event:")) eventName = line.slice(6).trim();
        else if (line.startsWith("data:")) dataLines.push(line.slice(5).trim());
      }
      if (dataLines.length === 0) continue;
      try {
        const data = JSON.parse(dataLines.join("\n"));
        dispatch(eventName, data);
      } catch (e) {
        console.warn("SSE frame parse failed:", e, frame);
      }
    }
  };

  while (true) {
    const { value, done } = await reader.read();
    if (done) break;
    buffer += decoder.decode(value, { stream: true });
    processBuffer();
  }
  buffer += decoder.decode();
  processBuffer();

  if (serverError) throw new Error(serverError);
  if (crisis) {
    return {
      crisis:          true,
      thesis:          "",
      recommendations: [],
      thematicChart:   [],
      psychProfile:    null,
      safety:          (crisisPayload && crisisPayload.safety) || { level: "crisis" },
      timings:         null,
      warning:         null,
    };
  }
  if (!donePayload) {
    throw new Error("Stream ended without 'done' event");
  }

  const recommendations = collectedItems.map((it) => {
    const creator =
      it.director ||
      (Array.isArray(it.creators) ? it.creators.join(", ") : it.creators) ||
      it.author ||
      "";
    const description = it.description || it.overview || it.anecdote || "";
    const artisticConnection = it.why || "";
    const personalIntro = it.personal_intro || it.personal_why || "";
    const details = {};
    if (it.runtime)  details.runtime  = `${it.runtime} min`;
    if (it.seasons)  details.seasons  = it.seasons;
    if (it.episodes) details.episodes = it.episodes;
    if (it.pages)    details.pages    = it.pages;
    if (it.rating)   details.rating   = `${it.rating}/10`;
    if (Array.isArray(it.genres) && it.genres.length)   details.genres = it.genres;
    if (Array.isArray(it.cast) && it.cast.length)       details.cast = it.cast.slice(0, 4);
    if (Array.isArray(it.networks) && it.networks.length) details.networks = it.networks;
    if (Array.isArray(it.subjects) && it.subjects.length) details.subjects = it.subjects.slice(0, 4);
    if (it.section)          details.section = it.section;
    if (it.birth_year_range) details.era = it.birth_year_range;
    if (it.decade)           details.decade = it.decade;
    if (it.mechanism)        details.mechanism = it.mechanism;
    if (Array.isArray(it.moods) && it.moods.length) details.moods = it.moods;
    return {
      id:                 it.id,
      type:               it.type,
      title:              it.title,
      year:               it.year,
      poster:             it.poster || null,
      tmdbId:             it.tmdb_id || null,
      olKey:              it.ol_key || null,
      creator,
      description,
      artisticConnection,
      intro:              personalIntro,
      paper:              it.paper || "",
      mechanism:          it.mechanism || null,
      details,
    };
  });

  return {
    crisis:          false,
    thesis:          donePayload.thesis || "",
    recommendations,
    thematicChart: (donePayload.thematic_data || []).map((t) => ({ name: t.name, value: t.value })),
    psychProfile:  donePayload.psych_profile || null,
    safety:        donePayload.safety || null,
    timings:       donePayload.timings || null,
    warning:       donePayload.warning || null,
  };
}

// ─────────────────────────────────────────────────────────────────
// Client-side crisis detection
// ─────────────────────────────────────────────────────────────────
const CRISIS_PATTERNS = [
  /\b(kill|hurt|harm)(ing|s|ed)?\s+(my\s*self|me)\b/i,
  /\b(thinking|think|thought|want|wanna|going|plan)\s+(of|about|to)\s+(kill|hurt|harm)/i,
  /\bsuicid(e|al|ality)\b/i,
  /\bend\s+(it\s+all|my\s+life|everything|things|it)\b/i,
  /\b(want|wanna|going|wish|hope)\s+(to\s+)?(die|be\s+dead|not\s+exist)\b/i,
  /\bi\s+(want|wanna|wish)\s+(to\s+)?die\b/i,
  /\bno\s+reason\s+to\s+live\b/i,
  /\bcan'?t\s+(go\s+on|take\s+it|do\s+this|live)\b/i,
  /\bdon'?t\s+want\s+to\s+(live|be\s+here|exist|wake\s+up)\b/i,
  /\b(self[-\s]?harm|cutting\s+myself|hurting\s+myself)\b/i,
  /\b(overdose|hang\s+myself|jump\s+off)\b/i,
  /\bbetter\s+off\s+(without\s+me|dead|gone)\b/i,
];
function detectCrisis(text) {
  const t = String(text || "").toLowerCase();
  return CRISIS_PATTERNS.some((re) => re.test(t));
}

// ─────────────────────────────────────────────────────────────────
// Helpers
// ─────────────────────────────────────────────────────────────────
function moodMaskIndex(mood) {
  const m = (mood || "").toLowerCase();
  if (/peace|calm|nostalg|hope|grateful|joy|smil|happy/.test(m))     return 5;
  if (/angry|cynic|exhaust|frustrat|tired/.test(m))                  return 4;
  if (/anxious|lost|overwhelm|scared|panic|worried/.test(m))         return 2;
  if (/sad|empty|discour|grief|grieving|broken|hopeless/.test(m))    return 3;
  return 1;
}

function upgradePosterQuality(url) {
  if (!url) return url;
  if (url.includes("image.tmdb.org/t/p/")) {
    return url.replace(/\/t\/p\/w\d+\//, "/t/p/w780/");
  }
  if (url.includes("covers.openlibrary.org")) {
    return url.replace(/-[SML]\.jpg$/, "-L.jpg");
  }
  return url;
}

function externalUrlFor(rec) {
  if (rec.tmdbId) {
    if (rec.type === "film")   return `https://www.themoviedb.org/movie/${rec.tmdbId}`;
    if (rec.type === "series") return `https://www.themoviedb.org/tv/${rec.tmdbId}`;
  }
  if (rec.olKey && rec.type === "novel") {
    return `https://openlibrary.org${rec.olKey}`;
  }
  const q = encodeURIComponent(`${rec.title || ""} ${rec.year && rec.year !== "Memory" ? rec.year : ""}`.trim());
  if (rec.type === "film")    return `https://www.themoviedb.org/search/movie?query=${q}`;
  if (rec.type === "series")  return `https://www.themoviedb.org/search/tv?query=${q}`;
  if (rec.type === "novel")   return `https://openlibrary.org/search?q=${q}`;
  return null;
}

function typeLabel(type, index) {
  if (type === "anecdote") return "A Little Story";
  if (type === "film")     return `Film #${index + 1}`;
  if (type === "series")   return `Series #${index + 1}`;
  if (type === "novel")    return `Novel #${index + 1}`;
  return `Discovery #${index + 1}`;
}

// ─────────────────────────────────────────────────────────────────
// Mechanism data
// ─────────────────────────────────────────────────────────────────
const MECHS = [
  {
    id: "nostalgia",
    label: "Nostalgia",
    freq: "88.1",
    color: "#FFBF00",
    desc: "A bittersweet, self-relevant social emotion triggered by meaningful past experiences. It increases perceived self-continuity, social connectedness, and sense of meaning, counteracting loneliness, emptiness, and meaninglessness.",
    paper: "Sedikides & Wildschut, 2018 — Finding Meaning in Nostalgia. Review of General Psychology.",
  },
  {
    id: "transport",
    label: "Narrative Transport",
    freq: "89.4",
    color: "#38BDF8",
    desc: "Complete absorption in a narrative world reduces self-consciousness, mental chatter, and rumination. The more fully transported, the stronger the emotional change, even toward dark stories. Best for overthinking and restlessness.",
    paper: "Green & Brock, 2000 — The Role of Transportation in the Persuasiveness of Public Narratives. Journal of Personality and Social Psychology.",
  },
  {
    id: "awe",
    label: "Awe",
    freq: "90.7",
    color: "#A2FF00",
    desc: "Exposure to something vast or beautiful that challenges current mental frameworks shrinks the perceived size of the self and its problems, expands perceived time, and increases curiosity. Very effective for anxiety.",
    paper: "Stellar et al., 2015 — Self-Transcendent Emotions and Their Social Functions. Emotion.",
  },
  {
    id: "elevation",
    label: "Elevation",
    freq: "92.3",
    color: "#D1FAE5",
    desc: "Witnessing extraordinary moral beauty, virtue, or human excellence (even in fiction) triggers a warm, uplifting sensation and a strong motivation to be a better person. Reduces cynicism and discouragement powerfully.",
    paper: "Algoe & Haidt, 2009 — Witnessing Excellence in Action. Journal of Positive Psychology.",
  },
  {
    id: "benign_masochism",
    label: "Benign Masochism",
    freq: "93.6",
    color: "#E8B4AA",
    desc: "Intense negative emotions (sadness, fear, tension, disgust) through fiction are paradoxically pleasurable because the brain registers the safety of the fictional frame. The emotional arousal is real but the threat is not, this gap creates catharsis.",
    paper: "Hanich et al., 2014 — Why We Like to Watch Sad Films. Psychology of Aesthetics, Creativity, and the Arts.",
  },
  {
    id: "parasocial",
    label: "Parasocial Bond",
    freq: "95.1",
    color: "#E6E6FA",
    desc: "One-sided emotional bonds with fictional characters activate the same neural systems as real friendships: they satisfy belonging needs, create a felt sense of being known, and reduce loneliness. The bond strengthens with repeated exposure.",
    paper: "Giles, 2002 — Parasocial Interaction: A Review of the Literature. Media Psychology.",
  },
  {
    id: "social_surrogate",
    label: "Social Surrogate",
    freq: "96.8",
    color: "#FED7AA",
    desc: "Familiar shows and beloved books function as social surrogates: their recurring characters and settings provide a felt sense of belonging and comfort. Revisiting a known show can be more effective than watching something new, familiarity is the therapeutic agent.",
    paper: "Derrick, Gabriel & Hugenberg, 2009 — Social Surrogacy. Journal of Experimental Social Psychology.",
  },
  {
    id: "self_expansion",
    label: "Self-Expansion",
    freq: "98.2",
    color: "#DBEAFE",
    desc: "Engaging with perspectives or worlds radically different from one's own expands the psychological self and generates a felt sense of growth and possibility. Most powerful for boredom, emptiness, feeling stuck, or loss of meaning.",
    paper: "Aron et al., 1992 — Inclusion of Other in the Self Scale. Journal of Personality and Social Psychology.",
  },
  {
    id: "prosocial",
    label: "Prosocial Narrative",
    freq: "99.9",
    color: "#FEF9C3",
    desc: "Stories centered on kindness, cooperation, solidarity, or altruism activate reward circuits and reduce ruminative thinking. Most effective for cynicism, discouragement, guilt; states where someone has lost faith in human goodness.",
    paper: "Raposa et al., 2016 — Prosocial Behavior Mitigates the Negative Effects of Stress. Clinical Psychological Science.",
  },
  {
    id: "involuntary_memory",
    label: "Involuntary Memory",
    freq: "101.3",
    color: "#EDE9FE",
    desc: "Specific sensory details (a melody, a visual palette, an ambient sound) bypass conscious recall and trigger vivid, emotionally rich autobiographical memories. The Proustian mechanism. Most effective for nostalgia, melancholy, and grief.",
    paper: "Chu & Downes, 2000 — Odour-Evoked Autobiographical Memories. Chemical Senses.",
  },
  {
    id: "incongruence",
    label: "Emotional Incongruence",
    freq: "103.7",
    color: "#FFE4E6",
    desc: "Exposure to a mood slightly different from one's current emotional state is often more therapeutic than mood-matching. The slight shift creates distance from the stuck state, it should feel like a gentle pivot, not an escape.",
    paper: "Zillmann, 1988 — Mood Management Through Communication Choices. American Behavioral Scientist.",
  },
];

// ─────────────────────────────────────────────────────────────────
// TunerDial
// ─────────────────────────────────────────────────────────────────
// ─────────────────────────────────────────────────────────────────
// MechanismPill — clickable mechanism badge with inline definition tooltip
// ─────────────────────────────────────────────────────────────────
function MechanismPill({ mechanismId, color }) {
  const [open, setOpen] = React.useState(false);
  const mech = MECHS.find((m) => m.id === mechanismId);
  if (!mech) return null;
  return (
    <div style={{ position: "relative", display: "inline-block" }}>
      <motion.button
        type="button"
        onClick={(e) => { e.stopPropagation(); setOpen((o) => !o); }}
        onPointerDown={(e) => e.stopPropagation()}
        onMouseDown={(e) => e.stopPropagation()}
        whileHover={{ scale: 1.06 }}
        whileTap={{ scale: 0.95 }}
        className="ami-mech-tag"
        style={{
          display: "inline-flex", alignItems: "center", gap: "0.3em",
          background: color || mech.color,
          border: "1.5px solid var(--color-fun-ink)",
          borderRadius: "9999px",
          padding: "0.2em 0.75em",
          fontFamily: "var(--font-body)",
          fontWeight: 800,
          fontSize: "0.7rem",
          letterSpacing: "0.08em",
          textTransform: "uppercase",
          color: "#000",
          cursor: "pointer",
          boxShadow: open ? "0 0 0 2px var(--color-fun-ink)" : "none",
          transition: "box-shadow 0.15s",
        }}
      >
        {mech.label}
        <span style={{ fontSize: "0.65em", opacity: 0.7 }}>{open ? "▲" : "▼"}</span>
      </motion.button>
      <AnimatePresence>
        {open && (
          <motion.div
            initial={{ opacity: 0, y: -6, scale: 0.95 }}
            animate={{ opacity: 1, y: 0, scale: 1 }}
            exit={{ opacity: 0, y: -6, scale: 0.95 }}
            transition={{ duration: 0.18 }}
            onPointerDown={(e) => e.stopPropagation()}
            onMouseDown={(e) => e.stopPropagation()}
            style={{
              position: "absolute", bottom: "calc(100% + 0.5rem)", left: "50%",
              transform: "translateX(-50%)",
              width: "clamp(200px, 28vw, 300px)",
              background: "var(--color-fun-white)",
              border: "2px solid var(--color-fun-ink)",
              borderRadius: "1rem",
              padding: "0.9rem 1rem",
              boxShadow: "0 8px 24px rgba(0,0,0,0.18)",
              zIndex: 50,
            }}
          >
            {/* Triangle pointer */}
            <div style={{
              position: "absolute", bottom: "-0.55rem", left: "50%",
              transform: "translateX(-50%)",
              width: 0, height: 0,
              borderLeft: "7px solid transparent",
              borderRight: "7px solid transparent",
              borderTop: "8px solid var(--color-fun-ink)",
            }} />
            <div style={{
              fontFamily: "var(--font-sketch)",
              fontSize: "0.68rem",
              letterSpacing: "0.15em",
              textTransform: "uppercase",
              color: "var(--text-muted)",
              marginBottom: "0.4rem",
            }}>{mech.label}</div>
            <p style={{
              fontFamily: "var(--font-body)",
              fontSize: "0.78rem",
              lineHeight: 1.55,
              color: "var(--color-fun-ink)",
              margin: "0 0 0.5rem",
            }}>{mech.desc}</p>
          </motion.div>
        )}
      </AnimatePresence>
    </div>
  );
}

function TunerDial({ mechs, dark }) {
  const [current, setCurrent] = React.useState(2);
  const [transitioning, setTransitioning] = React.useState(false);
  const [cardKey, setCardKey] = React.useState(0);

  const TOTAL = mechs.length;

  function getNeedleLeft(idx) {
    const w = 100 / TOTAL;
    return (idx * w + w / 2) + "%";
  }

  function getDialAngle(idx) {
    return -90 + (idx / (TOTAL - 1)) * 180;
  }

  function getHandCoords(idx) {
    const angle = getDialAngle(idx);
    const rad = (angle * Math.PI) / 180;
    const cx = 100, cy = 95, r = 70;
    return { x2: cx + r * Math.sin(rad), y2: cy - r * Math.cos(rad) };
  }

  function getArcPath(idx) {
    const cx = 100, cy = 95, r2 = 78;
    const startRad = (-90 * Math.PI) / 180;
    const endAngle = getDialAngle(idx);
    const endRad = (endAngle * Math.PI) / 180;
    const sx = cx + r2 * Math.sin(startRad);
    const sy = cy - r2 * Math.cos(startRad);
    const ex = cx + r2 * Math.sin(endRad);
    const ey = cy - r2 * Math.cos(endRad);
    const large = endAngle + 90 > 180 ? 1 : 0;
    return `M${sx} ${sy} A${r2} ${r2} 0 ${large} 1 ${ex} ${ey}`;
  }

  function tune(idx) {
    if (idx === current || transitioning) return;
    setTransitioning(true);
    setTimeout(() => {
      setCurrent(idx);
      setCardKey(k => k + 1);
      setTransitioning(false);
    }, 220);
  }

  const m = mechs[current];
  const hand = getHandCoords(current);
  const inkColor = dark ? "#cbd5e1" : "#000000";
  const bgColor  = dark ? "#0f172a" : "#ffffff";
  const mutedColor = dark ? "rgba(203,213,225,0.4)" : "rgba(0,0,0,0.25)";
  const trackBg   = dark ? "rgba(203,213,225,0.07)" : "rgba(0,0,0,0.04)";
  const signalBars = [8, 13, 17, 13, 8];
  const signalStrength = Math.round(3 + Math.abs(Math.sin(current * 1.7)) * 2);

  return (
    <motion.div
      initial={{ opacity: 0, y: 20 }}
      whileInView={{ opacity: 1, y: 0 }}
      viewport={{ once: true }}
      transition={{ duration: 0.5, delay: 0.1 }}
      style={{
        background: "var(--color-fun-white)",
        border: "2px solid var(--color-fun-ink)",
        borderRadius: "2rem",
        padding: "2rem 1.75rem",
        fontFamily: "var(--font-body)",
      }}
      className="sticker-shadow"
    >
      <div style={{ position: "relative", marginBottom: 0 }}>
        <div style={{
          display: "flex", height: "52px",
          background: trackBg,
          borderRadius: "0.75rem",
          border: "2px solid var(--color-fun-ink)",
          overflow: "hidden",
        }}>
          {mechs.map((mech, i) => (
            <motion.div
              key={mech.id}
              onClick={() => tune(i)}
              whileHover={{ backgroundColor: mech.color + "55" }}
              style={{
                flex: 1,
                display: "flex", flexDirection: "column",
                alignItems: "center", justifyContent: "center",
                cursor: "pointer", gap: "3px",
                borderRight: i < TOTAL - 1 ? "1px solid var(--color-fun-ink)" : "none",
                background: i === current ? mech.color + "44" : "transparent",
                transition: "background 0.2s ease",
              }}
            >
              <motion.div
                animate={{ scale: i === current ? 1.4 : 1 }}
                transition={{ type: "spring", stiffness: 400, damping: 20 }}
                style={{
                  width: i === current ? "8px" : "5px",
                  height: i === current ? "8px" : "5px",
                  borderRadius: "50%",
                  background: i === current ? m.color : mutedColor,
                  transition: "all 0.2s",
                }}
              />
              <span style={{
                fontSize: "9px",
                color: i === current ? "var(--color-fun-ink)" : mutedColor,
                fontWeight: i === current ? 700 : 400,
                letterSpacing: "0.02em",
                lineHeight: 1,
              }}>{mech.freq}</span>
            </motion.div>
          ))}
        </div>

        <motion.div
          animate={{ left: getNeedleLeft(current) }}
          transition={{ type: "spring", stiffness: 260, damping: 28 }}
          style={{
            position: "absolute", top: "-10px",
            width: "2px", background: "var(--color-fun-ink)",
            height: "72px", transform: "translateX(-50%)",
            borderRadius: "1px", pointerEvents: "none",
          }}
        >
          <div style={{
            position: "absolute", top: "-6px", left: "50%", transform: "translateX(-50%)",
            width: 0, height: 0,
            borderLeft: "5px solid transparent",
            borderRight: "5px solid transparent",
            borderBottom: `8px solid var(--color-fun-ink)`,
          }} />
        </motion.div>
      </div>

      <div style={{ display: "flex", alignItems: "flex-end", justifyContent: "center", gap: "2rem", margin: "1.5rem 0 1.25rem" }}>
        <svg width="180" height="90" viewBox="0 0 200 100" fill="none" xmlns="http://www.w3.org/2000/svg" aria-hidden="true">
          <path d="M12 95 A88 88 0 0 1 188 95" stroke={mutedColor} strokeWidth="1" fill="none"/>
          <path d="M22 95 A78 78 0 0 1 178 95" stroke={mutedColor} strokeWidth="3.5" fill="none" strokeLinecap="round"/>
          <motion.path
            animate={{ d: getArcPath(current) }}
            transition={{ type: "spring", stiffness: 220, damping: 26 }}
            stroke={m.color} strokeWidth="3.5" fill="none" strokeLinecap="round"
          />
          <circle cx="100" cy="95" r="13" fill={bgColor} stroke={inkColor} strokeWidth="1.5"/>
          <motion.line
            animate={{ x2: hand.x2, y2: hand.y2 }}
            transition={{ type: "spring", stiffness: 220, damping: 26 }}
            x1="100" y1="95" stroke={inkColor} strokeWidth="2.5" strokeLinecap="round"
          />
          <circle cx="100" cy="95" r="4" fill={inkColor}/>
          <text x="8" y="108" fontSize="8" fill={mutedColor} fontFamily="var(--font-body)">{mechs[0].freq}</text>
          <text x="158" y="108" fontSize="8" fill={mutedColor} fontFamily="var(--font-body)">{mechs[TOTAL-1].freq}</text>
        </svg>

        <div style={{ display: "flex", flexDirection: "column", alignItems: "center", gap: "6px", paddingBottom: "8px" }}>
          <div style={{ display: "flex", alignItems: "flex-end", gap: "3px", height: "22px" }}>
            {signalBars.map((h, bi) => (
              <motion.div
                key={bi}
                animate={{ height: `${h}px`, background: bi < signalStrength ? m.color : mutedColor }}
                transition={{ duration: 0.3 }}
                style={{ width: "5px", borderRadius: "2px" }}
              />
            ))}
          </div>
          <span style={{ fontSize: "11px", color: mutedColor, letterSpacing: "0.06em", fontFamily: "var(--font-body)" }}>
            {m.freq} FM
          </span>
        </div>
      </div>

      <AnimatePresence mode="wait">
        <motion.div
          key={cardKey}
          initial={{ opacity: 0, y: 10, filter: "blur(4px)" }}
          animate={{ opacity: transitioning ? 0.3 : 1, y: 0, filter: transitioning ? "blur(3px)" : "blur(0px)" }}
          exit={{ opacity: 0, y: -8, filter: "blur(4px)" }}
          transition={{ duration: 0.25 }}
          style={{
            borderTop: `3px solid ${m.color}`,
            paddingTop: "1.25rem",
          }}
        >
          <div style={{ display: "flex", alignItems: "baseline", gap: "0.75rem", marginBottom: "0.6rem", flexWrap: "wrap" }}>
            <FloatingWords
              text={m.label}
              style={{ fontFamily: "var(--font-display)", fontWeight: 900, fontSize: "clamp(1.3rem, 3vw, 1.7rem)", letterSpacing: "-0.01em" }}
              intensity={0.6}
            />
            <motion.span
              animate={{ backgroundColor: [m.color + "88", m.color + "dd", m.color + "88"] }}
              transition={{ duration: 3, repeat: Infinity }}
              className="ami-mech-tag"
              style={{
                fontSize: "0.65rem", fontWeight: 700, letterSpacing: "0.1em",
                color: "#000", borderRadius: "9999px",
                padding: "0.2rem 0.7rem", textTransform: "uppercase",
                fontFamily: "var(--font-body)", border: "1.5px solid var(--color-fun-ink)",
              }}
            >
              {m.label}
            </motion.span>
          </div>
          <FloatingWords
            text={m.desc}
            style={{ fontFamily: "var(--font-body)", fontSize: "clamp(0.95rem, 1.8vw, 1.05rem)", color: "var(--text-secondary)", lineHeight: 1.7, display: "block", marginBottom: "1rem" }}
            intensity={0.2}
          />
          <p style={{ fontSize: "0.75rem", fontFamily: "var(--font-body)", fontStyle: "italic", color: "var(--text-muted)", lineHeight: 1.5, borderTop: "1px solid var(--border-soft)", paddingTop: "0.75rem" }}>
            {m.paper}
          </p>
        </motion.div>
      </AnimatePresence>
    </motion.div>
  );
}


// ─────────────────────────────────────────────────────────────────
// MoodMaskGallery — 5 hand-drawn masks revealed on scroll,
// ordered so the smiling mask (5) is always last.
// Each mask reacts to hover (tilt + lift) and can be dragged.
// ─────────────────────────────────────────────────────────────────
const MASK_ORDER = [1, 2, 3, 4, 5]; // mask 5 = serene/smiling, always last

function MoodMaskGallery() {
  const rotations = [-6, 4, -3, 5, -2];
  const delays    = [0, 0.12, 0.24, 0.36, 0.48];
  const yOffsets  = [0, 20, -10, 15, -5];

  return (
    <div style={{
      width: "100%",
      maxWidth: "900px",
      margin: "0 auto",
      padding: "3rem 1.5rem 1rem",
      /* No overflow — all masks fully visible, no scrollbar */
      display: "grid",
      gridTemplateColumns: "repeat(5, 1fr)",
      alignItems: "flex-end",
      gap: "clamp(0.5rem, 1.5vw, 1.25rem)",
    }}>
      {MASK_ORDER.map((maskIdx, i) => (
        <motion.div
          key={maskIdx}
          initial={{ opacity: 0, y: 50, rotate: rotations[i] * 0.5 }}
          whileInView={{ opacity: 1, y: yOffsets[i], rotate: rotations[i] }}
          viewport={{ once: true, margin: "-60px" }}
          transition={{
            duration: 0.65,
            delay: delays[i],
            type: "spring",
            stiffness: 90,
            damping: 14,
          }}
          drag
          dragConstraints={{ left: 0, right: 0, top: 0, bottom: 0 }}
          dragElastic={0.15}
          whileHover={{
            y: yOffsets[i] - 16,
            rotate: rotations[i] * -0.5,
            scale: 1.06,
            zIndex: 10,
            transition: { type: "spring", stiffness: 300, damping: 18 },
          }}
          whileTap={{ scale: 0.95 }}
          style={{
            cursor: "grab",
            width: "100%",
            filter: "drop-shadow(0 10px 20px rgba(0,0,0,0.15))",
            position: "relative",
            zIndex: i,
          }}
        >
          <img
            src={`/static/mood-${maskIdx}.webp`}
            alt={`mood mask ${maskIdx}`}
            style={{
              width: "100%",
              height: "auto",
              display: "block",
              pointerEvents: "none",
              userSelect: "none",
            }}
            draggable={false}
          />
        </motion.div>
      ))}
    </div>
  );
}

function Ami() {
  const [loading, setLoading] = useState(false);
  const [progressMsg, setProgressMsg] = useState("");
  const [visionInsight, setVisionInsight] = useState(null);
  const [result, setResult] = useState(null);
  const [error, setError] = useState(null);
  const [scrolled, setScrolled] = useState(false);
  const [isResearching, setIsResearching] = useState(false);
  const [showAllMechanisms, setShowAllMechanisms] = useState(false);
  const [currentMood, setCurrentMood] = useState("");
  const [badgeVisible, setBadgeVisible] = useState(true);
  const [crisisOpen, setCrisisOpen] = useState(false);
  const [pendingContinue, setPendingContinue] = useState(null);
  const [detailsPayload, setDetailsPayload] = useState(null);
  const [thesisOpen, setThesisOpen] = useState(false);
  const [lastPayload, setLastPayload] = useState(null);
  const resultsRef = React.useRef(null);
  const visionInsightRef = React.useRef(null);

  useEffect(() => {
    if (!result) return;
    const t1 = requestAnimationFrame(() => {
      const t2 = setTimeout(() => {
        if (resultsRef.current) {
          resultsRef.current.scrollIntoView({ behavior: "smooth", block: "start" });
        }
      }, 150);
      resultsRef._t = t2;
    });
    return () => {
      cancelAnimationFrame(t1);
      if (resultsRef._t) clearTimeout(resultsRef._t);
    };
  }, [result]);

  // Scroll to First Reading card when it appears
  useEffect(() => {
    if (!visionInsight) return;
    const t = setTimeout(() => {
      if (visionInsightRef.current) {
        visionInsightRef.current.scrollIntoView({ behavior: "smooth", block: "center" });
      }
    }, 300);
    return () => clearTimeout(t);
  }, [visionInsight]);

  const initialDark = (() => {
    if (typeof window === "undefined") return false;
    const h = new Date().getHours();
    return h >= 19 || h < 7;
  })();
  const [darkMode, setDarkMode] = useState(initialDark);
  const [manualThemeOverride, setManualThemeOverride] = useState(false);

  useEffect(() => {
    if (manualThemeOverride) return;
    const tick = () => {
      const h = new Date().getHours();
      setDarkMode(h >= 19 || h < 7);
    };
    tick();
    const id = setInterval(tick, 5 * 60 * 1000);
    return () => clearInterval(id);
  }, [manualThemeOverride]);

  useEffect(() => {
    const handleScroll = () => setScrolled(window.scrollY > 50);
    window.addEventListener("scroll", handleScroll);
    return () => window.removeEventListener("scroll", handleScroll);
  }, []);

  const handleAnalyze = async (payload, bypassCrisis = false, nonce = null) => {
    const { freeText = "", selectedMoods = [], birthYear, imageB64 = null } = payload || {};
    setError(null);

    const moodLabel = freeText || selectedMoods.join(", ");
    setCurrentMood(moodLabel);
    setResult(null);
    setLastPayload(payload);

    const combinedForCrisis = [freeText, selectedMoods.join(" ")].filter(Boolean).join(" ");

    if (!bypassCrisis && detectCrisis(combinedForCrisis)) {
      setCrisisOpen(true);
      setPendingContinue(payload);
      return;
    }

    setLoading(true);
    setProgressMsg("");
    setVisionInsight(null);
    try {
      const data = await fetchRecommendations({
        freeText, selectedMoods, birthYear,
        continueAnyway: bypassCrisis, nonce, imageB64,
        onProgress: (p) => {
          if (!p) return;
          if (p.step === "vision_insight" && p.insight) setVisionInsight(p.insight);
          else if (p.msg) setProgressMsg(p.msg);
        },
      });
      if (data.crisis && !bypassCrisis) {
        setCrisisOpen(true);
        setPendingContinue(payload);
        setLoading(false);
        return;
      }
      setResult(data);
    } catch (err) {
      console.error(err);
      setError("The nostalgic ether is unstable right now. Please try again in a moment.");
    } finally {
      setLoading(false);
      setProgressMsg("");
    }
  };

  const handleSurpriseMe = () => {
    if (!lastPayload || loading) return;
    const newNonce = (typeof crypto !== "undefined" && crypto.randomUUID)
      ? crypto.randomUUID()
      : String(Date.now()) + Math.random().toString(36).slice(2);
    handleAnalyze(lastPayload, false, newNonce);
  };

  const handleReturnHome = () => {
    setIsResearching(false);
    setResult(null);
    setError(null);
    setVisionInsight(null);
    setProgressMsg("");
    window.scrollTo({ top: 0, behavior: "smooth" });
  };

  return (
    <>
      <style>{THEME_CSS}</style>
      <div className={`ami-root noise-overlay${darkMode ? " ami-dark" : ""}`}>
        <Background active={!isResearching || result} dark={darkMode} />

        {/* Navigation */}
        <nav style={{
          position: "fixed", top: 0, width: "100%", zIndex: 50,
          padding: scrolled ? "1rem 1.5rem" : "1.5rem",
          display: "flex", justifyContent: "space-between", alignItems: "center",
          transition: "all 0.5s",
          background: scrolled ? (darkMode ? "rgba(10,14,39,0.9)" : "rgba(255,255,255,0.9)") : "transparent",
          backdropFilter: scrolled ? "blur(12px)" : "none",
          borderBottom: scrolled ? (darkMode ? "2px solid rgba(255,255,255,0.1)" : "2px solid rgba(0,0,0,0.1)") : "none",
          boxShadow: scrolled ? "0 25px 50px -12px rgba(0,0,0,0.25)" : "none",
        }}>
          <motion.div
            onClick={handleReturnHome}
            style={{ display: "flex", alignItems: "center", cursor: "pointer" }}
            animate={{ y: [0, -3, 0], rotate: [0.5, -0.5, 0.5] }}
            transition={{ duration: 4, repeat: Infinity, ease: "easeInOut" }}
          >
            <span className="storybook-title" style={{ fontSize: "3rem", letterSpacing: "-0.04em" }}>
              <FloatingWords text="Ami." intensity={0.8} />
            </span>
          </motion.div>

          <div style={{ display: "flex", gap: "0.75rem", alignItems: "center" }}>
            <motion.button
              onClick={() => { setManualThemeOverride(true); setDarkMode((d) => !d); }}
              aria-label={darkMode ? "Switch to day mode" : "Switch to night mode"}
              title={darkMode ? "Switch to Day Mode" : "Switch to Night Mode"}
              whileHover={{ scale: 1.1, rotate: 8, transition: { duration: 0.2 } }}
              whileTap={{ scale: 0.92 }}
              animate={{ y: [0, 3, 0] }}
              transition={{ duration: 5, repeat: Infinity, ease: "easeInOut" }}
              style={{
                padding: "0.5rem",
                borderRadius: "9999px",
                background: "transparent",
                border: "2px solid color-mix(in srgb, var(--color-fun-ink) 10%, transparent)",
                color: darkMode ? "var(--color-fun-yellow)" : "var(--color-fun-ink)",
                display: "flex", alignItems: "center", justifyContent: "center",
                cursor: "pointer", flexShrink: 0,
                transition: "background-color 0.2s ease",
              }}
              onMouseEnter={(e) => { e.currentTarget.style.background = "color-mix(in srgb, var(--color-fun-ink) 5%, transparent)"; }}
              onMouseLeave={(e) => { e.currentTarget.style.background = "transparent"; }}
            >
              {darkMode ? <Sun size={20} fill="currentColor" /> : <Moon size={20} fill="currentColor" />}
            </motion.button>

            <motion.button
              onClick={() => setIsResearching(true)}
              animate={{ y: [0, -2, 0] }}
              transition={{ duration: 3, repeat: Infinity, ease: "easeInOut" }}
              whileHover={{ scale: 1.05, y: -3, transition: { duration: 0.2 } }}
              whileTap={{ scale: 0.95 }}
              style={{
                padding: "0.5rem 1.5rem", borderRadius: "9999px",
                background: "var(--color-fun-ink)",
                color: "var(--color-fun-white)",
                fontFamily: "var(--font-display)", fontWeight: 700,
                fontSize: "0.875rem", letterSpacing: "0.1em",
                border: "none", cursor: "pointer",
              }}
            >
              <FloatingWords text="LAUNCH APP" intensity={0.5} />
            </motion.button>
          </div>
        </nav>

        <AnimatePresence mode="wait">
          {!isResearching ? (
            <motion.main
              key="landing"
              initial={{ opacity: 0 }}
              animate={{ opacity: 1 }}
              exit={{ opacity: 0, y: -20 }}
              style={{ position: "relative", zIndex: 10, paddingTop: "10rem", paddingLeft: "1.5rem", paddingRight: "1.5rem" }}
            >
              {/* Hero */}
              <section style={{ minHeight: "80vh", display: "flex", flexDirection: "column", alignItems: "center", justifyContent: "center", textAlign: "center", marginBottom: "10rem" }}>
                <AnimatePresence>
                  {badgeVisible && (
                    <motion.div
                      initial={{ opacity: 0, scale: 0.8 }}
                      animate={{ opacity: 1, scale: 1 }}
                      exit={{ y: 500, rotate: 45, opacity: 0, transition: { duration: 0.8, ease: "backIn" } }}
                      style={{ marginBottom: "2rem" }}
                    >
                      <motion.div
                        drag
                        dragConstraints={{ left: 0, right: 0, top: 0, bottom: 0 }}
                        dragElastic={0.1}
                        onClick={() => setBadgeVisible(false)}
                        animate={{ rotate: [2, -2, 2], y: [0, -8, 0] }}
                        transition={{
                          rotate: { duration: 4, repeat: Infinity, ease: "easeInOut" },
                          y: { duration: 3.5, repeat: Infinity, ease: "easeInOut" },
                        }}
                        whileHover={{ rotateZ: -10, y: -12, scale: 1.05 }}
                        whileTap={{ scale: 0.95 }}
                        style={{
                          display: "inline-block", padding: "0.5rem 1.5rem",
                          background: "var(--color-fun-mint)", borderRadius: "9999px",
                          border: "2px solid black", color: "black",
                          fontFamily: "var(--font-display)", fontWeight: 700,
                          fontSize: "0.75rem", marginBottom: "2rem", cursor: "pointer",
                          userSelect: "none",
                        }}
                      >
                        SCIENCE-BACKED RECOMMENDATIONS
                      </motion.div>
                    </motion.div>
                  )}
                </AnimatePresence>

                <motion.h1
                  initial={{ opacity: 0, y: 20 }}
                  animate={{ opacity: 1, y: 0 }}
                  className="storybook-title"
                  style={{
                    fontSize: "clamp(3.75rem, 12vw, 10rem)",
                    lineHeight: 0.9, letterSpacing: "-0.02em",
                    marginBottom: "2rem",
                    filter: "drop-shadow(0 25px 25px rgba(0,0,0,0.15))",
                  }}
                >
                  <FloatingWords text="Tell me" intensity={1} />
                  <br />
                  <FloatingWords text="how are you," style={{ color: "var(--color-fun-orange)", fontFamily: "var(--font-weird)", letterSpacing: "-0.05em" }} intensity={2.5} />
                  <br />
                  <FloatingWords text="I know what to" className="outline-white-on-dark" style={{ color: "white" }} intensity={1} />
                  <br />
                  <FloatingWords text="hand you." className="outline-white-on-dark" style={{ color: "white" }} intensity={1} />
                </motion.h1>

                <div style={{ maxWidth: "48rem", margin: "0 auto", marginBottom: "3rem" }}>
                  <motion.p
                    animate={{ y: [0, 5, 0] }}
                    transition={{ duration: 10, repeat: Infinity, ease: "easeInOut" }}
                    data-secondary-text
                    style={{
                      fontSize: "clamp(1.125rem, 2vw, 1.25rem)",
                      fontFamily: "var(--font-body)",
                      fontStyle: "italic",
                      color: "var(--text-secondary)",
                      lineHeight: 1.6, marginBottom: "3rem",
                    }}
                  >
                    <FloatingWords text="A movie, a book, a series, an anecdote from your childhood." intensity={0.4} />
                    <br />
                    <FloatingWords text="Recommendations chosen based on what you feel and what research says about" intensity={0.4} />{" "}
                    <motion.span
                      style={{ display: "inline-block", whiteSpace: "nowrap", cursor: "default", fontWeight: 700 }}
                      animate={{
                        y: [0, -4, 2, 0],
                        rotate: [0, 0.5, -0.5, 0],
                        color: ["var(--color-fun-pink)", "#ff8eb0", "var(--color-fun-pink)"],
                      }}
                      whileHover={{
                        y: -15,
                        rotate: (Math.random() - 0.5) * 15,
                        zIndex: 10,
                        transition: { duration: 0.2, type: "spring", stiffness: 400, damping: 10 },
                      }}
                      whileTap={{ rotate: (Math.random() - 0.5) * 30 }}
                      transition={{
                        y: { duration: 12, repeat: Infinity, ease: "easeInOut" },
                        rotate: { duration: 12, repeat: Infinity, ease: "easeInOut" },
                        color: { duration: 4, repeat: Infinity },
                      }}
                    >
                      nostalgia
                    </motion.span>,{" "}
                    <motion.span
                      style={{ display: "inline-block", whiteSpace: "nowrap", cursor: "default", fontWeight: 700 }}
                      animate={{
                        y: [0, -4, 2, 0],
                        rotate: [0, -0.5, 0.5, 0],
                        color: ["var(--color-fun-mint)", "#98ff98", "var(--color-fun-mint)"],
                      }}
                      whileHover={{
                        y: -15,
                        rotate: (Math.random() - 0.5) * 15,
                        zIndex: 10,
                        transition: { duration: 0.2, type: "spring", stiffness: 400, damping: 10 },
                      }}
                      whileTap={{ rotate: (Math.random() - 0.5) * 30 }}
                      transition={{
                        y: { duration: 13, repeat: Infinity, ease: "easeInOut" },
                        rotate: { duration: 13, repeat: Infinity, ease: "easeInOut" },
                        color: { duration: 5, repeat: Infinity },
                      }}
                    >
                      awe
                    </motion.span>, <FloatingWords text="and the stories that repair us." intensity={0.4} />
                  </motion.p>
                  <motion.p
                    animate={{ y: [0, -3, 0] }}
                    transition={{ duration: 7, repeat: Infinity, ease: "easeInOut", delay: 1 }}
                    style={{ fontSize: "clamp(1rem, 2vw, 1.125rem)", fontFamily: "var(--font-sketch)", color: "var(--color-fun-pink)", fontWeight: 700, marginBottom: "1.5rem", textAlign: "center" }}
                  >
                    It may pop your bubble !
                  </motion.p>

                  <motion.div
                    animate={{ y: [0, -10, 0], rotate: [0.5, -0.5, 0.5] }}
                    transition={{ duration: 6, repeat: Infinity, ease: "easeInOut" }}
                  >
                    <motion.button
                      whileHover={{ scale: 1.05, rotate: -2, backgroundColor: "var(--color-fun-pink)" }}
                      whileTap={{ scale: 0.95 }}
                      onClick={() => setIsResearching(true)}
                      style={{
                        marginBottom: "5rem", padding: "1.5rem 4rem",
                        background: "var(--color-fun-ink)",
                        color: "var(--color-fun-white)",
                        borderRadius: "9999px", border: "none", cursor: "pointer",
                        fontFamily: "var(--font-display)", fontWeight: 900,
                        fontSize: "1.5rem", letterSpacing: "0.1em",
                        boxShadow: "0 25px 50px -12px rgba(0,0,0,0.25)",
                        transition: "background-color 0.3s ease",
                      }}
                    >
                      START NOW
                    </motion.button>
                  </motion.div>

                  {/* Features */}
                  <div style={{ display: "grid", gridTemplateColumns: "repeat(auto-fit, minmax(140px, 1fr))", gap: "1rem", marginBottom: "5rem" }}>
                    {[
                      { title: "Easy to use", desc: "Just your mood and a birth year" },
                      { title: "Privacy-minded", desc: "Your mood stays with you" },
                      { title: "Research-backed", desc: "Every reco is sourced" },
                      { title: "Gemma 4 powered", desc: "via Google AI Studio" },
                    ].map((item, i) => (
                      <motion.div
                        key={i}
                        animate={{ y: [0, -10, 0], rotate: [1, -1, 1] }}
                        transition={{ duration: 6 + i, repeat: Infinity, ease: "easeInOut", delay: i * 0.2 }}
                        whileHover={{ scale: 1.1, y: -20, zIndex: 20, backgroundColor: "rgba(255,255,255,0)" }}
                        whileTap={{ scale: 0.95 }}
                        className="sticker-shadow"
                        data-surface="white"
                        style={{
                          padding: "1rem",
                          background: "var(--color-fun-white)",
                          border: "2px solid var(--color-fun-ink)",
                          borderRadius: "1rem", transform: "rotate(1deg)",
                          display: "flex", flexDirection: "column", justifyContent: "center",
                          cursor: "default",
                        }}
                      >
                        <span style={{ fontSize: "0.75rem", fontFamily: "var(--font-display)", fontWeight: 900, marginBottom: "0.25rem", textTransform: "uppercase", letterSpacing: "-0.04em", color: "var(--color-fun-pink)" }}>{item.title}</span>
                        <span style={{ fontSize: "0.875rem", fontFamily: "var(--font-sketch)", lineHeight: 1.2 }}>{item.desc}</span>
                      </motion.div>
                    ))}
                  </div>

                  {/* Protocol */}
                  <motion.div
                    animate={{ y: [0, -15, 0], x: [0, 5, 0] }}
                    transition={{ duration: 12, repeat: Infinity, ease: "easeInOut" }}
                    className="sticker-shadow"
                    data-surface="white"
                    style={{
                      textAlign: "left", marginBottom: "5rem",
                      padding: "3rem", background: "var(--color-fun-white)",
                      border: "4px solid var(--color-fun-ink)", borderRadius: "3rem",
                      position: "relative",
                    }}
                  >
                    <h3 style={{ fontSize: "clamp(2rem, 5vw, 4rem)", fontFamily: "var(--font-display)", fontWeight: 900, marginBottom: "1.5rem", letterSpacing: "-0.02em" }}>
                      <FloatingWords text="PROTOCOL" /> <br />
                      <span style={{
                        display: "inline-block",
                        whiteSpace: "nowrap",
                        color: "var(--color-fun-blue)",
                        fontStyle: "italic",
                        fontFamily: "var(--font-sketch)",
                        WebkitTextStroke: darkMode ? "0px" : "1px black",
                      }}>Simple. Human. Intuitive.</span>
                    </h3>
                    <p data-secondary-text style={{ fontSize: "clamp(1.125rem, 2vw, 1.25rem)", fontFamily: "var(--font-body)", fontStyle: "italic", color: "var(--text-secondary)", marginBottom: "3rem", lineHeight: 1.6 }}>
                      <FloatingWords text="Most wellness apps ask you several questions and a subscription." intensity={0.4} /> <br />
                      <FloatingWords text="Ami asks just two and they are forgotten when you close the app." intensity={0.4} />
                    </p>
                    <div style={{ display: "grid", gridTemplateColumns: "repeat(auto-fit, minmax(280px, 1fr))", gap: "2rem" }}>
                      {[
                        { num: "01", title: "Describe your mood", desc: "Type, speak, drop an image or a meme (yes, a meme!), Ami hears and sees you." },
                        { num: "02", title: "Give your birth year", desc: "Ami uses it to target the nostalgia window (ages 8-16), where sensory memories are deepest." },
                        { num: "03", title: "Ami hears or sees you", desc: "She decodes your emotion and selects a helpful mechanism from the research." },
                        { num: "04", title: "Ami hands you 4 items", desc: "A film, a series, a novel, an anecdote, each sourced from peer-reviewed psychology." },
                      ].map((step, i) => (
                        <motion.div
                          key={i}
                          animate={{ y: [0, -5, 0] }}
                          transition={{ duration: 4 + i, repeat: Infinity, ease: "easeInOut", delay: i * 0.5 }}
                          style={{ display: "flex", gap: "1rem", alignItems: "flex-start" }}
                        >
                          <span data-step-num style={{ fontSize: "1.875rem", fontFamily: "var(--font-display)", fontWeight: 900, color: "var(--text-faint)" }}>{step.num}</span>
                          <div>
                            <h4 style={{ fontFamily: "var(--font-display)", fontWeight: 700, fontSize: "1.25rem", marginBottom: "0.25rem" }}>
                              <FloatingWords text={step.title} intensity={0.6} />
                            </h4>
                            <p data-secondary-text style={{ fontSize: "0.875rem", fontFamily: "var(--font-body)", color: "var(--text-secondary)", lineHeight: 1.6, fontWeight: 500 }}>
                              <FloatingWords text={step.desc} intensity={0.3} />
                            </p>
                          </div>
                        </motion.div>
                      ))}
                    </div>
                  </motion.div>
                </div>
              </section>


              {/* ── Mood mask gallery ──────────────────────────── */}
              <MoodMaskGallery />

              {/* ── Science section — Tuner ──────────────────────── */}
              <section style={{ padding: "2rem 2rem 0", maxWidth: "900px", margin: "0 auto" }}>

                <AnimatePresence>
                  {showAllMechanisms && (
                    <motion.div
                      key="all-mech"
                      initial={{ opacity: 0 }}
                      animate={{ opacity: 1 }}
                      exit={{ opacity: 0 }}
                      onClick={(e) => { if (e.target === e.currentTarget) setShowAllMechanisms(false); }}
                      style={{
                        position: "fixed", inset: 0, zIndex: 9999,
                        background: "rgba(0,0,0,0.75)", backdropFilter: "blur(6px)",
                        overflowY: "auto", padding: "2rem 1rem",
                        display: "flex", flexDirection: "column", alignItems: "center",
                        paddingTop: "5rem",
                      }}
                    >
                      <motion.div
                        initial={{ y: 60, opacity: 0 }}
                        animate={{ y: 0, opacity: 1 }}
                        exit={{ y: 60, opacity: 0 }}
                        transition={{ type: "spring", stiffness: 280, damping: 28 }}
                        style={{
                          background: "var(--color-fun-white)",
                          borderRadius: "2rem", padding: "3rem 2.5rem 2.5rem",
                          maxWidth: "760px", width: "100%",
                          border: "3px solid var(--color-fun-ink)",
                          boxShadow: "0 0 0 4px var(--shadow-stroke), 10px 10px 0 0 var(--shadow-color)",
                          position: "relative",
                          marginBottom: "2rem",
                        }}
                      >
                        <motion.button
                          whileHover={{ scale: 1.1, rotate: 10 }}
                          whileTap={{ scale: 0.9 }}
                          onClick={() => setShowAllMechanisms(false)}
                          style={{
                            position: "absolute", top: "1.25rem", right: "1.25rem",
                            background: "var(--color-fun-ink)", color: "var(--color-fun-white)",
                            border: "none", borderRadius: "9999px",
                            width: "2.5rem", height: "2.5rem",
                            fontSize: "1.1rem", cursor: "pointer",
                            display: "flex", alignItems: "center", justifyContent: "center",
                            fontFamily: "var(--font-display)", fontWeight: 900,
                          }}
                        >×</motion.button>

                        <h2 style={{ fontFamily: "var(--font-display)", fontWeight: 900, fontSize: "clamp(1.6rem,4vw,2.2rem)", marginBottom: "0.4rem", letterSpacing: "-0.02em" }}>
                          <FloatingWords text="11 psychological mechanisms" intensity={0.5} />
                        </h2>
                        <p style={{ color: "var(--text-secondary)", fontFamily: "var(--font-body)", marginBottom: "2.5rem", fontSize: "1rem", lineHeight: 1.6 }}>
                          <FloatingWords text="Every recommendation Ami makes is rooted in one of these peer-reviewed frameworks." intensity={0.3} />
                        </p>

                        <div style={{ display: "flex", flexDirection: "column", gap: "1.25rem" }}>
                          {MECHS.map((m, i) => (
                            <motion.div
                              key={m.id}
                              initial={{ opacity: 0, x: -16 }}
                              animate={{ opacity: 1, x: 0 }}
                              transition={{ delay: i * 0.045 }}
                              whileHover={{ x: 6 }}
                              style={{
                                padding: "1.25rem 1.5rem",
                                background: "var(--color-fun-white)",
                                borderRadius: "1.25rem",
                                border: "2px solid var(--color-fun-ink)",
                                borderLeft: `5px solid ${m.color}`,
                                boxShadow: `4px 4px 0 0 ${m.color}`,
                              }}
                            >
                              <div style={{ display: "flex", alignItems: "center", gap: "0.75rem", marginBottom: "0.6rem", flexWrap: "wrap" }}>
                                <strong style={{ fontFamily: "var(--font-display)", fontWeight: 800, fontSize: "1.15rem" }}>
                                  <FloatingWords text={m.label} intensity={0.4} />
                                </strong>
                                <span
                                  className="ami-mech-tag"
                                  style={{
                                    fontSize: "0.65rem", fontWeight: 700, letterSpacing: "0.08em",
                                    background: m.color, color: "#000", borderRadius: "9999px",
                                    padding: "0.15rem 0.6rem", textTransform: "uppercase",
                                    fontFamily: "var(--font-body)", flexShrink: 0,
                                  }}>{m.label}</span>
                              </div>
                              <p style={{ fontFamily: "var(--font-body)", fontSize: "0.925rem", color: "var(--text-secondary)", lineHeight: 1.65, margin: "0 0 0.5rem" }}>
                                <FloatingWords text={m.desc} intensity={0.2} />
                              </p>
                              <p style={{ fontSize: "0.72rem", fontFamily: "var(--font-body)", fontStyle: "italic", color: "var(--text-muted)", lineHeight: 1.4 }}>
                                {m.paper}
                              </p>
                            </motion.div>
                          ))}
                        </div>
                      </motion.div>
                    </motion.div>
                  )}
                </AnimatePresence>

                <motion.div
                  initial={{ opacity: 0, y: 24 }}
                  whileInView={{ opacity: 1, y: 0 }}
                  viewport={{ once: true }}
                  transition={{ duration: 0.5 }}
                  style={{ textAlign: "center", marginBottom: "3rem" }}
                >
                  <h2 style={{
                    fontFamily: "var(--font-display)", fontWeight: 900,
                    fontSize: "clamp(2rem, 5vw, 4rem)", marginBottom: "1rem",
                    letterSpacing: "-0.02em", lineHeight: 1.1,
                    color: "var(--color-fun-ink)",
                  }}>
                    <FloatingWords text="The science behind your" intensity={0.5} />{" "}
                    <motion.span
                      animate={{ color: ["var(--color-fun-pink)", "var(--color-fun-mint)", "var(--color-fun-blue)", "var(--color-fun-pink)"] }}
                      transition={{ duration: 7, repeat: Infinity }}
                      style={{ display: "inline-block" }}
                    >
                      <FloatingWords text="feelings" intensity={0.5} />
                    </motion.span>
                  </h2>
                  <motion.p
                    animate={{ y: [0, -3, 0] }}
                    transition={{ duration: 8, repeat: Infinity, ease: "easeInOut" }}
                    style={{ fontFamily: "var(--font-body)", fontSize: "clamp(1.05rem,2vw,1.2rem)", color: darkMode ? "var(--text-secondary)" : "rgba(255,255,255,0.85)", maxWidth: "540px", margin: "0 auto", lineHeight: 1.7 }}
                  >
                    <FloatingWords text="Ami matches your state to one of 11 validated psychological mechanisms, each backed by peer-reviewed research." intensity={0.25} />
                  </motion.p>
                </motion.div>

                <TunerDial mechs={MECHS} dark={darkMode} />

                <motion.div
                  style={{ textAlign: "center", marginTop: "5rem", marginBottom: "6rem" }}
                  animate={{ y: [0, -6, 0], rotate: [0.3, -0.3, 0.3] }}
                  transition={{ duration: 5, repeat: Infinity, ease: "easeInOut" }}
                >
                  <motion.button
                    whileHover={{ scale: 1.06, rotate: -2, backgroundColor: "var(--color-fun-mint)" }}
                    whileTap={{ scale: 0.95 }}
                    onClick={() => setShowAllMechanisms(true)}
                    className="sticker-shadow"
                    style={{
                      padding: "1rem 3rem",
                      background: "var(--color-fun-yellow)",
                      color: "#000000",
                      border: "2px solid #000000",
                      borderRadius: "9999px", cursor: "pointer",
                      fontFamily: "var(--font-display)", fontWeight: 900,
                      fontSize: "1.15rem", letterSpacing: "0.03em",
                      transition: "background-color 0.2s ease",
                    }}
                  >
                    <FloatingWords text="See all 11 mechanisms + papers" intensity={0.6} />
                  </motion.button>
                </motion.div>
              </section>


              {/* Footer */}
              <footer style={{
                padding: "6rem 2rem", background: "black", color: "white",
                borderRadius: "4rem", marginBottom: "2.5rem",
                overflow: "hidden", position: "relative", borderTop: "4px solid black",
              }}>
                <div style={{
                  position: "absolute", top: 0, right: 0,
                  width: "16rem", height: "16rem",
                  background: "var(--color-fun-pink)", filter: "blur(120px)",
                  opacity: 0.2, marginRight: "-5rem", marginTop: "-5rem",
                }} />
                <motion.div
                  animate={{ y: [0, -10, 0] }}
                  transition={{ duration: 6, repeat: Infinity, ease: "easeInOut" }}
                  style={{
                    maxWidth: "80rem", margin: "0 auto",
                    display: "flex", flexDirection: "row", flexWrap: "wrap",
                    justifyContent: "space-between", alignItems: "center",
                    gap: "5rem", position: "relative", zIndex: 10,
                  }}
                >
                  <div style={{ flex: 1, minWidth: "300px" }}>
                    <p style={{ fontSize: "clamp(1.875rem, 4vw, 3rem)", fontFamily: "var(--font-display)", fontWeight: 900, marginBottom: "1rem", lineHeight: 1.1 }}>
                      <FloatingWords text="What you feel, someone has already" intensity={0.8} /> <br />
                      <FloatingWords text="written," style={{ color: "var(--color-fun-pink)", fontStyle: "italic" }} intensity={0.8} />{" "}
                      <FloatingWords text="filmed," style={{ color: "var(--color-fun-blue)", fontStyle: "italic" }} intensity={0.8} />{" "}
                      <FloatingWords text="or" intensity={0.8} />{" "}
                      <FloatingWords text="painted it." style={{ color: "var(--color-fun-orange)", fontStyle: "italic" }} intensity={0.8} />
                    </p>
                    <p style={{ fontSize: "1.5rem", fontFamily: "var(--font-sketch)", color: "rgba(255,255,255,0.7)", fontStyle: "italic" }}>
                      <FloatingWords text="Ami just helps you find it again." intensity={0.5} />
                    </p>
                  </div>

                  <div style={{ display: "flex", flexDirection: "column", gap: "1.5rem", alignItems: "flex-end" }}>
                    <motion.button
                      animate={{ y: [0, -8, 0], scale: [1, 1.05, 1] }}
                      transition={{ duration: 4, repeat: Infinity, ease: "easeInOut" }}
                      onClick={() => { window.scrollTo({ top: 0, behavior: "smooth" }); setIsResearching(true); }}
                      whileHover={{ scale: 1.05, rotate: 2 }}
                      className="sticker-shadow"
                      style={{
                        padding: "1.5rem 3rem", borderRadius: "9999px",
                        background: "var(--color-fun-mint)", color: "black",
                        fontFamily: "var(--font-display)", fontWeight: 900,
                        fontSize: "1.5rem", border: "4px solid transparent",
                        cursor: "pointer",
                      }}
                    >
                      <FloatingWords text="TRY IT" />
                    </motion.button>
                    <span style={{ fontSize: "0.875rem", fontFamily: "monospace", opacity: 0.6, textTransform: "uppercase", letterSpacing: "0.1em" }}>
                      <FloatingWords text="© 2026 AMI · Powered by Gemma 4" />
                    </span>

                  </div>
                </motion.div>
              </footer>
              {/* ── Disclaimer ── */}
<p style={{
  textAlign: "center", fontFamily: "var(--font-body)",
  fontSize: "0.65rem", fontStyle: "italic",
  color: "var(--text-muted)", marginTop: "3rem",
  paddingBottom: "2rem", lineHeight: 1.5,
}}>
  Ami does not assess mental health, nor provide clinical advice,
  and is not a substitute for professional support.
</p>
            </motion.main>
          ) : (
            <motion.main
              key="research"
              initial={{ opacity: 0, scale: 0.95 }}
              animate={{ opacity: 1, scale: 1 }}
              exit={{ opacity: 0, scale: 1.05 }}
              style={{ position: "relative", zIndex: 10, paddingTop: "10rem", paddingLeft: "1.5rem", paddingRight: "1.5rem", paddingBottom: "10rem" }}
            >
              <div style={{ maxWidth: "72rem", margin: "0 auto" }}>
                <button
                  onClick={handleReturnHome}
                  style={{
                    marginBottom: "3rem", display: "flex", alignItems: "center", gap: "0.5rem",
                    color: "var(--text-secondary)", fontFamily: "var(--font-display)", fontWeight: 700,
                    background: "transparent", border: "none", cursor: "pointer",
                  }}
                >
                  ← <FloatingWords text="BACK TO HOME" />
                </button>

                <div style={{ marginBottom: "5rem" }}>
                  <ResearchInput onAnalyze={handleAnalyze} isLoading={loading} />
            {loading && progressMsg && (
              <motion.div
                initial={{ opacity: 0, y: 8 }} animate={{ opacity: 1, y: 0 }}
                style={{
                  marginTop: '1.5rem', padding: '1rem 1.5rem',
                  borderRadius: '1.5rem',
                  background: 'var(--color-fun-white)',
                  border: '3px solid var(--color-fun-ink)',
                  boxShadow: '0 0 0 3px var(--shadow-stroke), 4px 4px 0 0 var(--shadow-color)',
                  fontFamily: 'var(--font-display)', fontWeight: 700,
                  fontSize: '1.1rem', textAlign: 'center',
                }}
              >
                ✨ {progressMsg}
              </motion.div>
            )}
            {visionInsight && (
              <motion.div
                ref={visionInsightRef}
                className="vision-insight-card"
                initial={{ opacity: 0, y: 12, scale: 0.96 }}
                animate={{ opacity: 1, y: [0, -8, 0], scale: 1, rotate: [0, 0.4, -0.4, 0] }}
                transition={{
                  opacity: { duration: 0.4 },
                  scale: { duration: 0.4 },
                  y: { duration: 7, repeat: Infinity, ease: "easeInOut", delay: 0.4 },
                  rotate: { duration: 7, repeat: Infinity, ease: "easeInOut", delay: 0.4 },
                }}
                whileHover={{
                  y: -16,
                  rotate: 1,
                  scale: 1.01,
                  transition: { duration: 0.25, type: "spring", stiffness: 300, damping: 18 },
                }}
                whileTap={{ scale: 0.98, rotate: (Math.random() - 0.5) * 4 }}
                style={{
                  marginTop: "2.5rem", padding: "2.25rem 2.5rem",
                  borderRadius: "2rem",
                  background: "var(--color-fun-white)",
                  border: "4px solid var(--color-fun-ink)",
                  boxShadow: "0 0 0 4px var(--shadow-stroke), 12px 12px 0 0 var(--shadow-color)",
                  fontFamily: "var(--font-display)",
                  position: "relative",
                  cursor: "default",
                }}
              >
                {/* "First reading" badge — floats on its own */}
                <motion.div
                  animate={{ y: [0, -4, 0], rotate: [-1, 1, -1] }}
                  transition={{ duration: 5, repeat: Infinity, ease: "easeInOut" }}
                  whileTap={{ scale: 0.95, rotate: (Math.random() - 0.5) * 12 }}
                  style={{
                    position: "absolute", top: "-1rem", left: "2rem",
                    padding: "0.3rem 1rem", borderRadius: "9999px",
                    background: "var(--color-fun-ink)",
                    color: "var(--color-fun-white)",
                    fontSize: "0.62rem", fontWeight: 800,
                    letterSpacing: "0.28em", textTransform: "uppercase",
                  }}>
                  First reading
                </motion.div>
                {/* Input type label — floats */}
                <motion.div
                  animate={{ y: [0, -5, 0] }}
                  transition={{ duration: 6, repeat: Infinity, ease: "easeInOut", delay: 0.3 }}
                  whileTap={{ scale: 0.95 }}
                  style={{
                    fontSize: "0.9rem", fontWeight: 800,
                    letterSpacing: "0.1em", textTransform: "uppercase",
                    color: "var(--color-fun-pink)", marginBottom: "0.65rem",
                  }}>
                  {visionInsight.input_type === "image+text"
                    ? "👁️ Ami sees you and heard you"
                    : visionInsight.input_type === "text"
                    ? "👂 Ami heard you"
                    : "👁️ Ami sees you"}
                </motion.div>
                {/* Main understanding text — uses FloatingWords */}
                <div
                  className="vision-insight-understanding"
                  style={{
                  fontSize: "clamp(1.4rem, 2.6vw, 1.9rem)",
                  fontWeight: 800, lineHeight: 1.25,
                  marginBottom: "0.85rem",
                  color: "var(--color-fun-ink)",
                }}>
                  "<FloatingWords text={visionInsight.understanding || visionInsight.image_read || visionInsight.decoder_note || ""} intensity={0.6} />"
                </div>
                {visionInsight.quote_author && (
                  <motion.div
                    animate={{ y: [0, -3, 0] }}
                    transition={{ duration: 7, repeat: Infinity, ease: "easeInOut", delay: 0.6 }}
                    style={{
                      fontSize: "0.95rem", fontWeight: 700,
                      color: "var(--text-muted)",
                      marginBottom: "0.5rem",
                    }}>
                    — recognised as {visionInsight.quote_author}
                  </motion.div>
                )}
                {/* Tags row — each tag floats independently */}
                <div style={{
                  display: "flex", flexWrap: "wrap", gap: "0.5rem",
                  marginTop: "1rem", alignItems: "center",
                }}>
                  <motion.span
                    animate={{ y: [0, -3, 0] }}
                    transition={{ duration: 6, repeat: Infinity, ease: "easeInOut" }}
                    whileHover={{ y: -5, transition: { duration: 0.3 } }}
                    style={{
                      padding: "0.4rem 1rem", borderRadius: "9999px",
                      background: "var(--color-fun-mint)",
                      border: "2px solid var(--color-fun-ink)",
                      fontSize: "0.95rem", fontWeight: 800,
                      display: "inline-block", cursor: "default",
                    }}
                    className="vision-tag">
                    {visionInsight.emotion_core}
                  </motion.span>
                  {(visionInsight.active_moods || []).slice(0, 3).map((m, mi) => (
                    <motion.span
                      key={m}
                      animate={{ y: [0, -3, 0] }}
                      transition={{ duration: 6.5 + mi * 0.5, repeat: Infinity, ease: "easeInOut", delay: mi * 0.4 }}
                      whileHover={{ y: -5, transition: { duration: 0.3 } }}
                      style={{
                        padding: "0.35rem 0.85rem", borderRadius: "9999px",
                        background: "var(--color-fun-pink)",
                        border: "2px solid var(--color-fun-ink)",
                        fontSize: "0.85rem", fontWeight: 700,
                        display: "inline-block", cursor: "default",
                      }}
                      className="vision-tag">
                      {m}
                    </motion.span>
                  ))}
                </div>
                {visionInsight.decoder_note && visionInsight.input_type !== "text" && (
                  <motion.div
                    animate={{ y: [0, -4, 0] }}
                    transition={{ duration: 8, repeat: Infinity, ease: "easeInOut", delay: 1 }}
                    style={{
                      marginTop: "1rem", fontSize: "0.95rem",
                      fontStyle: "italic", color: "var(--text-secondary)",
                    }}>
                    {visionInsight.decoder_note}
                  </motion.div>
                )}
              </motion.div>
            )}
                </div>

                {error && (
                  <motion.p initial={{ opacity: 0 }} animate={{ opacity: 1 }} style={{
                    marginTop: "2rem", color: "var(--color-fun-ink)",
                    background: "var(--color-fun-pink)", padding: "1rem",
                    borderRadius: "1rem", border: "2px solid var(--color-fun-ink)", fontWeight: 700,
                  }}>
                    {error}
                  </motion.p>
                )}

                <AnimatePresence>
                  {result && (
                    <motion.section
                      initial={{ opacity: 0, y: 50 }}
                      animate={{ opacity: 1, y: 0 }}
                      style={{ marginTop: "5rem" }}
                    >
                      <h1
                        ref={resultsRef}
                        className="storybook-title"
                        style={{
                          fontSize: "clamp(3rem, 8vw, 6rem)",
                          lineHeight: 1.05,
                          marginBottom: "5rem",
                          textAlign: "left",
                          scrollMarginTop: "10rem",
                          letterSpacing: "-0.02em",
                          filter: "drop-shadow(0 25px 25px rgba(0,0,0,0.15))",
                        }}
                      >
                        <FloatingWords text="And here is" intensity={0.8} />
                        <br />
                        <FloatingWords text="what she" style={{ color: "var(--color-fun-pink)" }} intensity={1.2} />
                        {" "}
                        <FloatingWords text="hands you." style={{ color: "var(--color-fun-blue)" }} intensity={1.2} />
                      </h1>

                      <MagazineView
                        result={result}
                        currentMood={currentMood}
                        loading={loading}
                        onSurpriseMe={handleSurpriseMe}
                        onReturn={handleReturnHome}
                      />
                    </motion.section>
                  )}
                </AnimatePresence>
              {/* ── Disclaimer ── */}
<p style={{
  textAlign: "center", fontFamily: "var(--font-body)",
  fontSize: "0.65rem", fontStyle: "italic",
  color: "var(--text-muted)", marginTop: "3rem",
  paddingBottom: "2rem", lineHeight: 1.5,
}}>
  Ami does not assess mental health, nor provide clinical advice,
  and is not a substitute for professional support.
</p>
              </div>
            </motion.main>
          )}
        </AnimatePresence>

        <AnimatePresence>
          {crisisOpen && (
            <CrisisPanel
              onClose={() => { setCrisisOpen(false); setPendingContinue(null); }}
              onContinue={() => {
                setCrisisOpen(false);
                if (pendingContinue) {
                  handleAnalyze(pendingContinue, true);
                  setPendingContinue(null);
                }
              }}
            />
          )}
        </AnimatePresence>

        <AnimatePresence>
          {detailsPayload && (
            <DetailsModal
              payload={detailsPayload}
              onClose={() => setDetailsPayload(null)}
            />
          )}
        </AnimatePresence>

        <AnimatePresence>
          {thesisOpen && result && (
            <ThesisModal
              thesis={result.thesis}
              thematicChart={result.thematicChart}
              currentMood={currentMood}
              onClose={() => setThesisOpen(false)}
            />
          )}
        </AnimatePresence>
      </div>


    </>
  );
}
"""

with open('/kaggle/working/static/ami.jsx', 'w') as f:
    f.write(AMI_JSX)

print(f'✅ ami.jsx written ({len(AMI_JSX)} chars)')

✅ ami.jsx written (188342 chars)


In [16]:
# ════════════════════════════════════════════════
# 12 · HTML wrapper (Babel + import map for shared React)
#
# Why an import map: framer-motion uses React internals (contexts, hooks).
# If two copies of React are loaded (one UMD + one bundled by esm.sh) the
# motion.* components call useContext on the WRONG React instance and crash
# with "null is not an object (evaluating 'l.current.useContext')".
# An import map forces every ESM "import 'react'" to resolve to the SAME URL,
# guaranteeing a single React instance shared by Ami, framer-motion and
# lucide-react.
# ════════════════════════════════════════════════

import re

with open('/kaggle/working/static/ami.jsx', 'r') as f:
    jsx_src = f.read()

# Strip ES module imports — every dependency is provided as a global below
jsx_clean = re.sub(r'^import\s+React.*?;\n', '', jsx_src, flags=re.MULTILINE)
jsx_clean = re.sub(r'^import\s+\{[^}]+\}.*?from\s+["\'][^"\']+["\'];\n', '', jsx_clean, flags=re.MULTILINE)
jsx_clean = re.sub(r'^export\s+default\s+', '', jsx_clean, flags=re.MULTILINE)
jsx_clean = re.sub(r'^export\s+', '', jsx_clean, flags=re.MULTILINE)

remaining = re.findall(r"^import\s+", jsx_clean, re.MULTILINE)
print(f"Imports résiduels : {len(remaining)} (doit être 0)")

# Escape the JSX for embedding in a JS template literal
jsx_for_js = (
    jsx_clean
    .replace("\\", "\\\\")
    .replace("`", "\\`")
    .replace("${", "\\${")
)

html = """<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1.0" />
  <title>Ami</title>
  <style>
    * { margin: 0; padding: 0; box-sizing: border-box; }
    body { background: #38BDF8; }
    #root { min-height: 100vh; }
    #loading {
      display: flex; align-items: center; justify-content: center;
      height: 100vh; font-family: 'Fredoka', sans-serif;
      color: white; font-size: 18px; letter-spacing: 0.05em;
    }
  </style>
  <link href="https://fonts.googleapis.com/css2?family=Fredoka:wght@300;400;500;600;700&family=Quicksand:wght@300;400;500;600;700&family=Gochi+Hand&family=Lacquer&display=swap" rel="stylesheet">

  <!-- Import map: ensures every "import 'react'" in any ESM module resolves
       to the SAME URL. Without this, framer-motion bundles its own React copy
       and calls useContext on a different React instance -> crash. -->
  <script type="importmap">
  {
    "imports": {
      "react":            "https://esm.sh/react@18.3.1",
      "react/":           "https://esm.sh/react@18.3.1/",
      "react-dom":        "https://esm.sh/react-dom@18.3.1",
      "react-dom/":       "https://esm.sh/react-dom@18.3.1/",
      "react-dom/client": "https://esm.sh/react-dom@18.3.1/client"
    }
  }
  </script>
</head>
<body>
  <div id="root"><div id="loading">Loading Ami...</div></div>

  <!-- Babel standalone (sync, used inside the module below) -->
  <script src="https://unpkg.com/@babel/standalone@7.24.7/babel.min.js"></script>

  <script type="module">
    // All these imports go through the import map -> single React instance.
    import * as React from "react";
    import * as ReactDOMClient from "react-dom/client";
    import * as Lucide   from "https://esm.sh/lucide-react@0.383.0?external=react";
    import * as FM       from "https://esm.sh/framer-motion@11.18.2?external=react,react-dom";
    import * as Recharts from "https://esm.sh/recharts@2.12.7?external=react,react-dom";
    // react-pageflip wraps the StPageFlip library — same one used by
    // the prototype. Marking react + react-dom external is essential so
    // the bundle re-uses OUR React instance via the import map (avoids
    // hooks-out-of-order crashes from a duplicate React).
    import HTMLFlipBookDefault from "https://esm.sh/react-pageflip@2.0.3?external=react,react-dom";
    const HTMLFlipBook = HTMLFlipBookDefault.default || HTMLFlipBookDefault;

    const {
      Sparkles, Quote, Star, Loader2, Video, Tv, Book,
      Heart, Phone, X, ArrowUpRight, Sun, Moon,
      Mic, MicOff, ImagePlus,
      ChevronLeft, ChevronRight
    } = Lucide;

    const { motion, AnimatePresence } = FM;
    const { AreaChart, Area, ResponsiveContainer } = Recharts;

    const { useState, useEffect, useMemo } = React;

    const jsxSource = `__JSX_PLACEHOLDER__`;

    const jsCode = Babel.transform(jsxSource, { presets: ['react'] }).code;

    const moduleFactory = new Function(
      'React', 'useState', 'useEffect', 'useMemo',
      'motion', 'AnimatePresence',
      'Sparkles', 'Quote', 'Star', 'Loader2', 'Video', 'Tv', 'Book',
      'Heart', 'Phone', 'X', 'ArrowUpRight', 'Sun', 'Moon',
      'Mic', 'MicOff', 'ImagePlus',
      'ChevronLeft', 'ChevronRight',
      'AreaChart', 'Area', 'ResponsiveContainer',
      'HTMLFlipBook',
      jsCode + '; return Ami;'
    );

    const Ami = moduleFactory(
      React, useState, useEffect, useMemo,
      motion, AnimatePresence,
      Sparkles, Quote, Star, Loader2, Video, Tv, Book,
      Heart, Phone, X, ArrowUpRight, Sun, Moon,
      Mic, MicOff, ImagePlus,
      ChevronLeft, ChevronRight,
      AreaChart, Area, ResponsiveContainer,
      HTMLFlipBook
    );

    const root = ReactDOMClient.createRoot(document.getElementById('root'));
    root.render(React.createElement(Ami));
    console.log('✅ Ami mounted (single React instance via import map)');
  </script>
</body>
</html>"""

html = html.replace("__JSX_PLACEHOLDER__", jsx_for_js)

with open('/kaggle/working/index.html', 'w') as f:
    f.write(html)

print(f'✅ index.html régénéré ({len(html):,} chars)')


Imports résiduels : 0 (doit être 0)
✅ index.html régénéré (192,085 chars)


## 6. Server

Flask app exposing `/api/recommend` (the full pipeline with `agentic` toggle), `/api/health`, and the static frontend. Run on port 5000, exposed via ngrok.

In [17]:
# ════════════════════════════════════════════════
# 13 · Flask server with the v3 endpoint (agentic toggle)
# ════════════════════════════════════════════════

from flask import Response, Flask, request, jsonify, send_from_directory, send_file
from flask_cors import CORS
import threading, time, traceback, torch

app = Flask(__name__, static_folder='/kaggle/working/static')
CORS(app)

# ─── Frontend ───────────────────────────────────────────────────────
@app.route('/')
def index():
    return send_file('/kaggle/working/index.html')

@app.route('/static/<path:filename>')
def static_files(filename):
    return send_from_directory('/kaggle/working/static', filename)


# ─── Main endpoint: full pipeline with agentic toggle ───────────────
@app.route('/api/recommend', methods=['POST'])
def api_recommend():
    """
    Streaming SSE endpoint. Same body schema as before, but the response is
    a Server-Sent-Events stream with these events:
      event: status   data: {"step":"vision|safety|psy|librarian", "msg":"...", "elapsed":X}
      event: item     data: {<single recommendation item>}
      event: done     data: {<final payload: thesis, psych_profile, safety, timings, ...>}
      event: error    data: {"error":"..."}
      event: crisis   data: {<crisis payload>}
    """
    rid = int(time.time() * 1000) % 100000
    body           = request.get_json() or {}
    free_text      = (body.get("free_text") or "").strip()
    selected_moods = body.get("selected_moods", [])
    birth_year     = body.get("birth_year")
    lang           = body.get("lang", "en")
    bypass_safety  = body.get("continue_anyway", False)
    use_agentic    = bool(body.get("agentic", False))
    nonce          = body.get("nonce")
    image_b64      = body.get("image_b64")

    print(f"\n[{rid}] ▶ POST /api/recommend (streaming)", flush=True)

    def sse(event: str, payload: dict) -> str:
        """Format a Server-Sent Event frame."""
        return f"event: {event}\ndata: {json.dumps(payload, ensure_ascii=False)}\n\n"

    def generate():
        t0_total = time.time()
        try:
            # Validate input
            if not free_text and not selected_moods and not image_b64:
                yield sse("error", {"error": "Provide free_text, selected_moods, or image_b64"})
                return

            set_request_context(nonce=nonce)

            # ── Step 1 · Vision + Psychologist (fused if image present)
            if image_b64:
                yield sse("status", {"step": "vision_psy", "msg": "Reading your image and decoding emotion...", "elapsed": 0})
                t1 = time.time()
                psych_profile = run_psychologist_multimodal(
                    free_text, selected_moods, birth_year, image_b64
                )
                t_vision_psy = time.time() - t1
                t_vision = 0.0
                t_psych  = t_vision_psy
                # Build a synthetic "image_summary" for downstream UI compatibility
                image_summary = {
                    "scene":          psych_profile.get("image_read", ""),
                    "is_meme":        psych_profile.get("is_meme", False),
                    "meme_text":      "",
                    "emotional_tone": psych_profile.get("emotion_core", ""),
                    "likely_mood":    (psych_profile.get("active_moods") or ["undefined"])[0],
                    "subtext":        psych_profile.get("decoder_note", ""),
                }
                # Use image_read as the displayable enriched_text
                enriched_text = (free_text + " " if free_text else "") + f"[image: {psych_profile.get('image_read','')}]"
                print(f"[{rid}]   vision+psy fused done ({t_vision_psy:.1f}s) "
                      f"emotion={psych_profile.get('emotion_core')!r} "
                      f"mech={psych_profile.get('primary_mechanism')!r}", flush=True)
                yield sse("status", {"step": "vision_psy_done",
                                     "msg": f"Decoded: {psych_profile.get('emotion_core', '')}",
                                     "elapsed": round(t_vision_psy, 1)})

                # Send rich vision insight to the frontend so the user
                # can SEE what Ami understood about their image.
                yield sse("vision_insight", {
                    "input_type":   ("image+text" if free_text else "image"),
                    "image_read":   psych_profile.get("image_read", ""),
                    "understanding": psych_profile.get("image_read", "")
                                     or psych_profile.get("decoder_note", ""),
                    "emotion_core": psych_profile.get("emotion_core", ""),
                    "is_meme":      psych_profile.get("is_meme", False),
                    "is_quote":     psych_profile.get("is_quote", False),
                    "quote_author": psych_profile.get("quote_author"),
                    "active_moods": psych_profile.get("active_moods", []),
                    "intensity":    psych_profile.get("intensity", "medium"),
                    "decoder_note": psych_profile.get("decoder_note", ""),
                })
            else:
                image_summary = None
                enriched_text = free_text
                t_vision = 0.0
                # Text-only path: still need psy, but it runs after safety below
                psych_profile = None

            # ── Step 2 · Safety
            yield sse("status", {"step": "safety", "msg": "Checking safety signals...", "elapsed": round(time.time()-t0_total,1)})
            t1 = time.time()
            safety = classify_safety(enriched_text, selected_moods)
            t_safety = time.time() - t1
            print(f"[{rid}]   safety={safety['level']} ({t_safety:.1f}s)", flush=True)

            if safety["level"] == "crisis" and not bypass_safety:
                payload = {**crisis_payload(), "safety": safety}
                yield sse("crisis", payload)
                return

            # ── Step 3 · Text-only psychologist (only if no image was given)
            if psych_profile is None:
                yield sse("status", {"step": "psy", "msg": "Decoding your emotional state...", "elapsed": round(time.time()-t0_total,1)})
                t1 = time.time()
                psych_profile = run_psychologist(enriched_text, selected_moods, birth_year)
                t_psych = time.time() - t1
                print(f"[{rid}]   psy done ({t_psych:.1f}s)", flush=True)

            # Tone tweak based on safety
            if safety["level"] in ("high_distress", "crisis"):
                psych_profile["surprise_ok"] = False
                psych_profile["tone"]        = "tender"
            else:
                psych_profile.setdefault("tone", "warm")
            psych_profile["_nonce"] = nonce

            yield sse("status", {"step": "psy_done",
                                 "msg": psych_profile.get("decoder_note", "Understood."),
                                 "elapsed": round(time.time()-t0_total,1)})

            # Emit a vision_insight event so the frontend's understanding card
            # shows up for text-only inputs too. Only fire here if we didn't
            # already fire one in the image path above (i.e. no image_b64).
            if not image_b64:
                yield sse("vision_insight", {
                    "input_type":   "text",
                    "image_read":   "",
                    "understanding": psych_profile.get("decoder_note", "")
                                     or (free_text[:120] if free_text else ""),
                    "emotion_core": psych_profile.get("emotion_core", ""),
                    "is_meme":      False,
                    "is_quote":     False,
                    "quote_author": None,
                    "active_moods": psych_profile.get("active_moods", []),
                    "intensity":    psych_profile.get("intensity", "medium"),
                    "decoder_note": psych_profile.get("decoder_note", ""),
                })

            # ── Step 4 · Librarian (the slow one — keep-alive matters here)
            yield sse("status", {"step": "librarian", "msg": "Curating works for you...", "elapsed": round(time.time()-t0_total,1)})
            t1 = time.time()

            # Run librarian in a thread so we can heartbeat while it's working
            import threading
            result_holder = {}
            def _run_lib():
                try:
                    if use_agentic:
                        result_holder["data"] = run_librarian_agentic(psych_profile, verbose=False)
                    else:
                        result_holder["data"] = run_librarian(psych_profile)
                except Exception as e:
                    result_holder["error"] = e

            th = threading.Thread(target=_run_lib, daemon=True)
            th.start()
            heartbeat_msgs = [
                "Still working — Gemma is comparing your state to validated mechanisms...",
                "Selecting items from the catalogue...",
                "Writing personal explanations...",
                "Almost there...",
            ]
            i = 0
            while th.is_alive():
                th.join(timeout=15.0)  # wake up every 15s to heartbeat
                if th.is_alive():
                    msg = heartbeat_msgs[min(i, len(heartbeat_msgs)-1)]
                    yield sse("status", {"step": "librarian_progress", "msg": msg,
                                          "elapsed": round(time.time()-t0_total,1)})
                    i += 1

            if "error" in result_holder:
                raise result_holder["error"]
            result = result_holder["data"]
            t_lib = time.time() - t1
            n_items = len(result.get("items", []))
            print(f"[{rid}]   librarian done ({t_lib:.1f}s, {n_items} items)", flush=True)

            yield sse("status", {"step": "librarian_done",
                                 "msg": f"Found {n_items} works for you.",
                                 "elapsed": round(time.time()-t0_total,1)})

            # ── Step 5 · Stream items one by one (small artificial gap for UX)
            for idx, item in enumerate(result.get("items", [])):
                yield sse("item", {"index": idx, "total": n_items, "item": item})
                time.sleep(0.4)  # gentle stagger so the UI animates

            # ── Step 6 · Final payload
            t_total = time.time() - t0_total
            yield sse("done", {
                "thesis":        result.get("thesis", ""),
                "thematic_data": result.get("thematic_data", []),
                "psych_profile": {
                    "emotion_core":      psych_profile.get("emotion_core"),
                    "primary_mechanism": psych_profile.get("primary_mechanism"),
                    "decoder_note":      psych_profile.get("decoder_note"),
                    "intensity":         psych_profile.get("intensity"),
                    "surprise_ok":       psych_profile.get("surprise_ok"),
                    "tone":              psych_profile.get("tone", "warm"),
                },
                "image_summary": image_summary,
                "enriched_text": enriched_text,
                "safety":        safety,
                "agentic":       use_agentic,
                "tool_trace":    result.get("tool_trace", []),
                "warning":       result.get("warning"),
                "timings": {
                    "vision_s":       round(t_vision, 1),
                    "safety_s":       round(t_safety, 1),
                    "psychologist_s": round(t_psych, 1),
                    "librarian_s":    round(t_lib, 1),
                    "total_s":        round(t_total, 1),
                    "mode":           "agentic" if use_agentic else "classic",
                    "fused":          bool(image_b64),  # was vision+psy fused?
                }
            })
            print(f"[{rid}] ◀ streaming done total={t_total:.1f}s", flush=True)

        except Exception as e:
            print(f"[{rid}] ✖ EXCEPTION: {e}", flush=True)
            traceback.print_exc()
            yield sse("error", {"error": str(e), "trace": traceback.format_exc()[:500]})

    # SSE response. The crucial headers:
    #   - X-Accel-Buffering: no  → disables proxy buffering (ngrok respects this)
    #   - Cache-Control: no-cache → no intermediate caching
    return Response(generate(), mimetype="text/event-stream", headers={
        "Cache-Control":      "no-cache",
        "X-Accel-Buffering":  "no",
        "Connection":         "keep-alive",
    })



# ─── Health check ───────────────────────────────────────────────────
@app.route('/api/health')
def health():
    return jsonify({
        'status':    'ok',
        'model':     'gemma-4-31b-it',
        'device':    str(model.device),
        'catalogue': f"{len(ALL_ITEMS)} items",
        'vram_gb':   round(torch.cuda.memory_allocated()/1e9, 1),
    })


# ─── Run Flask in a daemon thread ───────────────────────────────────
def run_flask():
    app.run(host='0.0.0.0', port=FLASK_PORT, debug=False, use_reloader=False)

flask_thread = threading.Thread(target=run_flask, daemon=True)
flask_thread.start()
time.sleep(2)

print(f"✅ Flask server running on port {FLASK_PORT}")
print(f"   POST /api/recommend  ← full pipeline (agentic toggle)")
print(f"   GET  /api/health     ← sanity check")

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.19.2.2:5000
Press CTRL+C to quit


✅ Flask server running on port 5000
   POST /api/recommend  ← full pipeline (agentic toggle)
   GET  /api/health     ← sanity check


In [18]:
# ════════════════════════════════════════════════
# 14 · Open ngrok tunnel
# ════════════════════════════════════════════════

from pyngrok import ngrok, conf

# Configure l'authtoken
conf.get_default().auth_token = NGROK_TOKEN

# Ouvre le tunnel
tunnel = ngrok.connect(FLASK_PORT, 'http')
public_url = tunnel.public_url

print("=" * 60)
print("🎉  AMI EST EN LIGNE !")
print("=" * 60)
print(f"\n🌐  URL PUBLIQUE : {public_url}")
print(f"\n   → URL à partager")
print(f"   → Health check : {public_url}/api/health")
print("\n⚠️  L'URL expire quand tu fermes le notebook")
print("=" * 60)

t=2026-05-18T08:05:35+0000 lvl=warn msg="failed to start tunnel" pg=/api/tunnels id=10f4fac4aa5fd328 err="failed to start tunnel: The endpoint 'https://nemeses-portion-bartender.ngrok-free.dev' is already online. Either\n1. stop your existing endpoint first, or\n2. start both endpoints with `--pooling-enabled` to load balance between them.\r\n\r\nERR_NGROK_334\r\n"


PyngrokNgrokHTTPError: ngrok client exception, API returned 502: {"error_code":103,"status_code":502,"msg":"failed to start tunnel","details":{"err":"failed to start tunnel: The endpoint 'https://nemeses-portion-bartender.ngrok-free.dev' is already online. Either\n1. stop your existing endpoint first, or\n2. start both endpoints with `--pooling-enabled` to load balance between them.\r\n\r\nERR_NGROK_334\r\n"}}
